# WiSig Module3 Dual-Track v10

## Purpose
This notebook evaluates **open-set domain drift handling** for WiSig SEI under a dual-track development strategy:

1. **Dual-score diagnosis** distinguishes:
   - known-class domain drift
   - unknown / novel transmitters
2. **Diagnosis-guided adaptation** updates the classifier under pseudo-label uncertainty
3. The notebook compares:
   - classical pseudo-label adaptation baselines
   - RAINCOAT-style adaptation baselines
   - three-bucket curriculum / promotion baselines
   - the two current development lines:
     - **DG_RAINCOAT** (diagnosis-guided RAINCOAT, v3 weighting / rescue)
     - **ThreeBucketV8Adaptive** (v8.1 adaptive promotion line)

## Important rule for later versions
From **v9 onward**, each version must keep explicit markdown records of:
- version-specific code changes
- experimental design
- method abbreviations in console output
- expected observations / failure modes


## v10 update note

This version continues from `v9_docs_fix2` and makes **controlled algorithmic changes** rather than compatibility-only edits.

Main updates:
1. **DG_RAINCOAT -> v3 style weighting**
   - rescue logic now targets **high-domain-shift but identity-consistent** samples
   - support construction is tightened: not all non-unknown gate samples are supervised equally
   - ambiguous samples are kept, but with weaker supervision than reliable samples
   - unknown handling is pushed later so it interferes less with correction
2. **ThreeBucketV8Adaptive -> v8.1 style promotion**
   - stronger per-class promotion budgets
   - semi-weak samples can be promoted into ambiguous more easily
   - weak/cold-weak split is relaxed so fewer potentially useful samples are frozen out
3. **Experiment bookkeeping**
   - output directory name is updated to the v10 series to avoid mixing with older dual-track runs


## Version log

### v10 (this notebook)
Current code changes relative to `v9_docs_fix2`:
1. **DG_RAINCOAT upgraded toward v3**
   - rescue branch now targets **high-Sdom drift-known** candidates instead of low-Sdom candidates
   - support set is no longer built from almost all non-unknown gate samples
   - ambiguous supervision is weakened and reliable / rescued samples are weighted more strongly
   - alignment is kept broader than supervision to reduce over-filtering
2. **ThreeBucketV8Adaptive upgraded toward v8.1**
   - per-class ambiguous promotion is strengthened
   - semi-weak -> ambiguous promotion is loosened
   - cold-weak split is softened so more samples remain trainable
3. **Bookkeeping / reproducibility**
   - result directory renamed to the v10 suite

### v5
- Three-bucket v5 baseline family:
  - `ThreeBucketV5Soft`
  - `ThreeBucketV5EMA`
  - `ThreeBucketV5EMAProto`
- Main issue: extremely high pseudo-label purity on selected samples, but very low `sel_keep` / `sel_recall`.

### v6
- Added curriculum usage of weak samples.
- Goal: increase target coverage.
- Observed issue: coverage increased, but noisy supervision also increased, so overall gain was limited.

### v7
- Added promotion mechanism and two-stage training.
- Goal: make weak samples promotable instead of permanently weak.
- Observed issue: promotion logic could become unstable on hard TX splits.

### v8 dual-track
- Added two parallel development lines:
  - `DG_RAINCOAT`
  - `ThreeBucketV8Adaptive`
- Goal: compare an upgraded strong baseline line vs. an original bucket-promotion line.

### v9 (this notebook)
Current code changes relative to v8:
1. **DG_RAINCOAT upgraded toward v2**
   - hard filtering -> continuous weighting
   - added **drift rescue** branch
   - moved unknown handling later
   - support-set evaluation uses the actual support index set
2. **ThreeBucketV8Adaptive upgraded toward v8.1**
   - weak bucket split into:
     - `semi-weak`
     - `cold-weak`
   - more class-adaptive promotion budgeting
   - `semi-weak` gets low-weight soft supervision
   - `cold-weak` does not receive strong class supervision
3. **Bug fix**
   - all EMA updates now use:
     ```python
     ema_update_(teacher, adapter, momentum=ema_momentum)
     ```
     instead of the incorrect `mom=ema_momentum`

### v10 fix1 notebook compatibility patch

This patch only fixes a notebook/script compatibility issue and does **not** change algorithm behavior.

- `run_one_split()` passes `idx_weak_cold` and `w_weak_cold` to all three-bucket variants.
- In the original v10 notebook, `adapt_three_bucket_v6()` and `adapt_three_bucket_v7()` did not accept these optional arguments.
- This fix adds `idx_weak_cold=None, w_weak_cold=None` to the **v6/v7 function signatures only**.
- For v6/v7 these arguments are accepted and ignored, so results should remain behaviorally unchanged.
- V8 logic is unchanged.


## Method abbreviations in fold output

When each fold prints one-line results, the method tags mean:

- `NoAdapt`: no target adaptation
- `TriSoft`: tri-agreement soft pseudo-label adaptation
- `Refine`: prototype-refined pseudo-label adaptation
- `RAIN`: `RAINCOAT_Lite`
- `DGR`: `DG_RAINCOAT`
- `TBV5S`: `ThreeBucketV5Soft`
- `TBV5E`: `ThreeBucketV5EMA`
- `TBV5P`: `ThreeBucketV5EMAProto`
- `TBV6`: `ThreeBucketV6Curriculum`
- `TBV7`: `ThreeBucketV7Promotion`
- `TBV8`: `ThreeBucketV8Adaptive`
- `OracleLbl`: oracle-label upper bound

Pseudo-label diagnostics:
- `pLogit`: raw classifier pseudo-label accuracy on true drift pool
- `pLP`: label-propagation pseudo-label accuracy
- `pTri`: tri-agreement pseudo-label accuracy

## Experimental design

### Data split logic
- TX splits are built with **unique known-TX combinations**.
- Within each TX split:
  - four TX are treated as **known**
  - the remaining TX are treated as **unknown**
- Each TX split is evaluated across multiple folds.

### Core evaluation goal
The main problem is **not only adaptation accuracy**, but also whether the method can:
1. keep stable-domain known users accepted
2. recover known users under domain drift
3. reject unknown transmitters

### Key metrics to monitor
Primary:
- `drift_acc_all`
- `stable_acc`
- `FAR_unknown_accept`

Secondary:
- `FRR_stable`
- `miss_drift_rx`
- `miss_drift_day`
- `auc_unknown_eval`

Selection diagnostics:
- `sel_precision`
- `sel_recall`
- `sel_keep`
- `pseudo_acc_selected`

### Interpretation rule used in this project
A method is considered genuinely useful only if it improves **known-drift recovery**
without paying an excessive price in:
- stable-domain FRR
- unknown acceptance FAR

## v10 experimental hypotheses

### H1: DG_RAINCOAT should keep the operating-point gains while recovering some drift accuracy
Expected direction:
- lower `FRR_stable` than `RAINCOAT_Lite`
- higher `auc_unknown_eval` than `RAINCOAT_Lite`
- better `drift_acc_all` than the previous DG_RAINCOAT version because noisy ambiguous supervision is weakened

Risk:
- if support narrowing becomes too strict, `miss_drift_day` may remain high
- if rescued-common criteria are still too loose, drift-known noise will continue to hurt correction

### H2: ThreeBucketV8Adaptive should benefit from stronger class-aware promotion
Expected direction:
- better `drift_acc_all` than the previous `TBV8`
- stronger `sel_recall` / `sel_keep` without collapsing pseudo-label purity
- fewer samples trapped in weak / cold-weak buckets

Risk:
- if semi-weak -> ambiguous promotion is too loose, pseudo-label noise may rise sharply
- if class budgets remain imbalanced, hard classes may still fail to enter reliable / ambiguous buckets


## Notebook structure

1. **Imports and global configuration**
2. **Dataset utilities and split construction**
3. **Backbone / embedding model**
4. **Scoring, pseudo-labeling, and dual-score diagnosis**
5. **Adaptation methods**
   - pseudo-label baselines
   - RAINCOAT line
   - DG_RAINCOAT
   - three-bucket line
6. **Evaluation**
7. **Main experiment loop**
8. **Summary tables and diagnostic aggregation**

In [1]:

import os, json, time
import itertools
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve, auc, confusion_matrix
from sklearn.decomposition import PCA

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pickle

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

dataset_name = "ManySig"
dataset_path = "../ManySig.pkl/"

def load_compact_pkl_dataset(dataset_path,dataset_name):
    with open(dataset_path+dataset_name+'.pkl','rb') as f:
        dataset = pickle.load(f)
    return dataset

# from <your_loader_module> import load_compact_pkl_dataset
compact_dataset = load_compact_pkl_dataset(dataset_path, dataset_name)


TX_TOTAL_USE = 6
RX_TOTAL_USE = 12
DAY_TOTAL_USE = 4
KNOWN_TX_NUM = 4
SOURCE_RX_NUM = 3
TX_SPLIT_REPEATS = 5

TRAIN_DATES = ["2021_03_15"]
TEST_DATES_RX = ["2021_03_15"]
TEST_DATES_DAY = ["2021_03_01"]

Z_FROM_EQ = 1
D_FROM = "raw"
MAX_SIG_PER_TRIPLE = None

BATCH_SIZE = 128
EPOCHS = 60
LR = 1e-4
WEIGHT_DECAY = 1e-3
PATIENCE = 8
IN_PLANES = 64
DROPOUT = 0.3
D_DIM = 32

# fixed Module 2
FUSION_LAM = 0.5
FRR_TARGET = 0.05
FALSE_DRIFT_TARGET = 0.05

# stream split
ADAPT_FRAC = 0.5
MAX_ADAPT_PER_SET = 2000
MAX_EVAL_PER_SET = 3000

# adapter
ADAPTER_BOTTLENECK = 64
ADAPTER_SCALE = 0.3
ADAPT_EPOCHS = 20
ADAPT_LR = 1e-3
ADAPT_WEIGHT_DECAY = 1e-4
ADAPT_BATCH = 128

# replay
REPLAY_PER_CLASS = 200
LAMBDA_REPLAY_CE = 1.0
LAMBDA_REPLAY_ANCHOR = 1.0

# pseudo-label suite params
PSEUDO_MIN_KEEP = 64
SOFT_T = 2.0
PROTO_T = 0.10
KNN_TOPK = 15
KNN_MEM_PER_CLASS = 300

# fusion weights
W_LOGIT = 1.0
W_PROTO = 1.0
W_KNN = 1.0

# confidence threshold from stable source-like set A
CONF_STABLE_Q = 0.10

# new suite hyper-parameters
CONF_DRIFT_Q = 0.40
REFINE_ITERS = 2
UNKNOWN_SCORE_Q = 0.35
EMB_AUG_NOISE = 0.02
MMD_SIGMAS = [1.0, 2.0, 4.0, 8.0, 16.0]
LAMBDA_ALIGN = 0.35
LAMBDA_ENT_MIN = 0.02
LAMBDA_ENT_MAX = 0.05
LAMBDA_CONS = 0.05
LAMBDA_PROTO = 0.05
GCODWFA_ALIGN_MAX = 0.80
RAIN_STAGE1_EPOCHS = 8
RAIN_STAGE2_EPOCHS = 12


# v5 three-bucket parameters
RELIABLE_KEEP_Q = 0.12
AMBIG_KEEP_Q = 0.30
UNKNOWN_KEEP_Q = 0.08

AMBIG_TOPK = 2
RELIABLE_WEIGHT_FLOOR = 0.65
AMBIG_WEIGHT_FLOOR = 0.30
AMBIG_WEIGHT_SCALE = 0.70

EMA_MOMENTUM = 0.995
EMA_TEACHER_BLEND = 0.60
TEACHER_TEMP = 1.0

THREE_BUCKET_EPOCHS = 24
LAMBDA_AMB_SUP = 0.60
LAMBDA_SRC_PROTO = 0.08
LAMBDA_SRC_LOGIT = 0.30
LAMBDA_UNKNOWN_REPEL = 0.02
UNKNOWN_REPEL_MARGIN = 0.25
UNKNOWN_WARMUP_EPOCHS = 6
UNKNOWN_RAMP_EPOCHS = 8

# v6 curriculum three-bucket parameters
V6_RELIABLE_KEEP_Q = 0.22
V6_AMBIG_KEEP_Q = 0.30
V6_UNKNOWN_KEEP_Q = 0.05
V6_UNKNOWN_KEEP_Q_HARD = 0.08
V6_WEAK_MIN_KEEP = 64
V6_WEAK_TOPK = 3
V6_WEAK_SHARPEN = 0.95
V6_REL_WEIGHT_FLOOR = 0.72
V6_AMB_WEIGHT_FLOOR = 0.36
V6_AMB_WEIGHT_SCALE = 0.44
V6_WEAK_WEIGHT_FLOOR = 0.10
V6_WEAK_WEIGHT_SCALE = 0.20
V6_WEAK_START_EPOCH = 4
V6_WEAK_TEACHER_BLEND = 0.35
V6_LAMBDA_WEAK_SUP = 0.22
V6_LAMBDA_AMB_SUP = 0.50
V6_LAMBDA_ALIGN = 0.30
V6_LAMBDA_ENT_MIN = 0.01
V6_LAMBDA_ENT_MAX = 0.025
V6_LAMBDA_CONS = 0.04
V6_LAMBDA_PROTO = 0.04
V6_LAMBDA_UNKNOWN_REPEL = 0.008
V6_UNKNOWN_WARMUP_EPOCHS = 10
V6_UNKNOWN_RAMP_EPOCHS = 10

# v7 two-stage promotion three-bucket parameters
V7_RELIABLE_KEEP_Q = 0.16
V7_AMBIG_KEEP_Q = 0.28
V7_UNKNOWN_KEEP_Q = 0.04
V7_UNKNOWN_KEEP_Q_HARD = 0.06
V7_WEAK_MIN_KEEP = 96
V7_WEAK_TOPK = 3
V7_WEAK_SHARPEN = 1.00
V7_REL_WEIGHT_FLOOR = 0.80
V7_AMB_WEIGHT_FLOOR = 0.28
V7_AMB_WEIGHT_SCALE = 0.32
V7_WEAK_WEIGHT_FLOOR = 0.08
V7_WEAK_WEIGHT_SCALE = 0.12
V7_STAGE1_EPOCHS = 8
V7_PROMOTE_BLEND = 0.65
V7_PROMOTE_REL_FRACTION = 0.18
V7_PROMOTE_AMB_FRACTION = 0.32
V7_PROMOTE_MIN_REL = 24
V7_PROMOTE_MIN_AMB = 48
V7_PROMOTE_CONF = 0.72
V7_PROMOTE_MARGIN = 0.12
V7_AMB_CONF = 0.48
V7_AMB_MARGIN = 0.06
V7_LAMBDA_WEAK_SUP = 0.0
V7_LAMBDA_AMB_SUP = 0.42
V7_LAMBDA_ALIGN = 0.22
V7_LAMBDA_ENT_MIN = 0.008
V7_LAMBDA_ENT_MAX = 0.015
V7_LAMBDA_CONS = 0.06
V7_LAMBDA_PROTO = 0.05
V7_LAMBDA_UNKNOWN_REPEL = 0.003
V7_UNKNOWN_WARMUP_EPOCHS = 16
V7_UNKNOWN_RAMP_EPOCHS = 12



# DG-RAINCOAT (diagnosis-guided RAINCOAT-lite) parameters
DG_REL_KEEP_Q = 0.14
DG_AMB_KEEP_Q = 0.18
DG_UNK_KEEP_Q = 0.08
DG_STAGE1_EPOCHS = 8
DG_STAGE2_EPOCHS = 14
DG_WARM_SUP_SCALE = 0.75
DG_FINAL_SUP_SCALE = 1.00
DG_LAMBDA_ALIGN_STAGE1 = 0.26
DG_LAMBDA_ALIGN_STAGE2 = 0.16
DG_LAMBDA_ENT_MIN = 0.006
DG_LAMBDA_ENT_MAX = 0.020
DG_LAMBDA_CONS = 0.05
DG_LAMBDA_PROTO = 0.05
DG_LAMBDA_UNKNOWN_REPEL = 0.006
DG_MOVE_BLEND = 0.28
DG_STABLE_BLEND = 0.16

# v8 adaptive multi-refresh promotion parameters
V8_RELIABLE_KEEP_Q = 0.18
V8_AMBIG_KEEP_Q = 0.24
V8_UNKNOWN_KEEP_Q = 0.06
V8_UNKNOWN_KEEP_Q_HARD = 0.08
V8_REL_WEIGHT_FLOOR = 0.84
V8_AMB_WEIGHT_FLOOR = 0.26
V8_AMB_WEIGHT_SCALE = 0.30
V8_WEAK_WEIGHT_FLOOR = 0.06
V8_WEAK_WEIGHT_SCALE = 0.10
V8_STAGE0_EPOCHS = 6
V8_REFRESH_EPOCHS = 6
V8_REFRESH_ROUNDS = 3
V8_STABLE_K_REL = 2
V8_STABLE_K_AMB = 1
V8_PROMOTE_CONF_REL = 0.72
V8_PROMOTE_CONF_AMB = 0.50
V8_PROMOTE_MARGIN_REL = 0.12
V8_PROMOTE_MARGIN_AMB = 0.04
V8_PROMOTE_SCORE_REL_Q = 0.72
V8_PROMOTE_SCORE_AMB_Q = 0.52
V8_LAMBDA_ALIGN = 0.18
V8_LAMBDA_ENT_MIN = 0.006
V8_LAMBDA_ENT_MAX = 0.016
V8_LAMBDA_CONS = 0.05
V8_LAMBDA_PROTO = 0.05
V8_LAMBDA_UNKNOWN_REPEL = 0.004
V8_STABILITY_BLEND = 0.16
DG_RESCUE_SDOM_Q = 0.62
DG_RESCUE_CONF = 0.56
DG_RESCUE_MARGIN = 0.05
DG_SUPPORT_WEIGHT_FLOOR = 0.08
DG_SUPPORT_WEIGHT_SCALE = 0.72
DG_UNKNOWN_EXTREME_Q = 0.94

V81_COLD_WEAK_Q = 0.42
V81_COLD_WEIGHT_FLOOR = 0.02
V81_COLD_WEIGHT_SCALE = 0.04
V81_CLASS_REL_FRAC = 0.12
V81_CLASS_AMB_FRAC = 0.20
V81_SEMIWEAK_TOPK = 3

METHOD_ORDER = [

    "NoAdapt",
    "UngatedAdapt",
    "GatedAdapt",
    "GatedSelfSoft",
    "GatedProtoAgree",
    "GatedProtoFusionSoft",
    "GatedTriAgree",
    "GatedTriFusionSoft",
    "ConfThreshSoft",
    "AgreementConfSoft",
    "ProtoRefineSoft",
    "ReliableEntropySplit",
    "TwoStageUnknown",
    "GCODWFA_Lite",
    "RAINCOAT_Lite",
    "DG_RAINCOAT",
    "ThreeBucketV5Soft",
    "ThreeBucketV5EMA",
    "ThreeBucketV5EMAProto",
    "ThreeBucketV6Curriculum",
    "ThreeBucketV7Promotion",
    "ThreeBucketV8Adaptive",
    "OracleGatePseudoAdapt",
    "OracleLabelAdapt",
]

ts = time.strftime("%Y%m%d_%H%M%S")
RUN_DIR = f"./results_wisig_module3_v10_dualtrack_suite/run_{ts}"
os.makedirs(RUN_DIR, exist_ok=True)

cfg = dict(
    SEED=SEED, TX_TOTAL_USE=TX_TOTAL_USE, RX_TOTAL_USE=RX_TOTAL_USE, DAY_TOTAL_USE=DAY_TOTAL_USE,
    KNOWN_TX_NUM=KNOWN_TX_NUM, SOURCE_RX_NUM=SOURCE_RX_NUM, TX_SPLIT_REPEATS=TX_SPLIT_REPEATS,
    TRAIN_DATES=TRAIN_DATES, TEST_DATES_RX=TEST_DATES_RX, TEST_DATES_DAY=TEST_DATES_DAY,
    Z_FROM_EQ=Z_FROM_EQ, D_FROM=D_FROM, MAX_SIG_PER_TRIPLE=MAX_SIG_PER_TRIPLE,
    BATCH_SIZE=BATCH_SIZE, EPOCHS=EPOCHS, LR=LR, WEIGHT_DECAY=WEIGHT_DECAY, PATIENCE=PATIENCE,
    IN_PLANES=IN_PLANES, DROPOUT=DROPOUT, D_DIM=D_DIM,
    FUSION_LAM=FUSION_LAM, FRR_TARGET=FRR_TARGET, FALSE_DRIFT_TARGET=FALSE_DRIFT_TARGET,
    ADAPT_FRAC=ADAPT_FRAC, MAX_ADAPT_PER_SET=MAX_ADAPT_PER_SET, MAX_EVAL_PER_SET=MAX_EVAL_PER_SET,
    ADAPTER_BOTTLENECK=ADAPTER_BOTTLENECK, ADAPTER_SCALE=ADAPTER_SCALE,
    ADAPT_EPOCHS=ADAPT_EPOCHS, ADAPT_LR=ADAPT_LR, ADAPT_WEIGHT_DECAY=ADAPT_WEIGHT_DECAY, ADAPT_BATCH=ADAPT_BATCH,
    REPLAY_PER_CLASS=REPLAY_PER_CLASS, LAMBDA_REPLAY_CE=LAMBDA_REPLAY_CE, LAMBDA_REPLAY_ANCHOR=LAMBDA_REPLAY_ANCHOR,
    PSEUDO_MIN_KEEP=PSEUDO_MIN_KEEP, SOFT_T=SOFT_T, PROTO_T=PROTO_T, KNN_TOPK=KNN_TOPK, KNN_MEM_PER_CLASS=KNN_MEM_PER_CLASS,
    W_LOGIT=W_LOGIT, W_PROTO=W_PROTO, W_KNN=W_KNN, CONF_STABLE_Q=CONF_STABLE_Q,
    CONF_DRIFT_Q=CONF_DRIFT_Q, REFINE_ITERS=REFINE_ITERS, UNKNOWN_SCORE_Q=UNKNOWN_SCORE_Q,
    EMB_AUG_NOISE=EMB_AUG_NOISE, MMD_SIGMAS=MMD_SIGMAS,
    LAMBDA_ALIGN=LAMBDA_ALIGN, LAMBDA_ENT_MIN=LAMBDA_ENT_MIN, LAMBDA_ENT_MAX=LAMBDA_ENT_MAX,
    LAMBDA_CONS=LAMBDA_CONS, LAMBDA_PROTO=LAMBDA_PROTO,
GCODWFA_ALIGN_MAX=GCODWFA_ALIGN_MAX, RAIN_STAGE1_EPOCHS=RAIN_STAGE1_EPOCHS, RAIN_STAGE2_EPOCHS=RAIN_STAGE2_EPOCHS,
RELIABLE_KEEP_Q=RELIABLE_KEEP_Q, AMBIG_KEEP_Q=AMBIG_KEEP_Q, UNKNOWN_KEEP_Q=UNKNOWN_KEEP_Q,
AMBIG_TOPK=AMBIG_TOPK, AMBIG_WEIGHT_SCALE=AMBIG_WEIGHT_SCALE,
RELIABLE_WEIGHT_FLOOR=RELIABLE_WEIGHT_FLOOR, AMBIG_WEIGHT_FLOOR=AMBIG_WEIGHT_FLOOR,
EMA_MOMENTUM=EMA_MOMENTUM, EMA_TEACHER_BLEND=EMA_TEACHER_BLEND, TEACHER_TEMP=TEACHER_TEMP,
THREE_BUCKET_EPOCHS=THREE_BUCKET_EPOCHS, LAMBDA_AMB_SUP=LAMBDA_AMB_SUP,
LAMBDA_SRC_PROTO=LAMBDA_SRC_PROTO, LAMBDA_SRC_LOGIT=LAMBDA_SRC_LOGIT,
LAMBDA_UNKNOWN_REPEL=LAMBDA_UNKNOWN_REPEL, UNKNOWN_REPEL_MARGIN=UNKNOWN_REPEL_MARGIN,
UNKNOWN_WARMUP_EPOCHS=UNKNOWN_WARMUP_EPOCHS, UNKNOWN_RAMP_EPOCHS=UNKNOWN_RAMP_EPOCHS,
V7_RELIABLE_KEEP_Q=V7_RELIABLE_KEEP_Q, V7_AMBIG_KEEP_Q=V7_AMBIG_KEEP_Q,
V7_UNKNOWN_KEEP_Q=V7_UNKNOWN_KEEP_Q, V7_UNKNOWN_KEEP_Q_HARD=V7_UNKNOWN_KEEP_Q_HARD,
V7_WEAK_MIN_KEEP=V7_WEAK_MIN_KEEP, V7_WEAK_TOPK=V7_WEAK_TOPK, V7_WEAK_SHARPEN=V7_WEAK_SHARPEN,
V7_REL_WEIGHT_FLOOR=V7_REL_WEIGHT_FLOOR, V7_AMB_WEIGHT_FLOOR=V7_AMB_WEIGHT_FLOOR,
V7_AMB_WEIGHT_SCALE=V7_AMB_WEIGHT_SCALE, V7_WEAK_WEIGHT_FLOOR=V7_WEAK_WEIGHT_FLOOR,
V7_WEAK_WEIGHT_SCALE=V7_WEAK_WEIGHT_SCALE, V7_STAGE1_EPOCHS=V7_STAGE1_EPOCHS,
V7_PROMOTE_BLEND=V7_PROMOTE_BLEND, V7_PROMOTE_REL_FRACTION=V7_PROMOTE_REL_FRACTION,
V7_PROMOTE_AMB_FRACTION=V7_PROMOTE_AMB_FRACTION, V7_PROMOTE_MIN_REL=V7_PROMOTE_MIN_REL,
V7_PROMOTE_MIN_AMB=V7_PROMOTE_MIN_AMB, V7_PROMOTE_CONF=V7_PROMOTE_CONF, V7_PROMOTE_MARGIN=V7_PROMOTE_MARGIN,
V7_AMB_CONF=V7_AMB_CONF, V7_AMB_MARGIN=V7_AMB_MARGIN,
V7_LAMBDA_WEAK_SUP=V7_LAMBDA_WEAK_SUP, V7_LAMBDA_AMB_SUP=V7_LAMBDA_AMB_SUP,
V7_LAMBDA_ALIGN=V7_LAMBDA_ALIGN, V7_LAMBDA_ENT_MIN=V7_LAMBDA_ENT_MIN, V7_LAMBDA_ENT_MAX=V7_LAMBDA_ENT_MAX,
V7_LAMBDA_CONS=V7_LAMBDA_CONS, V7_LAMBDA_PROTO=V7_LAMBDA_PROTO,
V7_LAMBDA_UNKNOWN_REPEL=V7_LAMBDA_UNKNOWN_REPEL, V7_UNKNOWN_WARMUP_EPOCHS=V7_UNKNOWN_WARMUP_EPOCHS,
V7_UNKNOWN_RAMP_EPOCHS=V7_UNKNOWN_RAMP_EPOCHS,
    METHOD_ORDER=METHOD_ORDER,
)
with open(os.path.join(RUN_DIR, "config.json"), "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2)
print("RUN_DIR:", RUN_DIR)



Device: cuda
RUN_DIR: ./results_wisig_module3_v10_dualtrack_suite/run_20260313_094058


In [2]:
def get_idx(lst, val):
    return lst.index(val)

def subsample(sigs, max_sig):
    if max_sig is None:
        return sigs
    return sigs[:min(len(sigs), max_sig)]

def get_signals(compact_dataset, tx, rx, date, equalized):
    tx_i = get_idx(compact_dataset["tx_list"], tx)
    rx_i = get_idx(compact_dataset["rx_list"], rx)
    date_i = get_idx(compact_dataset["capture_date_list"], date)
    eq_i = get_idx(compact_dataset["equalized_list"], equalized)
    sigs = compact_dataset["data"][tx_i][rx_i][date_i][eq_i]
    return np.array(sigs, dtype=np.float32)

def iq_to_complex(x):
    return (x[...,0] + 1j*x[...,1]).astype(np.complex64)

def domain_feat_256_complex(x256_c, d_dim=32):
    x = x256_c / (np.sqrt(np.mean(np.abs(x256_c)**2)) + 1e-12)
    X = np.fft.fft(x, n=256)
    mag = np.abs(X) + 1e-12
    logm = np.log(mag)
    D = np.fft.rfft(logm, n=512)
    return np.real(D[:d_dim]).astype(np.float32)

def compute_domain_feats_batch(X, d_dim=32):
    Xc = iq_to_complex(X)
    return np.stack([domain_feat_256_complex(Xc[i], d_dim=d_dim) for i in range(Xc.shape[0])], axis=0).astype(np.float32)

tx_list = compact_dataset["tx_list"]
rx_list = compact_dataset["rx_list"]
date_list = compact_dataset["capture_date_list"]

TX_USE = tx_list[:TX_TOTAL_USE]
RX_USE = rx_list[:RX_TOTAL_USE]
DATE_USE = date_list[:DAY_TOTAL_USE]

rng_rx = np.random.RandomState(SEED)
SOURCE_RXS = list(rng_rx.choice(RX_USE, size=SOURCE_RX_NUM, replace=False))
SOURCE_RXS.sort()
DRIFT_RX = [r for r in RX_USE if r not in SOURCE_RXS]

print("TX_USE:", TX_USE)
print("RX_USE:", RX_USE)
print("DATE_USE:", DATE_USE)
print("SOURCE_RXS:", SOURCE_RXS)
print("DRIFT_RX:", DRIFT_RX)

def build_unique_tx_splits(tx_use, known_tx_num, n_splits, seed=42):
    all_known = [tuple(c) for c in itertools.combinations(list(tx_use), known_tx_num)]
    if n_splits > len(all_known):
        raise ValueError(
            f"Requested TX_SPLIT_REPEATS={n_splits}, but only {len(all_known)} unique TX splits "
            f"exist for C({len(tx_use)},{known_tx_num}). Reduce TX_SPLIT_REPEATS or increase TX_TOTAL_USE."
        )
    rng = np.random.RandomState(seed)
    order = rng.permutation(len(all_known))
    tx_splits = []
    for idx in order[:n_splits]:
        known = list(all_known[idx])
        unknown = [tx for tx in tx_use if tx not in known]
        tx_splits.append((known, unknown))
    return tx_splits

TX_SPLITS = build_unique_tx_splits(TX_USE, KNOWN_TX_NUM, TX_SPLIT_REPEATS, seed=SEED + 777)
print("Unique TX split pool:", len(list(itertools.combinations(TX_USE, KNOWN_TX_NUM))))
print("Selected unique TX splits:", len(TX_SPLITS))
for i, (known, unknown) in enumerate(TX_SPLITS, start=1):
    print(f"TX_SPLIT_{i}: KNOWN={known} | UNKNOWN={unknown}")



TX_USE: ['14-10', '14-7', '20-15', '20-19', '6-15', '8-20']
RX_USE: ['1-1', '1-19', '14-7', '18-2', '19-2', '2-1', '2-19', '20-1', '3-19', '7-14', '7-7', '8-8']
DATE_USE: ['2021_03_01', '2021_03_08', '2021_03_15', '2021_03_23']
SOURCE_RXS: ['1-1', '7-14', '7-7']
DRIFT_RX: ['1-19', '14-7', '18-2', '19-2', '2-1', '2-19', '20-1', '3-19', '8-8']
Unique TX split pool: 15
Selected unique TX splits: 5
TX_SPLIT_1: KNOWN=['14-7', '20-19', '6-15', '8-20'] | UNKNOWN=['14-10', '20-15']
TX_SPLIT_2: KNOWN=['14-10', '14-7', '20-19', '6-15'] | UNKNOWN=['20-15', '8-20']
TX_SPLIT_3: KNOWN=['14-10', '14-7', '6-15', '8-20'] | UNKNOWN=['20-15', '20-19']
TX_SPLIT_4: KNOWN=['14-7', '20-15', '20-19', '6-15'] | UNKNOWN=['14-10', '8-20']
TX_SPLIT_5: KNOWN=['14-10', '14-7', '20-15', '8-20'] | UNKNOWN=['20-19', '6-15']


In [3]:
class BasicBlock1D(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1, downsample=None, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv1d(in_planes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(dropout)
        self.conv2 = nn.Conv1d(planes, planes, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(planes)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.dropout(out)
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        return self.relu(out + identity)

class ResNet18_1D(nn.Module):
    def __init__(self, num_classes, in_planes=64, dropout=0.0):
        super().__init__()
        self.in_planes = in_planes
        self.conv1 = nn.Conv1d(2, in_planes, 7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(in_planes)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool1d(3, stride=2, padding=1)
        self.layer1 = self._make_layer(64, 2, stride=1, dropout=dropout)
        self.layer2 = self._make_layer(128, 2, stride=2, dropout=dropout)
        self.layer3 = self._make_layer(256, 2, stride=2, dropout=dropout)
        self.layer4 = self._make_layer(512, 2, stride=2, dropout=dropout)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, planes, blocks, stride, dropout):
        downsample = None
        if stride != 1 or self.in_planes != planes:
            downsample = nn.Sequential(
                nn.Conv1d(self.in_planes, planes, 1, stride=stride, bias=False),
                nn.BatchNorm1d(planes)
            )
        layers = [BasicBlock1D(self.in_planes, planes, stride, downsample, dropout)]
        self.in_planes = planes
        for _ in range(1, blocks):
            layers.append(BasicBlock1D(self.in_planes, planes, dropout=dropout))
        return nn.Sequential(*layers)

    def forward(self, x, return_embed=False):
        x = x.permute(0, 2, 1)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x).squeeze(-1)
        logits = self.fc(x)
        if return_embed:
            return logits, x
        return logits



In [4]:
class ArrayDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, i):
        return self.X[i], self.y[i]

def run_epoch(model, loader, optimizer=None):
    train = optimizer is not None
    model.train(train)
    crit = nn.CrossEntropyLoss()
    total_loss, total_correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        if train:
            optimizer.zero_grad()
        logits = model(xb)
        loss = crit(logits, yb)
        if train:
            loss.backward()
            optimizer.step()
        total_loss += float(loss.item()) * yb.size(0)
        total_correct += int((logits.argmax(1) == yb).sum().item())
        total += int(yb.size(0))
    return total_loss/max(1,total), total_correct/max(1,total)

def infer_logits_embed(model, X_np, batch=512):
    model.eval()
    ds = ArrayDataset(X_np, np.zeros((len(X_np),), dtype=np.int64))
    dl = DataLoader(ds, batch_size=batch, shuffle=False)
    logits_all, Z_all = [], []
    with torch.no_grad():
        for xb,_ in dl:
            xb = xb.to(device)
            logits, emb = model(xb, return_embed=True)
            logits_all.append(logits.cpu().numpy().astype(np.float32))
            Z_all.append(emb.cpu().numpy().astype(np.float32))
    return np.concatenate(logits_all,0), np.concatenate(Z_all,0)

def closed_set_acc_from_logits(logits, y_true):
    return float((np.argmax(logits,1) == y_true).mean())

def roc_auc(y_true, score):
    y_true = np.asarray(y_true).reshape(-1)
    score = np.asarray(score, dtype=np.float64).reshape(-1)
    finite = np.isfinite(score)
    if finite.sum() < 2:
        return 0.5
    y_true = y_true[finite]
    score = score[finite]
    if np.unique(y_true).size < 2:
        return 0.5
    try:
        fpr, tpr, _ = roc_curve(y_true, score)
        return float(auc(fpr, tpr))
    except Exception:
        return 0.5



In [5]:
def fit_emb_maha_diag(Z_train, y_train, K):
    mu = np.zeros((K, Z_train.shape[1]), dtype=np.float32)
    var = np.zeros((K, Z_train.shape[1]), dtype=np.float32)
    for k in range(K):
        Zk = Z_train[y_train==k]
        mu[k] = Zk.mean(axis=0)
        var[k] = Zk.var(axis=0) + 1e-3
    return mu, var

def sid_embmaha(Z, mu_z, var_z):
    dif = Z[:,None,:] - mu_z[None,:,:]
    dist = np.sum((dif*dif)/(var_z[None,:,:] + 1e-6), axis=2)
    return (-np.min(dist, axis=1)).astype(np.float32)

def sid_energy(logits):
    logits = np.asarray(logits, dtype=np.float32)
    logits = np.nan_to_num(logits, nan=-50.0, posinf=50.0, neginf=-50.0)
    m = logits.max(axis=1, keepdims=True)
    out = m + np.log(np.exp(np.clip(logits - m, -60.0, 60.0)).sum(axis=1, keepdims=True) + 1e-12)
    return np.nan_to_num(out.squeeze(1), nan=-1e6, posinf=1e6, neginf=-1e6).astype(np.float32)

def zscore_fixed(x, ref):
    x = np.asarray(x, dtype=np.float32)
    ref = np.asarray(ref, dtype=np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=1e6, neginf=-1e6)
    ref = np.nan_to_num(ref, nan=0.0, posinf=1e6, neginf=-1e6)
    mu = float(np.mean(ref))
    std = float(np.std(ref))
    if not np.isfinite(std) or std < 1e-12:
        std = 1.0
    out = (x - mu) / std
    return np.nan_to_num(out, nan=0.0, posinf=1e6, neginf=-1e6).astype(np.float32)

def sid_fusion_fixed(Z, logits, mu_z, var_z, ref_sid_emb_A, ref_sid_en_A, lam=0.5):
    out = (zscore_fixed(sid_embmaha(Z, mu_z, var_z), ref_sid_emb_A) +
           lam * zscore_fixed(sid_energy(logits), ref_sid_en_A))
    return np.nan_to_num(out, nan=-1e6, posinf=1e6, neginf=-1e6).astype(np.float32)

def mahalanobis_batch(D, mu, Sinv):
    X = D - mu.reshape(1,-1)
    return np.einsum("nd,dd,nd->n", X, Sinv, X).astype(np.float32)

def fit_gaussian(D, reg=1e-3):
    mu = D.mean(axis=0).astype(np.float32)
    C = np.cov(D.T, bias=False).astype(np.float32)
    C = C + reg*np.eye(C.shape[0], dtype=np.float32)
    Sinv = np.linalg.inv(C).astype(np.float32)
    sign, logdet = np.linalg.slogdet(C)
    if sign <= 0:
        logdet = float(np.log(np.maximum(np.linalg.det(C), 1e-12)))
    return mu, Sinv, float(logdet)

def logpdf_gaussian(D, mu, Sinv, logdet):
    d = D.shape[1]
    maha = mahalanobis_batch(D, mu, Sinv)
    return (-0.5*(maha + logdet + d*np.log(2*np.pi))).astype(np.float32)

def fit_device_rx_models(D_train, y_train, rx_train, K, source_rx_ids, reg=1e-3, min_n=20):
    models = {}
    fallback = {}
    for k in range(K):
        fallback[k] = fit_gaussian(D_train[y_train==k], reg=reg)
        for rxid in source_rx_ids:
            idx = np.where((y_train==k) & (rx_train==rxid))[0]
            if idx.size >= min_n:
                models[(k,rxid)] = fit_gaussian(D_train[idx], reg=reg)
    return models, fallback

def sdom_V1_mixNLL(D_eval, k_hat, models_kr, fallback_k, source_rx_ids):
    N = D_eval.shape[0]
    out = np.zeros((N,), dtype=np.float32)
    R = len(source_rx_ids)
    for i in range(N):
        k = int(k_hat[i])
        d = D_eval[i:i+1]
        logps = []
        for rxid in source_rx_ids:
            mu,Sinv,logdet = models_kr.get((k,rxid), fallback_k[k])
            logps.append(float(logpdf_gaussian(d, mu, Sinv, logdet)[0]))
        logps = np.array(logps, dtype=np.float32)
        m = logps.max()
        loglik = m + np.log(np.exp(logps-m).sum()+1e-12) - np.log(R)
        out[i] = float(-loglik)
    return out

def calibrate_module2_thresholds(Sid_A, Sdom_A, frr_target=0.05, false_drift_target=0.05):
    tau_id = float(np.quantile(Sid_A, frr_target))
    tau_dom = float(np.quantile(Sdom_A, 1.0 - false_drift_target))
    return tau_id, tau_dom

def gate_predict(Sid, Sdom, tau_id, tau_dom):
    pred = np.zeros_like(Sid, dtype=np.int64)
    pred[Sid < tau_id] = 2
    pred[(Sid >= tau_id) & (Sdom > tau_dom)] = 1
    return pred

def compute_module2_scores_from_cached(Z, logits, D, mu_z_src, var_z_src,
                                       ref_sid_emb_A, ref_sid_en_A,
                                       models_kr, fallback_k, source_rx_ids,
                                       tau_id=None, tau_dom=None):
    logits = np.nan_to_num(np.asarray(logits, dtype=np.float32), nan=0.0, posinf=50.0, neginf=-50.0)
    Z = np.nan_to_num(np.asarray(Z, dtype=np.float32), nan=0.0, posinf=1e4, neginf=-1e4)
    D = np.nan_to_num(np.asarray(D, dtype=np.float32), nan=0.0, posinf=1e4, neginf=-1e4)
    k_hat = np.argmax(logits, axis=1).astype(np.int64)
    Sid = sid_fusion_fixed(Z, logits, mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A, lam=FUSION_LAM)
    Sdom = sdom_V1_mixNLL(D, k_hat, models_kr, fallback_k, source_rx_ids)
    Sid = np.nan_to_num(Sid, nan=-1e6, posinf=1e6, neginf=-1e6).astype(np.float32)
    Sdom = np.nan_to_num(Sdom, nan=1e6, posinf=1e6, neginf=-1e6).astype(np.float32)
    out = dict(Sid=Sid, Sdom=Sdom, k_hat=k_hat)
    if tau_id is not None and tau_dom is not None:
        out["gate_pred"] = gate_predict(Sid, Sdom, tau_id, tau_dom)
    return out



In [6]:
def softmax_np(logits):
    z = logits - logits.max(axis=1, keepdims=True)
    ez = np.exp(z)
    return ez / (ez.sum(axis=1, keepdims=True) + 1e-12)

def normalize_rows(P):
    P = np.asarray(P, dtype=np.float32)
    P = np.maximum(P, 1e-12)
    P = P / P.sum(axis=1, keepdims=True)
    return P.astype(np.float32)

def msp_from_probs(P):
    return P.max(axis=1).astype(np.float32)

def msp_from_logits(logits):
    return msp_from_probs(softmax_np(logits))

def onehot_from_labels(y, K):
    y = np.asarray(y, dtype=np.int64)
    out = np.zeros((len(y), K), dtype=np.float32)
    out[np.arange(len(y)), y] = 1.0
    return out

def pseudo_acc_from_probs(P, y_true):
    pred = np.argmax(P, axis=1)
    mask = y_true >= 0
    return float((pred[mask] == y_true[mask]).mean()) if np.any(mask) else float("nan")

def fit_class_prototypes(Z_train, y_train, K, l2norm=True):
    Z = Z_train.copy().astype(np.float32)
    if l2norm:
        Z = Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12)
    protos = np.zeros((K, Z.shape[1]), dtype=np.float32)
    for k in range(K):
        z = Z[y_train == k]
        protos[k] = z.mean(axis=0)
    if l2norm:
        protos = protos / (np.linalg.norm(protos, axis=1, keepdims=True) + 1e-12)
    return protos

def proto_probs_cosine(Z, protos, temp=0.10, l2norm=True):
    Zx = Z.astype(np.float32)
    if l2norm:
        Zx = Zx / (np.linalg.norm(Zx, axis=1, keepdims=True) + 1e-12)
    scores = (Zx @ protos.T) / max(temp, 1e-6)
    return softmax_np(scores)

def build_knn_memory(Z_train, y_train, per_class=300, seed=0):
    rng = np.random.RandomState(seed)
    keep = []
    for k in np.unique(y_train):
        idx = np.where(y_train == k)[0]
        rng.shuffle(idx)
        keep.append(idx[:min(len(idx), per_class)])
    keep = np.concatenate(keep, axis=0)
    Zm = Z_train[keep].astype(np.float32)
    ym = y_train[keep].astype(np.int64)
    Zm = Zm / (np.linalg.norm(Zm, axis=1, keepdims=True) + 1e-12)
    return Zm, ym

def knn_class_probs(Z, Zm, ym, K, topk=15):
    Zx = Z.astype(np.float32)
    Zx = Zx / (np.linalg.norm(Zx, axis=1, keepdims=True) + 1e-12)
    sims = Zx @ Zm.T
    topk = min(topk, Zm.shape[0])
    idx = np.argpartition(-sims, kth=topk-1, axis=1)[:, :topk]
    probs = np.zeros((Zx.shape[0], K), dtype=np.float32)
    for i in range(Zx.shape[0]):
        ii = idx[i]
        ss = sims[i, ii]
        ww = np.exp(ss - ss.max())
        for j, w in zip(ii, ww):
            probs[i, ym[j]] += float(w)
    return normalize_rows(probs)

def weighted_prob_fusion(prob_list, weights):
    out = np.zeros_like(prob_list[0], dtype=np.float32)
    for P, w in zip(prob_list, weights):
        out += float(w) * P.astype(np.float32)
    return normalize_rows(out)

def select_idx_by_mask_with_fallback(base_idx, mask_keep, conf, min_keep=64):
    base_idx = np.asarray(base_idx, dtype=np.int64)
    if len(base_idx) == 0:
        return base_idx
    idx_keep = base_idx[mask_keep]
    if len(idx_keep) >= min_keep:
        return idx_keep
    order = np.argsort(-conf[base_idx])
    k = min(len(base_idx), min_keep)
    return base_idx[order[:k]]

def normalize_conf_weights(conf):
    conf = np.asarray(conf, dtype=np.float32)
    if len(conf) == 0:
        return conf
    cmin, cmax = float(conf.min()), float(conf.max())
    if cmax - cmin < 1e-12:
        return np.ones_like(conf, dtype=np.float32)
    return (0.5 + 0.5 * (conf - cmin) / (cmax - cmin)).astype(np.float32)

def summarize_method_on_selected(idx_sel, y_true, P):
    if len(idx_sel) == 0:
        return dict(sel_size=0, pseudo_acc=float("nan"))
    mask = y_true[idx_sel] >= 0
    if np.any(mask):
        pa = float((np.argmax(P[idx_sel], axis=1)[mask] == y_true[idx_sel][mask]).mean())
    else:
        pa = float("nan")
    return dict(sel_size=int(len(idx_sel)), pseudo_acc=pa)



In [7]:
def build_source_train(compact_dataset, KNOWN_TX):
    X_list, y_list, D_list, rx_id_list = [], [], [], []
    for tx in KNOWN_TX:
        for rx in SOURCE_RXS:
            for day in TRAIN_DATES:
                Xz = subsample(get_signals(compact_dataset, tx, rx, day, Z_FROM_EQ), MAX_SIG_PER_TRIPLE)
                Xd = subsample(get_signals(compact_dataset, tx, rx, day, 0 if D_FROM=="raw" else 1), MAX_SIG_PER_TRIPLE)
                n = min(len(Xz), len(Xd))
                Xz = Xz[:n]; Xd = Xd[:n]
                y = np.full((n,), KNOWN_TX.index(tx), dtype=np.int64)
                Df = compute_domain_feats_batch(Xd, d_dim=D_DIM)
                X_list.append(Xz); y_list.append(y); D_list.append(Df)
                rx_id_list.append(np.full((n,), RX_USE.index(rx), dtype=np.int64))
    return (
        np.concatenate(X_list,0).astype(np.float32),
        np.concatenate(y_list,0).astype(np.int64),
        np.concatenate(D_list,0).astype(np.float32),
        np.concatenate(rx_id_list,0).astype(np.int64),
    )

def build_eval_set(compact_dataset, KNOWN_TX, txs, rxs, days):
    X_list, y_list, D_list = [], [], []
    for tx in txs:
        for rx in rxs:
            for day in days:
                Xz = subsample(get_signals(compact_dataset, tx, rx, day, Z_FROM_EQ), MAX_SIG_PER_TRIPLE)
                Xd = subsample(get_signals(compact_dataset, tx, rx, day, 0 if D_FROM=="raw" else 1), MAX_SIG_PER_TRIPLE)
                n = min(len(Xz), len(Xd))
                Xz = Xz[:n]; Xd = Xd[:n]
                y = np.full((n,), KNOWN_TX.index(tx), dtype=np.int64) if tx in KNOWN_TX else np.full((n,), -1, dtype=np.int64)
                Df = compute_domain_feats_batch(Xd, d_dim=D_DIM)
                X_list.append(Xz); y_list.append(y); D_list.append(Df)
    return (
        np.concatenate(X_list,0).astype(np.float32),
        np.concatenate(y_list,0).astype(np.int64),
        np.concatenate(D_list,0).astype(np.float32),
    )

def split_adapt_eval(X, y, D, frac=0.5, max_adapt=None, max_eval=None, seed=0):
    rng = np.random.RandomState(seed)
    idx = rng.permutation(len(X))
    cut = int(len(idx) * frac)
    idx_ad = idx[:cut]
    idx_ev = idx[cut:]
    if max_adapt is not None and len(idx_ad) > max_adapt:
        idx_ad = idx_ad[:max_adapt]
    if max_eval is not None and len(idx_ev) > max_eval:
        idx_ev = idx_ev[:max_eval]
    return X[idx_ad], y[idx_ad], D[idx_ad], X[idx_ev], y[idx_ev], D[idx_ev]

def concat_sets(items):
    X = np.concatenate([it[0] for it in items], axis=0)
    y = np.concatenate([it[1] for it in items], axis=0)
    D = np.concatenate([it[2] for it in items], axis=0)
    return X, y, D



In [8]:

class EmbDatasetWeightedHard(Dataset):
    def __init__(self, Z, y, w=None):
        self.Z = torch.tensor(Z, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        if w is None:
            w = np.ones((len(Z),), dtype=np.float32)
        self.w = torch.tensor(w, dtype=torch.float32)
    def __len__(self):
        return self.Z.shape[0]
    def __getitem__(self, i):
        return self.Z[i], self.y[i], self.w[i]

class EmbDatasetWeightedSoft(Dataset):
    def __init__(self, Z, q, w=None):
        self.Z = torch.tensor(Z, dtype=torch.float32)
        self.q = torch.tensor(q, dtype=torch.float32)
        if w is None:
            w = np.ones((len(Z),), dtype=np.float32)
        self.w = torch.tensor(w, dtype=torch.float32)
    def __len__(self):
        return self.Z.shape[0]
    def __getitem__(self, i):
        return self.Z[i], self.q[i], self.w[i]

class EmbOnlyDataset(Dataset):
    def __init__(self, Z):
        self.Z = torch.tensor(Z, dtype=torch.float32)
    def __len__(self):
        return self.Z.shape[0]
    def __getitem__(self, i):
        return self.Z[i]

class ResidualAdapter(nn.Module):
    def __init__(self, dim=512, bottleneck=64, scale=0.3):
        super().__init__()
        self.fc1 = nn.Linear(dim, bottleneck)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Linear(bottleneck, dim)
        self.scale = scale
        nn.init.zeros_(self.fc2.weight)
        nn.init.zeros_(self.fc2.bias)
    def forward(self, z):
        return z + self.scale * self.fc2(self.relu(self.fc1(z)))


def clone_state_dict_cpu(module):
    return {k: v.detach().cpu().clone() for k, v in module.state_dict().items()}


def apply_adapter_and_fc(adapter, fc_layer, Z_np, batch=512):
    fc = nn.Linear(fc_layer.in_features, fc_layer.out_features, bias=True)
    fc.load_state_dict(fc_layer.state_dict())
    fc.eval()
    if adapter is None:
        with torch.no_grad():
            z = torch.tensor(Z_np, dtype=torch.float32)
            logits = fc(z).numpy().astype(np.float32)
        z_np = np.nan_to_num(np.asarray(Z_np, dtype=np.float32), nan=0.0, posinf=1e4, neginf=-1e4)
        logits = np.nan_to_num(logits, nan=0.0, posinf=50.0, neginf=-50.0).astype(np.float32)
        return z_np, logits

    fc = fc.to(device)
    for p in fc.parameters():
        p.requires_grad = False
    adapter.eval()

    ds = EmbDatasetWeightedHard(Z_np, np.zeros((len(Z_np),), dtype=np.int64))
    dl = DataLoader(ds, batch_size=batch, shuffle=False)
    Z2_all, logits_all = [], []
    with torch.no_grad():
        for zb,_,_ in dl:
            zb = zb.to(device)
            z2 = adapter(zb)
            logits = fc(z2)
            Z2_all.append(z2.cpu().numpy().astype(np.float32))
            logits_all.append(logits.cpu().numpy().astype(np.float32))
    Z2 = np.concatenate(Z2_all,0)
    logits = np.concatenate(logits_all,0)
    Z2 = np.nan_to_num(Z2, nan=0.0, posinf=1e4, neginf=-1e4).astype(np.float32)
    logits = np.nan_to_num(logits, nan=0.0, posinf=50.0, neginf=-50.0).astype(np.float32)
    return Z2, logits


def make_replay_subset(Z_src, y_src, per_class=200, seed=0):
    rng = np.random.RandomState(seed)
    keep = []
    for k in np.unique(y_src):
        idx = np.where(y_src == k)[0]
        rng.shuffle(idx)
        keep.append(idx[:min(len(idx), per_class)])
    keep = np.concatenate(keep, axis=0)
    return Z_src[keep], y_src[keep]


def normalize_vec_rows_np(X):
    X = np.asarray(X, dtype=np.float32)
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)


def sigmoid_np(x):
    x = np.asarray(x, dtype=np.float32)
    x = np.clip(x, -30.0, 30.0)
    return 1.0 / (1.0 + np.exp(-x))


def robust_unit_interval(x, idx_ref=None):
    x = np.asarray(x, dtype=np.float32)
    if idx_ref is None or len(idx_ref) == 0:
        ref = x
    else:
        ref = x[np.asarray(idx_ref, dtype=np.int64)]
    lo = float(np.quantile(ref, 0.05))
    hi = float(np.quantile(ref, 0.95))
    if hi - lo < 1e-8:
        return np.zeros_like(x, dtype=np.float32)
    y = (x - lo) / (hi - lo)
    return np.clip(y, 0.0, 1.0).astype(np.float32)


def top2_margin(P):
    P = np.asarray(P, dtype=np.float32)
    if P.shape[1] < 2:
        return np.ones((P.shape[0],), dtype=np.float32)
    s = np.sort(P, axis=1)
    return (s[:, -1] - s[:, -2]).astype(np.float32)


def prototype_margin_dist(Z, protos):
    Zx = normalize_vec_rows_np(Z)
    Px = normalize_vec_rows_np(protos)
    sims = Zx @ Px.T
    order = np.argsort(-sims, axis=1)
    y1 = order[:, 0]
    s1 = sims[np.arange(len(Zx)), y1]
    if sims.shape[1] > 1:
        s2 = sims[np.arange(len(Zx)), order[:, 1]]
    else:
        s2 = np.zeros_like(s1)
    margin = (s1 - s2).astype(np.float32)
    dist = (1.0 - s1).astype(np.float32)
    return y1.astype(np.int64), margin, dist


def refine_probs_multi_stage(Z, P_seed, protos, idx_support=None, iters=2, src_mix=0.75):
    Zx = normalize_vec_rows_np(Z)
    P = normalize_rows(P_seed)
    Psrc = normalize_vec_rows_np(protos)
    if idx_support is None or len(idx_support) == 0:
        idx_support = np.arange(len(Zx))
    idx_support = np.asarray(idx_support, dtype=np.int64)
    for _ in range(max(1, int(iters))):
        Ps = normalize_rows(P[idx_support])
        Zs = Zx[idx_support]
        mass = Ps.sum(axis=0, keepdims=False)[:, None]
        proto_t = Psrc.copy()
        valid = mass.squeeze(1) > 1e-6
        if np.any(valid):
            proto_est = (Ps.T @ Zs) / np.maximum(mass, 1e-6)
            proto_t[valid] = normalize_vec_rows_np(src_mix * Psrc[valid] + (1.0 - src_mix) * proto_est[valid])
        P_proto = proto_probs_cosine(Zx, proto_t, temp=PROTO_T, l2norm=False)
        P = normalize_rows(0.60 * P + 0.40 * P_proto)
    return P.astype(np.float32)


def split_common_unknown_candidates(Z, P_logit, P_proto, P_knn, gate_pred, protos, tau_conf, min_keep=64):
    idx_gate = np.where(gate_pred == 1)[0]
    common_score = np.zeros((len(gate_pred),), dtype=np.float32)
    if len(idx_gate) == 0:
        return idx_gate, idx_gate, common_score, dict(threshold=float("nan"), gate_size=0)

    P_tri = weighted_prob_fusion([P_logit, P_proto, P_knn], [W_LOGIT, W_PROTO, W_KNN])
    y_logit = np.argmax(P_logit, axis=1)
    y_proto = np.argmax(P_proto, axis=1)
    y_knn = np.argmax(P_knn, axis=1)
    conf_tri = msp_from_probs(P_tri)
    margin_tri = top2_margin(P_tri)
    _, proto_margin, proto_dist = prototype_margin_dist(Z, protos)

    agree = ((y_logit == y_proto).astype(np.float32) + (y_logit == y_knn).astype(np.float32) + (y_proto == y_knn).astype(np.float32)) / 3.0
    margin_tri_n = robust_unit_interval(margin_tri, idx_gate)
    proto_margin_n = robust_unit_interval(proto_margin, idx_gate)
    proto_dist_n = robust_unit_interval(proto_dist, idx_gate)
    conf_n = robust_unit_interval(conf_tri, idx_gate)

    common_score = (
        0.40 * conf_n +
        0.25 * agree +
        0.20 * margin_tri_n +
        0.15 * proto_margin_n -
        0.15 * proto_dist_n
    ).astype(np.float32)

    thr_q = float(np.quantile(common_score[idx_gate], UNKNOWN_SCORE_Q))
    mask_common = (common_score >= thr_q) & (conf_tri >= min(float(tau_conf), float(np.max(conf_tri[idx_gate]))))
    idx_common = select_idx_by_mask_with_fallback(idx_gate, mask_common[idx_gate], common_score, min_keep=min_keep)
    idx_unknown = np.setdiff1d(idx_gate, idx_common, assume_unique=False)
    info = dict(threshold=thr_q, gate_size=int(len(idx_gate)), common_size=int(len(idx_common)), unknown_size=int(len(idx_unknown)))
    return idx_common.astype(np.int64), idx_unknown.astype(np.int64), common_score, info


def soft_ce_from_probs(logits, q):
    return -(q * F.log_softmax(logits, dim=1)).sum(dim=1)


def entropy_from_logits_torch(logits):
    p = torch.softmax(logits, dim=1)
    return -(p * torch.log(p + 1e-12)).sum(dim=1)


def symmetric_kl_from_logits(logits_a, logits_b):
    pa = torch.softmax(logits_a, dim=1)
    pb = torch.softmax(logits_b, dim=1)
    log_pa = torch.log(pa + 1e-12)
    log_pb = torch.log(pb + 1e-12)
    kl_ab = (pa * (log_pa - log_pb)).sum(dim=1)
    kl_ba = (pb * (log_pb - log_pa)).sum(dim=1)
    return 0.5 * (kl_ab + kl_ba)


def pairwise_sq_dists(x, y):
    x2 = (x * x).sum(dim=1, keepdim=True)
    y2 = (y * y).sum(dim=1, keepdim=True).T
    return torch.clamp(x2 + y2 - 2.0 * (x @ y.T), min=0.0)


def mmd_rbf(x, y, sigmas=None):
    if sigmas is None:
        sigmas = MMD_SIGMAS
    d_xx = pairwise_sq_dists(x, x)
    d_yy = pairwise_sq_dists(y, y)
    d_xy = pairwise_sq_dists(x, y)
    K_xx = 0.0
    K_yy = 0.0
    K_xy = 0.0
    for s in sigmas:
        gamma = 1.0 / max(2.0 * float(s) * float(s), 1e-12)
        K_xx = K_xx + torch.exp(-gamma * d_xx)
        K_yy = K_yy + torch.exp(-gamma * d_yy)
        K_xy = K_xy + torch.exp(-gamma * d_xy)
    return K_xx.mean() + K_yy.mean() - 2.0 * K_xy.mean()


def embedding_jitter(z, noise_std=EMB_AUG_NOISE):
    if noise_std <= 0:
        return z
    scale = z.detach().std(dim=0, keepdim=True).mean().clamp(min=1e-4)
    return z + noise_std * scale * torch.randn_like(z)


def proto_targets_torch(protos, y=None, q=None, device=device):
    P = torch.tensor(protos, dtype=torch.float32, device=device)
    P = F.normalize(P, dim=1)
    if y is not None:
        return P[y]
    if q is not None:
        q_t = torch.as_tensor(q, dtype=torch.float32, device=device)
        return q_t @ P
    raise ValueError("Either y or q must be provided for proto_targets_torch.")


def cycle_loader(dl):
    while True:
        for batch in dl:
            yield batch


def adapt_on_embeddings_generic(
    fc_layer,
    Z_replay,
    y_replay,
    Z_sup=None,
    y_sup=None,
    q_sup=None,
    w_sup=None,
    Z_align=None,
    Z_unrel=None,
    proto_bank=None,
    bottleneck=64,
    scale=0.3,
    epochs=20,
    lr=1e-3,
    wd=1e-4,
    batch=128,
    lambda_replay_ce=1.0,
    lambda_replay_anchor=1.0,
    lambda_sup=1.0,
    lambda_align=0.0,
    lambda_ent_min=0.0,
    lambda_ent_max=0.0,
    lambda_cons=0.0,
    lambda_proto=0.0,
    dynamic_align=False,
    init_state=None,
):
    has_sup = (Z_sup is not None) and (len(Z_sup) > 0)
    has_align = (Z_align is not None) and (len(Z_align) > 0)
    has_unrel = (Z_unrel is not None) and (len(Z_unrel) > 0)
    if (not has_sup) and (not has_align) and (not has_unrel):
        return None

    adapter = ResidualAdapter(dim=Z_replay.shape[1], bottleneck=bottleneck, scale=scale).to(device)
    if init_state is not None:
        adapter.load_state_dict(init_state)

    fc = nn.Linear(fc_layer.in_features, fc_layer.out_features, bias=True).to(device)
    fc.load_state_dict(fc_layer.state_dict())
    fc.eval()
    for p in fc.parameters():
        p.requires_grad = False

    ds_s = EmbDatasetWeightedHard(Z_replay, y_replay, None)
    dl_s = DataLoader(ds_s, batch_size=batch, shuffle=True, drop_last=False)
    it_s = cycle_loader(dl_s)

    dl_sup = None
    if has_sup:
        if y_sup is not None:
            ds_sup = EmbDatasetWeightedHard(Z_sup, y_sup, w_sup)
        else:
            ds_sup = EmbDatasetWeightedSoft(Z_sup, q_sup, w_sup)
        dl_sup = DataLoader(ds_sup, batch_size=batch, shuffle=True, drop_last=False)
        it_sup = cycle_loader(dl_sup)

    dl_align = None
    if has_align:
        ds_align = EmbOnlyDataset(Z_align)
        dl_align = DataLoader(ds_align, batch_size=batch, shuffle=True, drop_last=False)
        it_align = cycle_loader(dl_align)

    dl_unrel = None
    if has_unrel:
        ds_unrel = EmbOnlyDataset(Z_unrel)
        dl_unrel = DataLoader(ds_unrel, batch_size=batch, shuffle=True, drop_last=False)
        it_unrel = cycle_loader(dl_unrel)

    opt = optim.Adam(adapter.parameters(), lr=lr, weight_decay=wd)
    ce_none = nn.CrossEntropyLoss(reduction="none")
    ce = nn.CrossEntropyLoss()
    mse = nn.MSELoss()

    steps_per_epoch = max(
        len(dl_s),
        len(dl_sup) if dl_sup is not None else 0,
        len(dl_align) if dl_align is not None else 0,
        len(dl_unrel) if dl_unrel is not None else 0,
        1,
    )

    for ep in range(max(1, int(epochs))):
        adapter.train()
        lam_align_ep = float(lambda_align) * ((ep + 1) / max(1, int(epochs))) if dynamic_align else float(lambda_align)
        for _ in range(steps_per_epoch):
            zs, ys, _ = next(it_s)
            zs, ys = zs.to(device), ys.to(device)

            opt.zero_grad()
            zs2 = adapter(zs)
            logits_s = fc(zs2)
            loss = lambda_replay_ce * ce(logits_s, ys) + lambda_replay_anchor * mse(zs2, zs)

            if has_sup:
                batch_sup = next(it_sup)
                if y_sup is not None:
                    zsup, yb, wb = batch_sup
                    zsup, yb, wb = zsup.to(device), yb.to(device), wb.to(device)
                    zsup2 = adapter(zsup)
                    logits_sup = fc(zsup2)
                    loss = loss + lambda_sup * (ce_none(logits_sup, yb) * wb).mean()
                    if lambda_proto > 0 and proto_bank is not None:
                        tgt = proto_targets_torch(proto_bank, y=yb, device=device)
                        loss = loss + lambda_proto * ((F.normalize(zsup2, dim=1) - tgt) ** 2).mean()
                else:
                    zsup, qb, wb = batch_sup
                    zsup, qb, wb = zsup.to(device), qb.to(device), wb.to(device)
                    zsup2 = adapter(zsup)
                    logits_sup = fc(zsup2)
                    loss = loss + lambda_sup * (soft_ce_from_probs(logits_sup, qb) * wb).mean()
                    if lambda_proto > 0 and proto_bank is not None:
                        tgt = proto_targets_torch(proto_bank, q=qb, device=device)
                        loss = loss + lambda_proto * ((F.normalize(zsup2, dim=1) - tgt) ** 2).mean()

            if has_align:
                zt = next(it_align).to(device)
                zt2 = adapter(zt)
                logits_t = fc(zt2)
                if lam_align_ep > 0:
                    loss = loss + lam_align_ep * mmd_rbf(zs2, zt2)
                if lambda_ent_min > 0:
                    loss = loss + lambda_ent_min * entropy_from_logits_torch(logits_t).mean()
                if lambda_cons > 0:
                    logits_aug = fc(adapter(embedding_jitter(zt)))
                    loss = loss + lambda_cons * symmetric_kl_from_logits(logits_t, logits_aug).mean()

            if has_unrel and lambda_ent_max > 0:
                zu = next(it_unrel).to(device)
                logits_u = fc(adapter(zu))
                uniform = torch.full_like(logits_u, 1.0 / logits_u.shape[1])
                loss = loss + lambda_ent_max * F.kl_div(F.log_softmax(logits_u, dim=1), uniform, reduction="batchmean")

            loss.backward()
            opt.step()
    return adapter


def adapt_on_embeddings_hard(fc_layer, Z_target, y_target, Z_replay, y_replay,
                             target_weights=None,
                             bottleneck=64, scale=0.3, epochs=20, lr=1e-3, wd=1e-4, batch=128,
                             lambda_replay_ce=1.0, lambda_replay_anchor=1.0):
    if len(Z_target) == 0:
        return None
    return adapt_on_embeddings_generic(
        fc_layer=fc_layer,
        Z_replay=Z_replay, y_replay=y_replay,
        Z_sup=Z_target, y_sup=y_target, w_sup=target_weights,
        bottleneck=bottleneck, scale=scale, epochs=epochs, lr=lr, wd=wd, batch=batch,
        lambda_replay_ce=lambda_replay_ce, lambda_replay_anchor=lambda_replay_anchor,
    )


def adapt_on_embeddings_soft(fc_layer, Z_target, q_target, Z_replay, y_replay,
                             target_weights=None,
                             bottleneck=64, scale=0.3, epochs=20, lr=1e-3, wd=1e-4, batch=128,
                             lambda_replay_ce=1.0, lambda_replay_anchor=1.0):
    if len(Z_target) == 0:
        return None
    return adapt_on_embeddings_generic(
        fc_layer=fc_layer,
        Z_replay=Z_replay, y_replay=y_replay,
        Z_sup=Z_target, q_sup=q_target, w_sup=target_weights,
        bottleneck=bottleneck, scale=scale, epochs=epochs, lr=lr, wd=wd, batch=batch,
        lambda_replay_ce=lambda_replay_ce, lambda_replay_anchor=lambda_replay_anchor,
    )


def adapt_raincoat_lite(fc_layer, Z_target, idx_gate, idx_common_seed, idx_unknown_seed, Z_replay, y_replay,
                        protos, tau_conf, min_keep=64,
                        bottleneck=64, scale=0.3, epochs_stage1=8, epochs_stage2=12,
                        lr=1e-3, wd=1e-4, batch=128,
                        lambda_replay_ce=1.0, lambda_replay_anchor=1.0):
    if len(idx_gate) == 0:
        return None, dict(stage1_common_size=0, stage1_unknown_size=0)

    warm_adapter = adapt_on_embeddings_generic(
        fc_layer=fc_layer,
        Z_replay=Z_replay, y_replay=y_replay,
        Z_align=Z_target[idx_gate],
        proto_bank=protos,
        bottleneck=bottleneck, scale=scale, epochs=epochs_stage1,
        lr=lr, wd=wd, batch=batch,
        lambda_replay_ce=lambda_replay_ce, lambda_replay_anchor=lambda_replay_anchor,
        lambda_align=LAMBDA_ALIGN, lambda_ent_min=LAMBDA_ENT_MIN, lambda_cons=LAMBDA_CONS,
        dynamic_align=True,
    )

    if warm_adapter is None:
        return None, dict(stage1_common_size=0, stage1_unknown_size=0)

    Z_gate_1, logits_gate_1 = apply_adapter_and_fc(warm_adapter, fc_layer, Z_target[idx_gate], batch=512)
    P_logit_1 = softmax_np(logits_gate_1)
    P_proto_1 = proto_probs_cosine(Z_gate_1, protos, temp=PROTO_T, l2norm=True)
    P_knn_1 = P_proto_1.copy()
    gate_local = np.ones((len(idx_gate),), dtype=np.int64)
    common_local, unknown_local, _, split_info = split_common_unknown_candidates(
        Z_gate_1, P_logit_1, P_proto_1, P_knn_1, gate_local, protos, tau_conf=tau_conf, min_keep=min_keep
    )
    idx_common = idx_gate[common_local]
    idx_unknown = idx_gate[unknown_local]

    support_idx = idx_common if len(idx_common) > 0 else idx_common_seed
    P_seed = weighted_prob_fusion([P_logit_1, P_proto_1], [1.0, 1.0])
    P_refined = refine_probs_multi_stage(Z_gate_1, P_seed, protos, idx_support=common_local if len(common_local) > 0 else np.arange(len(idx_gate)), iters=REFINE_ITERS)
    w_common = normalize_conf_weights(msp_from_probs(P_refined[common_local])) if len(common_local) > 0 else None

    final_adapter = adapt_on_embeddings_generic(
        fc_layer=fc_layer,
        Z_replay=Z_replay, y_replay=y_replay,
        Z_sup=Z_target[idx_common] if len(idx_common) > 0 else None,
        q_sup=P_refined[common_local] if len(common_local) > 0 else None,
        w_sup=w_common,
        Z_align=Z_target[idx_gate],
        Z_unrel=Z_target[idx_unknown] if len(idx_unknown) > 0 else None,
        proto_bank=protos,
        bottleneck=bottleneck, scale=scale, epochs=epochs_stage2,
        lr=lr, wd=wd, batch=batch,
        lambda_replay_ce=lambda_replay_ce, lambda_replay_anchor=lambda_replay_anchor,
        lambda_align=LAMBDA_ALIGN, lambda_ent_min=LAMBDA_ENT_MIN * 0.5,
        lambda_ent_max=LAMBDA_ENT_MAX, lambda_cons=LAMBDA_CONS, lambda_proto=LAMBDA_PROTO,
        init_state=clone_state_dict_cpu(warm_adapter),
    )
    info = dict(
        stage1_common_size=int(len(idx_common)),
        stage1_unknown_size=int(len(idx_unknown)),
        seed_common_size=int(len(idx_common_seed)),
        seed_unknown_size=int(len(idx_unknown_seed)),
        split_threshold=float(split_info["threshold"]) if np.isfinite(split_info["threshold"]) else float("nan"),
    )
    return final_adapter, info


def plot_bar_compare(metric_dict, out_png, title):
    keys = list(metric_dict.keys())
    vals = [metric_dict[k] for k in keys]
    plt.figure(figsize=(max(8, 0.55*len(keys)), 4.2))
    plt.bar(np.arange(len(keys)), vals)
    plt.xticks(np.arange(len(keys)), keys, rotation=55, ha="right")
    plt.ylabel("value")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_png, dpi=160)
    plt.close()



def select_bottom_idx_by_mask_with_fallback(base_idx, mask_keep, score, min_keep=64):
    base_idx = np.asarray(base_idx, dtype=np.int64)
    if len(base_idx) == 0:
        return base_idx
    idx_keep = base_idx[mask_keep]
    if len(idx_keep) >= min_keep:
        return idx_keep
    order = np.argsort(score[base_idx])
    k = min(len(base_idx), min_keep)
    return base_idx[order[:k]]


def restrict_topk_probs(P, k=2, sharpen=1.0):
    P = normalize_rows(P)
    if P.shape[1] <= max(1, int(k)):
        if abs(float(sharpen) - 1.0) < 1e-8:
            return P.astype(np.float32)
        return normalize_rows(P ** float(sharpen)).astype(np.float32)
    k = max(1, int(k))
    out = np.zeros_like(P, dtype=np.float32)
    order = np.argsort(-P, axis=1)[:, :k]
    rows = np.arange(P.shape[0])[:, None]
    out[rows, order] = P[rows, order]
    if abs(float(sharpen) - 1.0) > 1e-8:
        out = out ** float(sharpen)
    return normalize_rows(out).astype(np.float32)


def build_three_bucket_partition(
    Z_adapt,
    gate_pred,
    Sid_adapt,
    Sdom_adapt,
    P_logit_adapt,
    P_proto_adapt,
    P_knn_adapt,
    protos,
    tau_conf,
    min_keep=64,
    reliable_keep_q=RELIABLE_KEEP_Q,
    ambig_keep_q=AMBIG_KEEP_Q,
    unknown_keep_q=UNKNOWN_KEEP_Q,
):
    idx_gate = np.where(gate_pred == 1)[0]
    n = len(gate_pred)
    empty = np.zeros((0,), dtype=np.int64)
    if len(idx_gate) == 0:
        aux = dict(
            bucket_score=np.zeros((n,), dtype=np.float32),
            unknown_score=np.zeros((n,), dtype=np.float32),
            P_seed=weighted_prob_fusion([P_logit_adapt, P_proto_adapt, P_knn_adapt], [W_LOGIT, W_PROTO, W_KNN]),
            threshold_reliable=float("nan"),
            threshold_unknown=float("nan"),
        )
        info = dict(reliable_size=0, ambiguous_size=0, unknown_size=0)
        return empty, empty, empty, aux, info

    P_tri = weighted_prob_fusion([P_logit_adapt, P_proto_adapt, P_knn_adapt], [W_LOGIT, W_PROTO, W_KNN])
    y_logit = np.argmax(P_logit_adapt, axis=1)
    y_proto = np.argmax(P_proto_adapt, axis=1)
    y_knn = np.argmax(P_knn_adapt, axis=1)

    conf_tri = msp_from_probs(P_tri)
    margin_tri = top2_margin(P_tri)
    _, proto_margin, proto_dist = prototype_margin_dist(Z_adapt, protos)

    agree = ((y_logit == y_proto).astype(np.float32) + (y_logit == y_knn).astype(np.float32) + (y_proto == y_knn).astype(np.float32)) / 3.0
    sid_n = robust_unit_interval(Sid_adapt, idx_gate)
    sdom_src_n = 1.0 - robust_unit_interval(Sdom_adapt, idx_gate)
    conf_n = robust_unit_interval(conf_tri, idx_gate)
    margin_n = robust_unit_interval(margin_tri, idx_gate)
    proto_margin_n = robust_unit_interval(proto_margin, idx_gate)
    proto_dist_n = robust_unit_interval(proto_dist, idx_gate)

    bucket_score = (
        0.30 * sid_n +
        0.22 * conf_n +
        0.16 * agree +
        0.12 * margin_n +
        0.12 * proto_margin_n +
        0.10 * sdom_src_n -
        0.10 * proto_dist_n
    ).astype(np.float32)

    unknown_score = (
        0.48 * (1.0 - bucket_score) +
        0.22 * proto_dist_n +
        0.18 * (1.0 - conf_n) +
        0.12 * (1.0 - margin_n)
    ).astype(np.float32)

    ng = len(idx_gate)
    min_rel = min(ng, max(8, min_keep))
    min_amb = min(max(0, ng - min_rel), max(16, min_keep))
    min_unk = min(max(0, ng - min_rel), max(8, min_keep // 2))

    k_rel = min(ng, max(int(round(float(reliable_keep_q) * ng)), min_rel))
    k_unk = min(max(0, ng - k_rel), max(int(round(float(unknown_keep_q) * ng)), min_unk))
    k_amb = min(max(0, ng - k_rel - k_unk), max(int(round(float(ambig_keep_q) * ng)), min_amb))
    if k_amb <= 0 and ng - k_rel - k_unk > 0:
        k_amb = ng - k_rel - k_unk

    order_rel = idx_gate[np.argsort(-bucket_score[idx_gate])]
    idx_rel = order_rel[:k_rel]

    cand_after_rel = np.setdiff1d(idx_gate, idx_rel, assume_unique=False)
    order_unk = cand_after_rel[np.argsort(-unknown_score[cand_after_rel])]
    idx_unk = order_unk[:k_unk]

    rem = np.setdiff1d(cand_after_rel, idx_unk, assume_unique=False)
    order_amb = rem[np.argsort(-bucket_score[rem])]
    idx_amb = order_amb[:k_amb]

    thr_rel = float(np.min(bucket_score[idx_rel])) if len(idx_rel) > 0 else float("nan")
    thr_unk = float(np.min(unknown_score[idx_unk])) if len(idx_unk) > 0 else float("nan")

    support_idx = np.concatenate([idx_rel, idx_amb], axis=0) if (len(idx_rel) + len(idx_amb)) > 0 else idx_gate
    P_seed = refine_probs_multi_stage(Z_adapt, P_tri, protos, idx_support=support_idx, iters=max(REFINE_ITERS, 2), src_mix=0.78)

    if len(idx_amb) > 0:
        P_seed[idx_amb] = restrict_topk_probs(P_seed[idx_amb], k=AMBIG_TOPK, sharpen=1.10)

    w_rel = np.asarray([], dtype=np.float32)
    if len(idx_rel) > 0:
        rel_score = 0.60 * bucket_score[idx_rel] + 0.40 * conf_tri[idx_rel]
        w_rel = RELIABLE_WEIGHT_FLOOR + (1.0 - RELIABLE_WEIGHT_FLOOR) * normalize_conf_weights(rel_score)

    w_amb = np.asarray([], dtype=np.float32)
    if len(idx_amb) > 0:
        amb_score = 0.50 * bucket_score[idx_amb] + 0.50 * conf_tri[idx_amb]
        w_amb = AMBIG_WEIGHT_FLOOR + AMBIG_WEIGHT_SCALE * normalize_conf_weights(amb_score)

    aux = dict(
        bucket_score=bucket_score.astype(np.float32),
        unknown_score=unknown_score.astype(np.float32),
        P_seed=P_seed.astype(np.float32),
        threshold_reliable=thr_rel,
        threshold_unknown=thr_unk,
        w_rel=w_rel.astype(np.float32),
        w_amb=w_amb.astype(np.float32),
    )
    info = dict(
        reliable_size=int(len(idx_rel)),
        ambiguous_size=int(len(idx_amb)),
        unknown_size=int(len(idx_unk)),
        reliable_keep=float(len(idx_rel) / max(1, len(gate_pred))),
        ambiguous_keep=float(len(idx_amb) / max(1, len(gate_pred))),
        unknown_keep=float(len(idx_unk) / max(1, len(gate_pred))),
        bucket_score_gate_mean=float(bucket_score[idx_gate].mean()),
        unknown_score_gate_mean=float(unknown_score[idx_gate].mean()),
        reliable_threshold=float(thr_rel),
        unknown_threshold=float(thr_unk),
    )
    return idx_rel.astype(np.int64), idx_amb.astype(np.int64), idx_unk.astype(np.int64), aux, info



def normalize_rows_torch(q):
    q = torch.clamp(q, min=1e-12)
    return q / q.sum(dim=1, keepdim=True)


def ema_update_(teacher, student, momentum=0.995):
    with torch.no_grad():
        for pt, ps in zip(teacher.parameters(), student.parameters()):
            pt.data.mul_(momentum).add_(ps.data, alpha=1.0 - momentum)


def proto_repulsion_loss(z, proto_bank, margin=UNKNOWN_REPEL_MARGIN):
    if proto_bank is None or len(proto_bank) == 0 or z.numel() == 0:
        return torch.tensor(0.0, device=z.device)
    P = torch.tensor(proto_bank, dtype=torch.float32, device=z.device)
    P = F.normalize(P, dim=1)
    sims = F.normalize(z, dim=1) @ P.T
    max_sim = sims.max(dim=1).values
    return torch.relu(max_sim - float(margin)).mean()


def mixed_teacher_targets(q_seed, teacher_logits=None, blend=EMA_TEACHER_BLEND, temp=1.0):
    q_seed = normalize_rows_torch(q_seed)
    if teacher_logits is None:
        return q_seed
    q_teacher = torch.softmax(teacher_logits / max(float(temp), 1e-6), dim=1)
    q_mix = float(blend) * q_teacher + (1.0 - float(blend)) * q_seed
    return normalize_rows_torch(q_mix)


def adapt_three_bucket_v5(
    fc_layer,
    Z_replay,
    y_replay,
    Z_target,
    idx_gate,
    idx_rel,
    idx_amb,
    idx_unk,
    q_seed,
    w_rel=None,
    w_amb=None,
    proto_bank=None,
    bottleneck=64,
    scale=0.3,
    epochs=20,
    lr=1e-3,
    wd=1e-4,
    batch=128,
    lambda_replay_ce=1.0,
    lambda_replay_anchor=1.0,
    lambda_rel_sup=1.0,
    lambda_amb_sup=LAMBDA_AMB_SUP,
    lambda_align=LAMBDA_ALIGN,
    lambda_ent_min=LAMBDA_ENT_MIN,
    lambda_ent_max=LAMBDA_ENT_MAX,
    lambda_cons=LAMBDA_CONS,
    lambda_proto=LAMBDA_PROTO,
    lambda_src_proto=LAMBDA_SRC_PROTO,
    lambda_src_logit=LAMBDA_SRC_LOGIT,
    lambda_u_repel=LAMBDA_UNKNOWN_REPEL,
    use_ema=True,
    ema_momentum=EMA_MOMENTUM,
    teacher_blend=EMA_TEACHER_BLEND,
    unknown_warmup_epochs=UNKNOWN_WARMUP_EPOCHS,
    unknown_ramp_epochs=UNKNOWN_RAMP_EPOCHS,
    init_state=None,
):
    has_rel = len(idx_rel) > 0
    has_amb = len(idx_amb) > 0
    has_unk = len(idx_unk) > 0
    has_align = len(idx_gate) > 0
    if not (has_rel or has_amb or has_unk or has_align):
        return None

    adapter = ResidualAdapter(dim=Z_replay.shape[1], bottleneck=bottleneck, scale=scale).to(device)
    if init_state is not None:
        adapter.load_state_dict(init_state)

    teacher = None
    if use_ema:
        teacher = ResidualAdapter(dim=Z_replay.shape[1], bottleneck=bottleneck, scale=scale).to(device)
        teacher.load_state_dict(adapter.state_dict())
        teacher.eval()
        for p in teacher.parameters():
            p.requires_grad = False

    fc = nn.Linear(fc_layer.in_features, fc_layer.out_features, bias=True).to(device)
    fc.load_state_dict(fc_layer.state_dict())
    fc.eval()
    for p in fc.parameters():
        p.requires_grad = False

    ds_s = EmbDatasetWeightedHard(Z_replay, y_replay, None)
    dl_s = DataLoader(ds_s, batch_size=batch, shuffle=True, drop_last=False)
    it_s = cycle_loader(dl_s)

    dl_rel = it_rel = None
    if has_rel:
        if w_rel is None or len(w_rel) == 0:
            w_rel = np.ones((len(idx_rel),), dtype=np.float32)
        ds_rel = EmbDatasetWeightedSoft(Z_target[idx_rel], q_seed[idx_rel], w_rel)
        dl_rel = DataLoader(ds_rel, batch_size=batch, shuffle=True, drop_last=False)
        it_rel = cycle_loader(dl_rel)

    dl_amb = it_amb = None
    if has_amb:
        if w_amb is None or len(w_amb) == 0:
            w_amb = np.ones((len(idx_amb),), dtype=np.float32) * AMBIG_WEIGHT_FLOOR
        ds_amb = EmbDatasetWeightedSoft(Z_target[idx_amb], q_seed[idx_amb], w_amb)
        dl_amb = DataLoader(ds_amb, batch_size=batch, shuffle=True, drop_last=False)
        it_amb = cycle_loader(dl_amb)

    dl_align = it_align = None
    if has_align:
        ds_align = EmbOnlyDataset(Z_target[idx_gate])
        dl_align = DataLoader(ds_align, batch_size=batch, shuffle=True, drop_last=False)
        it_align = cycle_loader(dl_align)

    dl_unk = it_unk = None
    if has_unk:
        ds_unk = EmbOnlyDataset(Z_target[idx_unk])
        dl_unk = DataLoader(ds_unk, batch_size=batch, shuffle=True, drop_last=False)
        it_unk = cycle_loader(dl_unk)

    opt = optim.Adam(adapter.parameters(), lr=lr, weight_decay=wd)
    ce = nn.CrossEntropyLoss()
    mse = nn.MSELoss()

    steps_per_epoch = max(
        len(dl_s),
        len(dl_rel) if dl_rel is not None else 0,
        len(dl_amb) if dl_amb is not None else 0,
        len(dl_align) if dl_align is not None else 0,
        len(dl_unk) if dl_unk is not None else 0,
        1,
    )

    src_proto_bank = None
    if proto_bank is not None:
        src_proto_bank = torch.tensor(proto_bank, dtype=torch.float32, device=device)
        src_proto_bank = F.normalize(src_proto_bank, dim=1)

    for ep in range(max(1, int(epochs))):
        adapter.train()
        lam_align_ep = float(lambda_align) * min(1.0, (ep + 1) / max(1, int(epochs)))
        if ep < int(unknown_warmup_epochs):
            unknown_scale = 0.0
        else:
            unknown_scale = min(1.0, float(ep - int(unknown_warmup_epochs) + 1) / max(1, int(unknown_ramp_epochs)))

        for _ in range(steps_per_epoch):
            zs, ys, _ = next(it_s)
            zs, ys = zs.to(device), ys.to(device)

            opt.zero_grad()
            zs2 = adapter(zs)
            logits_s = fc(zs2)
            with torch.no_grad():
                logits_s_ref = fc(zs)

            loss = lambda_replay_ce * ce(logits_s, ys)
            loss = loss + lambda_replay_anchor * mse(zs2, zs)

            if lambda_src_logit > 0:
                loss = loss + float(lambda_src_logit) * symmetric_kl_from_logits(logits_s, logits_s_ref).mean()

            if src_proto_bank is not None and lambda_src_proto > 0:
                tgt_src = src_proto_bank[ys]
                loss = loss + float(lambda_src_proto) * ((F.normalize(zs2, dim=1) - tgt_src) ** 2).mean()

            if has_rel:
                zrel, qrel_seed, wrel_t = next(it_rel)
                zrel = zrel.to(device); qrel_seed = qrel_seed.to(device); wrel_t = wrel_t.to(device)
                zrel2 = adapter(zrel)
                logits_rel = fc(zrel2)
                teacher_logits = None
                if teacher is not None:
                    with torch.no_grad():
                        teacher_logits = fc(teacher(zrel))
                qrel = mixed_teacher_targets(qrel_seed, teacher_logits, blend=teacher_blend)
                loss = loss + float(lambda_rel_sup) * (soft_ce_from_probs(logits_rel, qrel) * wrel_t).mean()
                if proto_bank is not None and lambda_proto > 0:
                    tgt_rel = proto_targets_torch(proto_bank, q=qrel, device=device)
                    loss = loss + float(lambda_proto) * ((F.normalize(zrel2, dim=1) - tgt_rel) ** 2).mean()

            if has_amb:
                zamb, qamb_seed, wamb_t = next(it_amb)
                zamb = zamb.to(device); qamb_seed = qamb_seed.to(device); wamb_t = wamb_t.to(device)
                zamb2 = adapter(zamb)
                logits_amb = fc(zamb2)
                teacher_logits = None
                if teacher is not None:
                    with torch.no_grad():
                        teacher_logits = fc(teacher(zamb))
                qamb = mixed_teacher_targets(qamb_seed, teacher_logits, blend=teacher_blend)
                loss = loss + float(lambda_amb_sup) * (soft_ce_from_probs(logits_amb, qamb) * wamb_t).mean()
                if lambda_cons > 0:
                    logits_amb_aug = fc(adapter(embedding_jitter(zamb)))
                    loss = loss + float(lambda_cons) * (symmetric_kl_from_logits(logits_amb, logits_amb_aug) * wamb_t).mean()
                if lambda_ent_min > 0:
                    loss = loss + float(lambda_ent_min) * (entropy_from_logits_torch(logits_amb) * wamb_t).mean()
                if proto_bank is not None and lambda_proto > 0:
                    tgt_amb = proto_targets_torch(proto_bank, q=qamb, device=device)
                    loss = loss + 0.5 * float(lambda_proto) * ((F.normalize(zamb2, dim=1) - tgt_amb) ** 2).mean()

            if has_align and lam_align_ep > 0:
                zt = next(it_align).to(device)
                zt2 = adapter(zt)
                loss = loss + lam_align_ep * mmd_rbf(zs2, zt2)

            if has_unk and unknown_scale > 0:
                zu = next(it_unk).to(device)
                zu2 = adapter(zu)
                logits_u = fc(zu2)
                if lambda_ent_max > 0:
                    uniform = torch.full_like(logits_u, 1.0 / logits_u.shape[1])
                    loss = loss + unknown_scale * float(lambda_ent_max) * F.kl_div(F.log_softmax(logits_u, dim=1), uniform, reduction="batchmean")
                if proto_bank is not None and lambda_u_repel > 0:
                    loss = loss + unknown_scale * float(lambda_u_repel) * proto_repulsion_loss(zu2, proto_bank, margin=UNKNOWN_REPEL_MARGIN)

            loss.backward()
            opt.step()
            if teacher is not None:
                ema_update_(teacher, adapter, momentum=ema_momentum)

    return adapter



def adapt_three_bucket_v4(*args, **kwargs):
    return adapt_three_bucket_v5(*args, **kwargs)



def select_topk_class_balanced(indices, pseudo_y, scores, k_total, K, min_per_class=1):
    indices = np.asarray(indices, dtype=np.int64)
    if len(indices) == 0 or k_total <= 0:
        return np.zeros((0,), dtype=np.int64)
    pseudo_y = np.asarray(pseudo_y, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float32)

    present = [k for k in range(int(K)) if np.any(pseudo_y[indices] == k)]
    if len(present) == 0:
        order = indices[np.argsort(-scores[indices])]
        return order[:min(len(order), int(k_total))].astype(np.int64)

    base = max(int(min_per_class), int(np.floor(float(k_total) / max(1, len(present)))))
    chosen = []
    for k in present:
        idx_k = indices[pseudo_y[indices] == k]
        if len(idx_k) == 0:
            continue
        order_k = idx_k[np.argsort(-scores[idx_k])]
        take = min(len(order_k), base)
        chosen.extend(order_k[:take].tolist())

    # fill the remaining budget globally by score
    chosen = np.array(sorted(set(chosen)), dtype=np.int64)
    if len(chosen) < int(k_total):
        rem = np.setdiff1d(indices, chosen, assume_unique=False)
        if len(rem) > 0:
            rem = rem[np.argsort(-scores[rem])]
            need = int(k_total) - len(chosen)
            chosen = np.concatenate([chosen, rem[:need]], axis=0)

    if len(chosen) > int(k_total):
        chosen = chosen[np.argsort(-scores[chosen])[:int(k_total)]]
    return chosen.astype(np.int64)


def build_three_bucket_partition_v6(
    K,
    Z_adapt,
    gate_pred,
    Sid_adapt,
    Sdom_adapt,
    P_logit_adapt,
    P_proto_adapt,
    P_knn_adapt,
    protos,
    tau_conf,
    min_keep=64,
):
    idx_gate = np.where(gate_pred == 1)[0]
    n = len(gate_pred)
    empty = np.zeros((0,), dtype=np.int64)
    if len(idx_gate) == 0:
        aux = dict(
            bucket_score=np.zeros((n,), dtype=np.float32),
            unknown_score=np.zeros((n,), dtype=np.float32),
            P_seed=weighted_prob_fusion([P_logit_adapt, P_proto_adapt, P_knn_adapt], [W_LOGIT, W_PROTO, W_KNN]),
            threshold_reliable=float("nan"),
            threshold_unknown=float("nan"),
            w_rel=np.zeros((0,), dtype=np.float32),
            w_amb=np.zeros((0,), dtype=np.float32),
            w_weak=np.zeros((0,), dtype=np.float32),
        )
        info = dict(
            reliable_size=0, ambiguous_size=0, weak_size=0, unknown_size=0,
            reliable_keep=0.0, ambiguous_keep=0.0, weak_keep=0.0, unknown_keep=0.0,
            bucket_score_gate_mean=float("nan"),
        )
        return empty, empty, empty, empty, aux, info

    P_tri = weighted_prob_fusion([P_logit_adapt, P_proto_adapt, P_knn_adapt], [W_LOGIT, W_PROTO, W_KNN])
    y_tri = np.argmax(P_tri, axis=1)
    y_logit = np.argmax(P_logit_adapt, axis=1)
    y_proto = np.argmax(P_proto_adapt, axis=1)
    y_knn = np.argmax(P_knn_adapt, axis=1)

    conf_tri = msp_from_probs(P_tri)
    margin_tri = top2_margin(P_tri)
    _, proto_margin, proto_dist = prototype_margin_dist(Z_adapt, protos)

    agree = ((y_logit == y_proto).astype(np.float32) + (y_logit == y_knn).astype(np.float32) + (y_proto == y_knn).astype(np.float32)) / 3.0
    sid_n = robust_unit_interval(Sid_adapt, idx_gate)
    sdom_src_n = 1.0 - robust_unit_interval(Sdom_adapt, idx_gate)
    conf_n = robust_unit_interval(conf_tri, idx_gate)
    margin_n = robust_unit_interval(margin_tri, idx_gate)
    proto_margin_n = robust_unit_interval(proto_margin, idx_gate)
    proto_dist_n = robust_unit_interval(proto_dist, idx_gate)

    bucket_score = (
        0.34 * sid_n +
        0.22 * conf_n +
        0.15 * agree +
        0.11 * margin_n +
        0.10 * proto_margin_n +
        0.08 * sdom_src_n -
        0.08 * proto_dist_n
    ).astype(np.float32)

    unknown_score = (
        0.42 * (1.0 - bucket_score) +
        0.24 * proto_dist_n +
        0.20 * (1.0 - conf_n) +
        0.14 * (1.0 - margin_n)
    ).astype(np.float32)

    ng = len(idx_gate)
    unk_gap = float(np.quantile(unknown_score[idx_gate], 0.90) - np.quantile(unknown_score[idx_gate], 0.60)) if ng >= 10 else 0.0
    unk_q = V6_UNKNOWN_KEEP_Q_HARD if unk_gap >= 0.08 else V6_UNKNOWN_KEEP_Q

    k_rel = min(ng, max(int(round(V6_RELIABLE_KEEP_Q * ng)), max(16, min_keep)))
    k_amb = min(max(0, ng - k_rel), max(int(round(V6_AMBIG_KEEP_Q * ng)), max(16, min_keep)))
    k_unk = min(max(0, ng - k_rel - k_amb), max(int(round(unk_q * ng)), max(8, min_keep // 3)))

    idx_rel = select_topk_class_balanced(idx_gate, y_tri, bucket_score, k_rel, K, min_per_class=1)

    rem1 = np.setdiff1d(idx_gate, idx_rel, assume_unique=False)
    idx_unk = rem1[np.argsort(-unknown_score[rem1])[:k_unk]] if len(rem1) > 0 and k_unk > 0 else np.zeros((0,), dtype=np.int64)

    rem2 = np.setdiff1d(rem1, idx_unk, assume_unique=False)
    idx_amb = select_topk_class_balanced(rem2, y_tri, bucket_score, k_amb, K, min_per_class=1)

    idx_weak = np.setdiff1d(rem2, idx_amb, assume_unique=False)
    if len(idx_weak) < min(V6_WEAK_MIN_KEEP, len(rem2)):
        need = min(V6_WEAK_MIN_KEEP, len(rem2))
        order = rem2[np.argsort(-bucket_score[rem2])]
        keep = order[:need]
        idx_weak = np.setdiff1d(keep, idx_amb, assume_unique=False)

    thr_rel = float(np.min(bucket_score[idx_rel])) if len(idx_rel) > 0 else float("nan")
    thr_unk = float(np.min(unknown_score[idx_unk])) if len(idx_unk) > 0 else float("nan")

    support_idx = np.concatenate([idx_rel, idx_amb], axis=0) if (len(idx_rel) + len(idx_amb)) > 0 else idx_gate
    P_seed = refine_probs_multi_stage(Z_adapt, P_tri, protos, idx_support=support_idx, iters=max(REFINE_ITERS, 3), src_mix=0.80)

    if len(idx_amb) > 0:
        P_seed[idx_amb] = restrict_topk_probs(P_seed[idx_amb], k=max(AMBIG_TOPK, 2), sharpen=1.05)
    if len(idx_weak) > 0:
        P_seed[idx_weak] = restrict_topk_probs(P_seed[idx_weak], k=max(V6_WEAK_TOPK, 2), sharpen=V6_WEAK_SHARPEN)

    w_rel = np.asarray([], dtype=np.float32)
    if len(idx_rel) > 0:
        rel_score = 0.60 * bucket_score[idx_rel] + 0.40 * conf_tri[idx_rel]
        w_rel = V6_REL_WEIGHT_FLOOR + (1.0 - V6_REL_WEIGHT_FLOOR) * normalize_conf_weights(rel_score)

    w_amb = np.asarray([], dtype=np.float32)
    if len(idx_amb) > 0:
        amb_score = 0.55 * bucket_score[idx_amb] + 0.45 * conf_tri[idx_amb]
        w_amb = V6_AMB_WEIGHT_FLOOR + V6_AMB_WEIGHT_SCALE * normalize_conf_weights(amb_score)

    w_weak = np.asarray([], dtype=np.float32)
    if len(idx_weak) > 0:
        weak_score = 0.50 * bucket_score[idx_weak] + 0.50 * conf_tri[idx_weak]
        w_weak = V6_WEAK_WEIGHT_FLOOR + V6_WEAK_WEIGHT_SCALE * normalize_conf_weights(weak_score)

    aux = dict(
        bucket_score=bucket_score.astype(np.float32),
        unknown_score=unknown_score.astype(np.float32),
        P_seed=P_seed.astype(np.float32),
        threshold_reliable=thr_rel,
        threshold_unknown=thr_unk,
        w_rel=w_rel.astype(np.float32),
        w_amb=w_amb.astype(np.float32),
        w_weak=w_weak.astype(np.float32),
    )
    info = dict(
        reliable_size=int(len(idx_rel)),
        ambiguous_size=int(len(idx_amb)),
        weak_size=int(len(idx_weak)),
        unknown_size=int(len(idx_unk)),
        reliable_keep=float(len(idx_rel) / max(1, len(gate_pred))),
        ambiguous_keep=float(len(idx_amb) / max(1, len(gate_pred))),
        weak_keep=float(len(idx_weak) / max(1, len(gate_pred))),
        unknown_keep=float(len(idx_unk) / max(1, len(gate_pred))),
        bucket_score_gate_mean=float(bucket_score[idx_gate].mean()),
        unknown_score_gate_mean=float(unknown_score[idx_gate].mean()),
        reliable_threshold=float(thr_rel),
        unknown_threshold=float(thr_unk),
        unknown_gap_q90_q60=float(unk_gap),
    )
    return idx_rel.astype(np.int64), idx_amb.astype(np.int64), idx_weak.astype(np.int64), idx_unk.astype(np.int64), aux, info


def adapt_three_bucket_v6(
    fc_layer,
    Z_replay,
    y_replay,
    Z_target,
    idx_gate,
    idx_rel,
    idx_amb,
    idx_weak,
    idx_unk,
    q_seed,
    w_rel=None,
    w_amb=None,
    w_weak=None,
    proto_bank=None,
    idx_weak_cold=None,
    w_weak_cold=None,
    bottleneck=64,
    scale=0.3,
    epochs=24,
    lr=1e-3,
    wd=1e-4,
    batch=128,
    lambda_replay_ce=1.0,
    lambda_replay_anchor=1.0,
    lambda_rel_sup=1.0,
    lambda_amb_sup=V6_LAMBDA_AMB_SUP,
    lambda_weak_sup=V6_LAMBDA_WEAK_SUP,
    lambda_align=V6_LAMBDA_ALIGN,
    lambda_ent_min=V6_LAMBDA_ENT_MIN,
    lambda_ent_max=V6_LAMBDA_ENT_MAX,
    lambda_cons=V6_LAMBDA_CONS,
    lambda_proto=V6_LAMBDA_PROTO,
    lambda_src_proto=LAMBDA_SRC_PROTO,
    lambda_src_logit=LAMBDA_SRC_LOGIT,
    lambda_u_repel=V6_LAMBDA_UNKNOWN_REPEL,
    use_ema=True,
    ema_momentum=EMA_MOMENTUM,
    teacher_blend=EMA_TEACHER_BLEND,
    weak_teacher_blend=V6_WEAK_TEACHER_BLEND,
    weak_start_epoch=V6_WEAK_START_EPOCH,
    unknown_warmup_epochs=V6_UNKNOWN_WARMUP_EPOCHS,
    unknown_ramp_epochs=V6_UNKNOWN_RAMP_EPOCHS,
    init_state=None,
):
    has_rel = len(idx_rel) > 0
    has_amb = len(idx_amb) > 0
    has_weak = len(idx_weak) > 0
    has_unk = len(idx_unk) > 0
    has_align = len(idx_gate) > 0
    if not (has_rel or has_amb or has_weak or has_unk or has_align):
        return None

    adapter = ResidualAdapter(dim=Z_replay.shape[1], bottleneck=bottleneck, scale=scale).to(device)
    if init_state is not None:
        adapter.load_state_dict(init_state)

    teacher = None
    if use_ema:
        teacher = ResidualAdapter(dim=Z_replay.shape[1], bottleneck=bottleneck, scale=scale).to(device)
        teacher.load_state_dict(adapter.state_dict())
        teacher.eval()
        for p in teacher.parameters():
            p.requires_grad = False

    fc = nn.Linear(fc_layer.in_features, fc_layer.out_features, bias=True).to(device)
    fc.load_state_dict(fc_layer.state_dict())
    fc.eval()
    for p in fc.parameters():
        p.requires_grad = False

    ds_s = EmbDatasetWeightedHard(Z_replay, y_replay, None)
    dl_s = DataLoader(ds_s, batch_size=batch, shuffle=True, drop_last=False)
    it_s = cycle_loader(dl_s)

    dl_rel = it_rel = None
    if has_rel:
        if w_rel is None or len(w_rel) == 0:
            w_rel = np.ones((len(idx_rel),), dtype=np.float32)
        ds_rel = EmbDatasetWeightedSoft(Z_target[idx_rel], q_seed[idx_rel], w_rel)
        dl_rel = DataLoader(ds_rel, batch_size=batch, shuffle=True, drop_last=False)
        it_rel = cycle_loader(dl_rel)

    dl_amb = it_amb = None
    if has_amb:
        if w_amb is None or len(w_amb) == 0:
            w_amb = np.ones((len(idx_amb),), dtype=np.float32) * V6_AMB_WEIGHT_FLOOR
        ds_amb = EmbDatasetWeightedSoft(Z_target[idx_amb], q_seed[idx_amb], w_amb)
        dl_amb = DataLoader(ds_amb, batch_size=batch, shuffle=True, drop_last=False)
        it_amb = cycle_loader(dl_amb)

    dl_weak = it_weak = None
    if has_weak:
        if w_weak is None or len(w_weak) == 0:
            w_weak = np.ones((len(idx_weak),), dtype=np.float32) * V6_WEAK_WEIGHT_FLOOR
        ds_weak = EmbDatasetWeightedSoft(Z_target[idx_weak], q_seed[idx_weak], w_weak)
        dl_weak = DataLoader(ds_weak, batch_size=batch, shuffle=True, drop_last=False)
        it_weak = cycle_loader(dl_weak)

    dl_align = it_align = None
    if has_align:
        ds_align = EmbOnlyDataset(Z_target[idx_gate])
        dl_align = DataLoader(ds_align, batch_size=batch, shuffle=True, drop_last=False)
        it_align = cycle_loader(dl_align)

    dl_unk = it_unk = None
    if has_unk:
        ds_unk = EmbOnlyDataset(Z_target[idx_unk])
        dl_unk = DataLoader(ds_unk, batch_size=batch, shuffle=True, drop_last=False)
        it_unk = cycle_loader(dl_unk)

    opt = optim.Adam(adapter.parameters(), lr=lr, weight_decay=wd)
    ce = nn.CrossEntropyLoss()
    mse = nn.MSELoss()

    steps_per_epoch = max(
        len(dl_s),
        len(dl_rel) if dl_rel is not None else 0,
        len(dl_amb) if dl_amb is not None else 0,
        len(dl_weak) if dl_weak is not None else 0,
        len(dl_align) if dl_align is not None else 0,
        len(dl_unk) if dl_unk is not None else 0,
        1,
    )

    src_proto_bank = None
    if proto_bank is not None:
        src_proto_bank = torch.tensor(proto_bank, dtype=torch.float32, device=device)
        src_proto_bank = F.normalize(src_proto_bank, dim=1)

    for ep in range(max(1, int(epochs))):
        adapter.train()
        lam_align_ep = float(lambda_align) * min(1.0, (ep + 1) / max(4, int(epochs) // 2))
        if ep < int(unknown_warmup_epochs):
            unknown_scale = 0.0
        else:
            unknown_scale = min(1.0, float(ep - int(unknown_warmup_epochs) + 1) / max(1, int(unknown_ramp_epochs)))

        for _ in range(steps_per_epoch):
            zs, ys, _ = next(it_s)
            zs, ys = zs.to(device), ys.to(device)

            opt.zero_grad()
            zs2 = adapter(zs)
            logits_s = fc(zs2)
            with torch.no_grad():
                logits_s_ref = fc(zs)

            loss = lambda_replay_ce * ce(logits_s, ys)
            loss = loss + lambda_replay_anchor * mse(zs2, zs)

            if lambda_src_logit > 0:
                loss = loss + float(lambda_src_logit) * symmetric_kl_from_logits(logits_s, logits_s_ref).mean()

            if src_proto_bank is not None and lambda_src_proto > 0:
                tgt_src = src_proto_bank[ys]
                loss = loss + float(lambda_src_proto) * ((F.normalize(zs2, dim=1) - tgt_src) ** 2).mean()

            if has_rel:
                zrel, qrel_seed, wrel_t = next(it_rel)
                zrel = zrel.to(device); qrel_seed = qrel_seed.to(device); wrel_t = wrel_t.to(device)
                zrel2 = adapter(zrel)
                logits_rel = fc(zrel2)
                teacher_logits = None
                if teacher is not None:
                    with torch.no_grad():
                        teacher_logits = fc(teacher(zrel))
                qrel = mixed_teacher_targets(qrel_seed, teacher_logits, blend=teacher_blend)
                loss = loss + float(lambda_rel_sup) * (soft_ce_from_probs(logits_rel, qrel) * wrel_t).mean()
                if proto_bank is not None and lambda_proto > 0:
                    tgt_rel = proto_targets_torch(proto_bank, q=qrel, device=device)
                    loss = loss + float(lambda_proto) * ((F.normalize(zrel2, dim=1) - tgt_rel) ** 2).mean()

            if has_amb:
                zamb, qamb_seed, wamb_t = next(it_amb)
                zamb = zamb.to(device); qamb_seed = qamb_seed.to(device); wamb_t = wamb_t.to(device)
                zamb2 = adapter(zamb)
                logits_amb = fc(zamb2)
                teacher_logits = None
                if teacher is not None:
                    with torch.no_grad():
                        teacher_logits = fc(teacher(zamb))
                qamb = mixed_teacher_targets(qamb_seed, teacher_logits, blend=teacher_blend)
                loss = loss + float(lambda_amb_sup) * (soft_ce_from_probs(logits_amb, qamb) * wamb_t).mean()
                if lambda_cons > 0:
                    logits_amb_aug = fc(adapter(embedding_jitter(zamb)))
                    loss = loss + float(lambda_cons) * (symmetric_kl_from_logits(logits_amb, logits_amb_aug) * wamb_t).mean()
                if lambda_ent_min > 0:
                    loss = loss + float(lambda_ent_min) * (entropy_from_logits_torch(logits_amb) * wamb_t).mean()

            if has_weak and ep >= int(weak_start_epoch):
                zweak, qweak_seed, wweak_t = next(it_weak)
                zweak = zweak.to(device); qweak_seed = qweak_seed.to(device); wweak_t = wweak_t.to(device)
                zweak2 = adapter(zweak)
                logits_weak = fc(zweak2)
                teacher_logits = None
                if teacher is not None:
                    with torch.no_grad():
                        teacher_logits = fc(teacher(zweak))
                qweak = mixed_teacher_targets(qweak_seed, teacher_logits, blend=weak_teacher_blend)
                loss = loss + float(lambda_weak_sup) * (soft_ce_from_probs(logits_weak, qweak) * wweak_t).mean()
                if lambda_cons > 0:
                    logits_weak_aug = fc(adapter(embedding_jitter(zweak)))
                    loss = loss + 0.5 * float(lambda_cons) * (symmetric_kl_from_logits(logits_weak, logits_weak_aug) * wweak_t).mean()

            if has_align and lam_align_ep > 0:
                zt = next(it_align).to(device)
                zt2 = adapter(zt)
                loss = loss + lam_align_ep * mmd_rbf(zs2, zt2)

            if has_unk and unknown_scale > 0:
                zu = next(it_unk).to(device)
                zu2 = adapter(zu)
                logits_u = fc(zu2)
                if lambda_ent_max > 0:
                    uniform = torch.full_like(logits_u, 1.0 / logits_u.shape[1])
                    loss = loss + unknown_scale * float(lambda_ent_max) * F.kl_div(F.log_softmax(logits_u, dim=1), uniform, reduction="batchmean")
                if proto_bank is not None and lambda_u_repel > 0:
                    loss = loss + unknown_scale * float(lambda_u_repel) * proto_repulsion_loss(zu2, proto_bank, margin=UNKNOWN_REPEL_MARGIN)

            loss.backward()
            opt.step()
            if teacher is not None:
                ema_update_(teacher, adapter, momentum=ema_momentum)

    return teacher if (teacher is not None) else adapter



def build_three_bucket_partition_v7(
    K,
    Z_adapt,
    gate_pred,
    Sid_adapt,
    Sdom_adapt,
    P_logit_adapt,
    P_proto_adapt,
    P_knn_adapt,
    protos,
    tau_conf,
    min_keep=64,
):
    idx_gate = np.where(gate_pred == 1)[0]
    n = len(gate_pred)
    empty = np.zeros((0,), dtype=np.int64)
    if len(idx_gate) == 0:
        aux = dict(
            bucket_score=np.zeros((n,), dtype=np.float32),
            unknown_score=np.zeros((n,), dtype=np.float32),
            P_seed=weighted_prob_fusion([P_logit_adapt, P_proto_adapt, P_knn_adapt], [W_LOGIT, W_PROTO, W_KNN]),
            threshold_reliable=float("nan"),
            threshold_unknown=float("nan"),
            w_rel=np.zeros((0,), dtype=np.float32),
            w_amb=np.zeros((0,), dtype=np.float32),
            w_weak=np.zeros((0,), dtype=np.float32),
        )
        info = dict(
            reliable_size=0, ambiguous_size=0, weak_size=0, unknown_size=0,
            reliable_keep=0.0, ambiguous_keep=0.0, weak_keep=0.0, unknown_keep=0.0,
            bucket_score_gate_mean=float("nan"),
        )
        return empty, empty, empty, empty, aux, info

    P_tri = weighted_prob_fusion([P_logit_adapt, P_proto_adapt, P_knn_adapt], [W_LOGIT, W_PROTO, W_KNN])
    y_tri = np.argmax(P_tri, axis=1)
    y_logit = np.argmax(P_logit_adapt, axis=1)
    y_proto = np.argmax(P_proto_adapt, axis=1)
    y_knn = np.argmax(P_knn_adapt, axis=1)

    conf_tri = msp_from_probs(P_tri)
    margin_tri = top2_margin(P_tri)
    _, proto_margin, proto_dist = prototype_margin_dist(Z_adapt, protos)

    agree = ((y_logit == y_proto).astype(np.float32) + (y_logit == y_knn).astype(np.float32) + (y_proto == y_knn).astype(np.float32)) / 3.0
    sid_n = robust_unit_interval(Sid_adapt, idx_gate)
    sdom_src_n = 1.0 - robust_unit_interval(Sdom_adapt, idx_gate)
    conf_n = robust_unit_interval(conf_tri, idx_gate)
    margin_n = robust_unit_interval(margin_tri, idx_gate)
    proto_margin_n = robust_unit_interval(proto_margin, idx_gate)
    proto_dist_n = robust_unit_interval(proto_dist, idx_gate)

    bucket_score = (
        0.36 * sid_n +
        0.22 * conf_n +
        0.14 * agree +
        0.12 * margin_n +
        0.10 * proto_margin_n +
        0.08 * sdom_src_n -
        0.08 * proto_dist_n
    ).astype(np.float32)

    unknown_score = (
        0.40 * (1.0 - bucket_score) +
        0.24 * proto_dist_n +
        0.20 * (1.0 - conf_n) +
        0.16 * (1.0 - margin_n)
    ).astype(np.float32)

    ng = len(idx_gate)
    unk_gap = float(np.quantile(unknown_score[idx_gate], 0.90) - np.quantile(unknown_score[idx_gate], 0.60)) if ng >= 10 else 0.0
    unk_q = V7_UNKNOWN_KEEP_Q_HARD if unk_gap >= 0.08 else V7_UNKNOWN_KEEP_Q

    k_rel = min(ng, max(int(round(V7_RELIABLE_KEEP_Q * ng)), max(16, min_keep // 2)))
    k_amb = min(max(0, ng - k_rel), max(int(round(V7_AMBIG_KEEP_Q * ng)), max(16, min_keep // 2)))
    k_unk = min(max(0, ng - k_rel - k_amb), max(int(round(unk_q * ng)), max(8, min_keep // 4)))

    idx_rel = select_topk_class_balanced(idx_gate, y_tri, bucket_score, k_rel, K, min_per_class=1)
    rem1 = np.setdiff1d(idx_gate, idx_rel, assume_unique=False)
    idx_unk = rem1[np.argsort(-unknown_score[rem1])[:k_unk]] if len(rem1) > 0 and k_unk > 0 else np.zeros((0,), dtype=np.int64)

    rem2 = np.setdiff1d(rem1, idx_unk, assume_unique=False)
    idx_amb = select_topk_class_balanced(rem2, y_tri, bucket_score, k_amb, K, min_per_class=1)

    idx_weak = np.setdiff1d(rem2, idx_amb, assume_unique=False)
    if len(idx_weak) < min(V7_WEAK_MIN_KEEP, len(rem2)):
        need = min(V7_WEAK_MIN_KEEP, len(rem2))
        order = rem2[np.argsort(-bucket_score[rem2])]
        keep = order[:need]
        idx_weak = np.setdiff1d(keep, idx_amb, assume_unique=False)

    support_idx = np.concatenate([idx_rel, idx_amb], axis=0) if (len(idx_rel) + len(idx_amb)) > 0 else idx_gate
    P_seed = refine_probs_multi_stage(Z_adapt, P_tri, protos, idx_support=support_idx, iters=max(REFINE_ITERS, 3), src_mix=0.82)
    if len(idx_amb) > 0:
        P_seed[idx_amb] = restrict_topk_probs(P_seed[idx_amb], k=max(AMBIG_TOPK, 2), sharpen=1.02)
    if len(idx_weak) > 0:
        P_seed[idx_weak] = restrict_topk_probs(P_seed[idx_weak], k=max(V7_WEAK_TOPK, 2), sharpen=V7_WEAK_SHARPEN)

    w_rel = np.asarray([], dtype=np.float32)
    if len(idx_rel) > 0:
        rel_score = 0.62 * bucket_score[idx_rel] + 0.38 * conf_tri[idx_rel]
        w_rel = V7_REL_WEIGHT_FLOOR + (1.0 - V7_REL_WEIGHT_FLOOR) * normalize_conf_weights(rel_score)

    w_amb = np.asarray([], dtype=np.float32)
    if len(idx_amb) > 0:
        amb_score = 0.58 * bucket_score[idx_amb] + 0.42 * conf_tri[idx_amb]
        w_amb = V7_AMB_WEIGHT_FLOOR + V7_AMB_WEIGHT_SCALE * normalize_conf_weights(amb_score)

    w_weak = np.asarray([], dtype=np.float32)
    if len(idx_weak) > 0:
        weak_score = 0.50 * bucket_score[idx_weak] + 0.50 * conf_tri[idx_weak]
        w_weak = V7_WEAK_WEIGHT_FLOOR + V7_WEAK_WEIGHT_SCALE * normalize_conf_weights(weak_score)

    aux = dict(
        bucket_score=bucket_score.astype(np.float32),
        unknown_score=unknown_score.astype(np.float32),
        P_seed=P_seed.astype(np.float32),
        threshold_reliable=float(np.min(bucket_score[idx_rel])) if len(idx_rel) > 0 else float("nan"),
        threshold_unknown=float(np.min(unknown_score[idx_unk])) if len(idx_unk) > 0 else float("nan"),
        w_rel=w_rel.astype(np.float32),
        w_amb=w_amb.astype(np.float32),
        w_weak=w_weak.astype(np.float32),
    )
    info = dict(
        reliable_size=int(len(idx_rel)),
        ambiguous_size=int(len(idx_amb)),
        weak_size=int(len(idx_weak)),
        unknown_size=int(len(idx_unk)),
        reliable_keep=float(len(idx_rel) / max(1, len(gate_pred))),
        ambiguous_keep=float(len(idx_amb) / max(1, len(gate_pred))),
        weak_keep=float(len(idx_weak) / max(1, len(gate_pred))),
        unknown_keep=float(len(idx_unk) / max(1, len(gate_pred))),
        bucket_score_gate_mean=float(bucket_score[idx_gate].mean()),
        unknown_score_gate_mean=float(unknown_score[idx_gate].mean()),
    )
    return idx_rel.astype(np.int64), idx_amb.astype(np.int64), idx_weak.astype(np.int64), idx_unk.astype(np.int64), aux, info


def adapt_three_bucket_v7(
    fc_layer,
    Z_replay,
    y_replay,
    Z_target,
    idx_gate,
    idx_rel,
    idx_amb,
    idx_weak,
    idx_unk,
    q_seed,
    w_rel=None,
    w_amb=None,
    w_weak=None,
    proto_bank=None,
    bucket_score=None,
    unknown_score=None,
    idx_weak_cold=None,
    w_weak_cold=None,
    bottleneck=64,
    scale=0.3,
    epochs=24,
    stage1_epochs=V7_STAGE1_EPOCHS,
    lr=1e-3,
    wd=1e-4,
    batch=128,
    lambda_replay_ce=1.0,
    lambda_replay_anchor=1.0,
    lambda_rel_sup=1.0,
    lambda_amb_sup=V7_LAMBDA_AMB_SUP,
    lambda_weak_sup=V7_LAMBDA_WEAK_SUP,
    lambda_align=V7_LAMBDA_ALIGN,
    lambda_ent_min=V7_LAMBDA_ENT_MIN,
    lambda_ent_max=V7_LAMBDA_ENT_MAX,
    lambda_cons=V7_LAMBDA_CONS,
    lambda_proto=V7_LAMBDA_PROTO,
    lambda_src_proto=LAMBDA_SRC_PROTO,
    lambda_src_logit=LAMBDA_SRC_LOGIT,
    lambda_u_repel=V7_LAMBDA_UNKNOWN_REPEL,
    use_ema=True,
    ema_momentum=EMA_MOMENTUM,
    teacher_blend=EMA_TEACHER_BLEND,
    promote_blend=V7_PROMOTE_BLEND,
    promote_rel_fraction=V7_PROMOTE_REL_FRACTION,
    promote_amb_fraction=V7_PROMOTE_AMB_FRACTION,
    promote_conf=V7_PROMOTE_CONF,
    promote_margin=V7_PROMOTE_MARGIN,
    unknown_warmup_epochs=V7_UNKNOWN_WARMUP_EPOCHS,
    unknown_ramp_epochs=V7_UNKNOWN_RAMP_EPOCHS,
    init_state=None,
):
    if (len(idx_gate) + len(idx_rel) + len(idx_amb) + len(idx_weak) + len(idx_unk)) == 0:
        return None, {}

    K = q_seed.shape[1]
    empty = np.zeros((0,), dtype=np.int64)

    adapter = ResidualAdapter(dim=Z_replay.shape[1], bottleneck=bottleneck, scale=scale).to(device)
    if init_state is not None:
        adapter.load_state_dict(init_state)

    teacher = None
    if use_ema:
        teacher = ResidualAdapter(dim=Z_replay.shape[1], bottleneck=bottleneck, scale=scale).to(device)
        teacher.load_state_dict(adapter.state_dict())
        teacher.eval()
        for p in teacher.parameters():
            p.requires_grad = False

    fc = nn.Linear(fc_layer.in_features, fc_layer.out_features, bias=True).to(device)
    fc.load_state_dict(fc_layer.state_dict())
    fc.eval()
    for p in fc.parameters():
        p.requires_grad = False

    ds_s = EmbDatasetWeightedHard(Z_replay, y_replay, None)
    dl_s = DataLoader(ds_s, batch_size=batch, shuffle=True, drop_last=False)
    it_s = cycle_loader(dl_s)

    opt = optim.Adam(adapter.parameters(), lr=lr, weight_decay=wd)
    ce = nn.CrossEntropyLoss()
    mse = nn.MSELoss()

    src_proto_bank = normalize_vec_rows_np(proto_bank).astype(np.float32) if proto_bank is not None else None
    if bucket_score is None:
        bucket_score = np.zeros((len(Z_target),), dtype=np.float32)

    q_curr = normalize_rows(q_seed.astype(np.float32).copy())
    idx_rel_curr = np.asarray(idx_rel, dtype=np.int64)
    idx_amb_curr = np.asarray(idx_amb, dtype=np.int64)
    idx_weak_curr = np.asarray(idx_weak, dtype=np.int64)
    idx_unk_curr = np.asarray(idx_unk, dtype=np.int64)

    w_rel_curr = np.ones((len(idx_rel_curr),), dtype=np.float32) if (w_rel is None or len(w_rel) == 0) else np.asarray(w_rel, dtype=np.float32)
    w_amb_curr = np.ones((len(idx_amb_curr),), dtype=np.float32) * V7_AMB_WEIGHT_FLOOR if (w_amb is None or len(w_amb) == 0) else np.asarray(w_amb, dtype=np.float32)
    w_weak_curr = np.ones((len(idx_weak_curr),), dtype=np.float32) * V7_WEAK_WEIGHT_FLOOR if (w_weak is None or len(w_weak) == 0) else np.asarray(w_weak, dtype=np.float32)

    def _make_soft_loader(indices, weights):
        if len(indices) == 0:
            return None, None
        ds = EmbDatasetWeightedSoft(Z_target[indices], q_curr[indices], weights)
        dl = DataLoader(ds, batch_size=batch, shuffle=True, drop_last=False)
        return dl, cycle_loader(dl)

    def _make_align_loader(indices):
        if len(indices) == 0:
            return None, None
        ds = EmbOnlyDataset(Z_target[indices])
        dl = DataLoader(ds, batch_size=batch, shuffle=True, drop_last=False)
        return dl, cycle_loader(dl)

    def _target_proto_from_reliable(adapter_ref):
        if src_proto_bank is None:
            return None
        proto_np = src_proto_bank.copy()
        if len(idx_rel_curr) == 0:
            return proto_np
        Z_rel2, _ = apply_adapter_and_fc(adapter_ref, fc, Z_target[idx_rel_curr], batch=batch)
        Z_rel2 = normalize_vec_rows_np(Z_rel2)
        q_rel = normalize_rows(q_curr[idx_rel_curr])
        mass = q_rel.sum(axis=0, keepdims=False)[:, None]
        valid = mass.squeeze(1) > 1e-6
        if np.any(valid):
            proto_est = (q_rel.T @ Z_rel2) / np.maximum(mass, 1e-6)
            proto_est = normalize_vec_rows_np(proto_est)
            proto_np[valid] = proto_est[valid]
        return proto_np.astype(np.float32)

    def _train_range(ep_start, ep_end, use_unknown=False, use_weak=False):
        nonlocal adapter, teacher, idx_rel_curr, idx_amb_curr, idx_weak_curr, idx_unk_curr, w_rel_curr, w_amb_curr, w_weak_curr
        dl_rel, it_rel = _make_soft_loader(idx_rel_curr, w_rel_curr)
        dl_amb, it_amb = _make_soft_loader(idx_amb_curr, w_amb_curr)
        dl_weak, it_weak = _make_soft_loader(idx_weak_curr, w_weak_curr)
        dl_align, it_align = _make_align_loader(np.asarray(idx_gate, dtype=np.int64))
        dl_unk, it_unk = _make_align_loader(idx_unk_curr if use_unknown else empty)

        steps_per_epoch = max(
            len(dl_s),
            len(dl_rel) if dl_rel is not None else 0,
            len(dl_amb) if dl_amb is not None else 0,
            len(dl_weak) if (dl_weak is not None and use_weak) else 0,
            len(dl_align) if dl_align is not None else 0,
            len(dl_unk) if dl_unk is not None else 0,
            1,
        )

        for ep in range(int(ep_start), int(ep_end)):
            adapter.train()
            lam_align_ep = float(lambda_align) * min(1.0, (ep + 1) / max(6, int(epochs) // 2))
            if ep < int(unknown_warmup_epochs):
                unknown_scale = 0.0
            else:
                unknown_scale = min(1.0, float(ep - int(unknown_warmup_epochs) + 1) / max(1, int(unknown_ramp_epochs)))

            proto_np_ep = _target_proto_from_reliable(teacher if teacher is not None else adapter)

            for _ in range(steps_per_epoch):
                zs, ys, _ = next(it_s)
                zs, ys = zs.to(device), ys.to(device)

                opt.zero_grad()
                zs2 = adapter(zs)
                logits_s = fc(zs2)
                with torch.no_grad():
                    logits_s_ref = fc(zs)

                loss = float(lambda_replay_ce) * ce(logits_s, ys)
                loss = loss + float(lambda_replay_anchor) * mse(zs2, zs)

                if lambda_src_logit > 0:
                    loss = loss + float(lambda_src_logit) * symmetric_kl_from_logits(logits_s, logits_s_ref).mean()

                if src_proto_bank is not None and lambda_src_proto > 0:
                    tgt_src = torch.tensor(src_proto_bank[ys.detach().cpu().numpy()], dtype=torch.float32, device=device)
                    loss = loss + float(lambda_src_proto) * ((F.normalize(zs2, dim=1) - tgt_src) ** 2).mean()

                if dl_rel is not None:
                    zrel, qrel_seed, wrel_t = next(it_rel)
                    zrel = zrel.to(device); qrel_seed = qrel_seed.to(device); wrel_t = wrel_t.to(device)
                    zrel2 = adapter(zrel)
                    logits_rel = fc(zrel2)
                    teacher_logits = None
                    if teacher is not None:
                        with torch.no_grad():
                            teacher_logits = fc(teacher(zrel))
                    qrel = mixed_teacher_targets(qrel_seed, teacher_logits, blend=teacher_blend)
                    loss = loss + float(lambda_rel_sup) * (soft_ce_from_probs(logits_rel, qrel) * wrel_t).mean()
                    if proto_np_ep is not None and lambda_proto > 0:
                        tgt_rel = proto_targets_torch(proto_np_ep, q=qrel.detach().cpu().numpy(), device=device)
                        loss = loss + float(lambda_proto) * ((F.normalize(zrel2, dim=1) - tgt_rel) ** 2).mean()

                if dl_amb is not None:
                    zamb, qamb_seed, wamb_t = next(it_amb)
                    zamb = zamb.to(device); qamb_seed = qamb_seed.to(device); wamb_t = wamb_t.to(device)
                    zamb2 = adapter(zamb)
                    logits_amb = fc(zamb2)
                    teacher_logits = None
                    if teacher is not None:
                        with torch.no_grad():
                            teacher_logits = fc(teacher(zamb))
                    qamb = mixed_teacher_targets(qamb_seed, teacher_logits, blend=teacher_blend)
                    loss = loss + float(lambda_amb_sup) * (soft_ce_from_probs(logits_amb, qamb) * wamb_t).mean()
                    if lambda_cons > 0:
                        logits_amb_aug = fc(adapter(embedding_jitter(zamb)))
                        loss = loss + 0.75 * float(lambda_cons) * (symmetric_kl_from_logits(logits_amb, logits_amb_aug) * wamb_t).mean()
                    if lambda_ent_min > 0:
                        loss = loss + float(lambda_ent_min) * (entropy_from_logits_torch(logits_amb) * wamb_t).mean()

                if use_weak and dl_weak is not None:
                    zweak, _, wweak_t = next(it_weak)
                    zweak = zweak.to(device); wweak_t = wweak_t.to(device)
                    logits_weak = fc(adapter(zweak))
                    weak_loss = 0.0
                    if teacher is not None:
                        with torch.no_grad():
                            logits_teacher = fc(teacher(zweak))
                        weak_loss = weak_loss + 0.5 * (symmetric_kl_from_logits(logits_weak, logits_teacher) * wweak_t).mean()
                    logits_weak_aug = fc(adapter(embedding_jitter(zweak)))
                    weak_loss = weak_loss + 0.5 * (symmetric_kl_from_logits(logits_weak, logits_weak_aug) * wweak_t).mean()
                    loss = loss + float(max(lambda_cons, 1e-8)) * weak_loss
                    if lambda_ent_min > 0:
                        loss = loss + 0.25 * float(lambda_ent_min) * (entropy_from_logits_torch(logits_weak) * wweak_t).mean()

                if dl_align is not None and lam_align_ep > 0:
                    zt = next(it_align).to(device)
                    zt2 = adapter(zt)
                    loss = loss + lam_align_ep * mmd_rbf(zs2, zt2)

                if use_unknown and dl_unk is not None and unknown_scale > 0:
                    zu = next(it_unk).to(device)
                    zu2 = adapter(zu)
                    logits_u = fc(zu2)
                    if lambda_ent_max > 0:
                        uniform = torch.full_like(logits_u, 1.0 / logits_u.shape[1])
                        loss = loss + unknown_scale * float(lambda_ent_max) * F.kl_div(F.log_softmax(logits_u, dim=1), uniform, reduction="batchmean")
                    if proto_np_ep is not None and lambda_u_repel > 0:
                        loss = loss + unknown_scale * float(lambda_u_repel) * proto_repulsion_loss(zu2, proto_np_ep, margin=UNKNOWN_REPEL_MARGIN)

                loss.backward()
                opt.step()
                if teacher is not None:
                    ema_update_(teacher, adapter, momentum=ema_momentum)

    def _promote():
        nonlocal idx_rel_curr, idx_amb_curr, idx_weak_curr, idx_unk_curr, q_curr, w_rel_curr, w_amb_curr, w_weak_curr
        adapter_eval = teacher if teacher is not None else adapter
        Z2_all, logits_all = apply_adapter_and_fc(adapter_eval, fc, Z_target, batch=batch)
        P_model = softmax_np(logits_all / max(1e-6, TEACHER_TEMP))
        P_mix = normalize_rows((1.0 - float(promote_blend)) * q_curr + float(promote_blend) * P_model)

        proto_np = _target_proto_from_reliable(adapter_eval)
        proto_for_score = src_proto_bank if proto_np is None else proto_np
        y_proto, proto_margin, proto_dist = prototype_margin_dist(Z2_all, proto_for_score)

        pred_seed = np.argmax(q_curr, axis=1)
        pred_mix = np.argmax(P_mix, axis=1)
        conf_mix = msp_from_probs(P_mix)
        margin_mix = top2_margin(P_mix)

        idx_eval = np.asarray(idx_gate, dtype=np.int64) if len(idx_gate) > 0 else np.arange(len(Z_target))
        conf_n = robust_unit_interval(conf_mix, idx_eval)
        margin_n = robust_unit_interval(margin_mix, idx_eval)
        proto_margin_n = robust_unit_interval(proto_margin, idx_eval)
        proto_dist_n = robust_unit_interval(proto_dist, idx_eval)
        bucket_n = robust_unit_interval(bucket_score, idx_eval) if np.any(np.isfinite(bucket_score)) else np.zeros_like(conf_n)

        stable = (pred_mix == pred_seed).astype(np.float32)
        proto_agree = (pred_mix == y_proto).astype(np.float32)
        promote_score = (
            0.30 * conf_n +
            0.20 * margin_n +
            0.16 * stable +
            0.16 * proto_agree +
            0.10 * proto_margin_n +
            0.08 * bucket_n -
            0.08 * proto_dist_n
        ).astype(np.float32)
        unknown_model = (
            0.36 * (1.0 - conf_n) +
            0.22 * proto_dist_n +
            0.18 * (1.0 - margin_n) +
            0.14 * (1.0 - stable) +
            0.10 * (1.0 - proto_agree)
        ).astype(np.float32)

        idx_known_pool = np.setdiff1d(np.asarray(idx_gate, dtype=np.int64), idx_unk_curr, assume_unique=False)
        idx_candidate = np.setdiff1d(idx_known_pool, idx_rel_curr, assume_unique=False)

        rel_mask = (
            (conf_mix >= float(promote_conf)) &
            (margin_mix >= float(promote_margin)) &
            (pred_mix == pred_seed) &
            (pred_mix == y_proto)
        )
        idx_rel_cand = idx_candidate[rel_mask[idx_candidate]]
        k_rel_extra = min(
            len(idx_rel_cand),
            max(int(round(float(promote_rel_fraction) * max(1, len(idx_candidate)))), int(V7_PROMOTE_MIN_REL)),
        ) if len(idx_rel_cand) > 0 else 0
        idx_rel_extra = select_topk_class_balanced(idx_rel_cand, pred_mix, promote_score, k_rel_extra, K, min_per_class=1) if k_rel_extra > 0 else empty
        idx_rel_new = np.unique(np.concatenate([idx_rel_curr, idx_rel_extra], axis=0)).astype(np.int64)

        rem = np.setdiff1d(idx_known_pool, idx_rel_new, assume_unique=False)
        amb_mask = (
            (conf_mix >= float(V7_AMB_CONF)) |
            ((margin_mix >= float(V7_AMB_MARGIN)) & ((pred_mix == pred_seed) | (pred_mix == y_proto)))
        )
        idx_amb_cand = rem[amb_mask[rem]]
        target_amb = max(
            len(idx_amb_curr),
            int(round(float(promote_amb_fraction) * max(1, len(idx_known_pool)))),
            int(V7_PROMOTE_MIN_AMB),
        )
        idx_amb_new = select_topk_class_balanced(idx_amb_cand, pred_mix, promote_score, min(len(idx_amb_cand), target_amb), K, min_per_class=1) if len(idx_amb_cand) > 0 else empty

        rem2 = np.setdiff1d(rem, idx_amb_new, assume_unique=False)
        k_unk_new = min(
            max(0, len(rem2) // 8),
            max(0, int(round(V7_UNKNOWN_KEEP_Q * max(1, len(idx_gate)))))
        )
        idx_unk_extra = rem2[np.argsort(-unknown_model[rem2])[:k_unk_new]] if k_unk_new > 0 else empty
        idx_unk_new = np.unique(np.concatenate([idx_unk_curr, idx_unk_extra], axis=0)).astype(np.int64)

        idx_weak_new = np.setdiff1d(np.asarray(idx_gate, dtype=np.int64), np.unique(np.concatenate([idx_rel_new, idx_amb_new, idx_unk_new], axis=0)), assume_unique=False)
        if len(idx_weak_new) < min(V7_WEAK_MIN_KEEP, len(idx_gate)):
            keep_order = np.asarray(idx_gate, dtype=np.int64)[np.argsort(-promote_score[np.asarray(idx_gate, dtype=np.int64)])]
            keep_extra = keep_order[:min(V7_WEAK_MIN_KEEP, len(keep_order))]
            idx_weak_new = np.unique(np.concatenate([idx_weak_new, keep_extra], axis=0))
            idx_weak_new = np.setdiff1d(idx_weak_new, np.unique(np.concatenate([idx_rel_new, idx_amb_new, idx_unk_new], axis=0)), assume_unique=False)

        q_curr = normalize_rows(q_curr.copy())
        if len(idx_rel_new) > 0:
            q_curr[idx_rel_new] = normalize_rows(0.30 * q_curr[idx_rel_new] + 0.70 * P_mix[idx_rel_new])
        if len(idx_amb_new) > 0:
            q_curr[idx_amb_new] = restrict_topk_probs(P_mix[idx_amb_new], k=max(AMBIG_TOPK, 2), sharpen=1.02)
        if len(idx_weak_new) > 0:
            q_curr[idx_weak_new] = restrict_topk_probs(P_mix[idx_weak_new], k=max(V7_WEAK_TOPK, 2), sharpen=V7_WEAK_SHARPEN)

        if len(idx_rel_new) > 0:
            rel_score = 0.65 * promote_score[idx_rel_new] + 0.35 * conf_mix[idx_rel_new]
            w_rel_curr = V7_REL_WEIGHT_FLOOR + (1.0 - V7_REL_WEIGHT_FLOOR) * normalize_conf_weights(rel_score)
        else:
            w_rel_curr = np.zeros((0,), dtype=np.float32)

        if len(idx_amb_new) > 0:
            amb_score = 0.60 * promote_score[idx_amb_new] + 0.40 * conf_mix[idx_amb_new]
            w_amb_curr = V7_AMB_WEIGHT_FLOOR + V7_AMB_WEIGHT_SCALE * normalize_conf_weights(amb_score)
        else:
            w_amb_curr = np.zeros((0,), dtype=np.float32)

        if len(idx_weak_new) > 0:
            weak_score = 0.55 * promote_score[idx_weak_new] + 0.45 * conf_mix[idx_weak_new]
            w_weak_curr = V7_WEAK_WEIGHT_FLOOR + V7_WEAK_WEIGHT_SCALE * normalize_conf_weights(weak_score)
        else:
            w_weak_curr = np.zeros((0,), dtype=np.float32)

        promo_info = dict(
            promoted_reliable=int(len(np.setdiff1d(idx_rel_new, idx_rel_curr, assume_unique=False))),
            promoted_ambiguous=int(len(np.setdiff1d(idx_amb_new, idx_amb_curr, assume_unique=False))),
            stage2_reliable_size=int(len(idx_rel_new)),
            stage2_ambiguous_size=int(len(idx_amb_new)),
            stage2_weak_size=int(len(idx_weak_new)),
            stage2_unknown_size=int(len(idx_unk_new)),
            promote_score_gate_mean=float(promote_score[idx_eval].mean()) if len(idx_eval) > 0 else float("nan"),
        )

        idx_rel_curr = idx_rel_new.astype(np.int64)
        idx_amb_curr = idx_amb_new.astype(np.int64)
        idx_weak_curr = idx_weak_new.astype(np.int64)
        idx_unk_curr = idx_unk_new.astype(np.int64)
        return promo_info

    total_epochs = max(1, int(epochs))
    stage1_epochs = int(max(1, min(int(stage1_epochs), total_epochs - 1 if total_epochs > 1 else 1)))
    _train_range(0, stage1_epochs, use_unknown=False, use_weak=False)
    promo_info = _promote()
    _train_range(stage1_epochs, total_epochs, use_unknown=True, use_weak=True)

    info = dict(
        reliable_size=int(len(idx_rel)),
        ambiguous_size=int(len(idx_amb)),
        weak_size=int(len(idx_weak)),
        unknown_size=int(len(idx_unk)),
        reliable_keep=float(len(idx_rel) / max(1, len(idx_gate))),
        ambiguous_keep=float(len(idx_amb) / max(1, len(idx_gate))),
        weak_keep=float(len(idx_weak) / max(1, len(idx_gate))),
        unknown_keep=float(len(idx_unk) / max(1, len(idx_gate))),
        stage1_epochs=int(stage1_epochs),
    )
    info.update(promo_info)
    return (teacher if (teacher is not None) else adapter), info




def _concat_unique_idx(*parts):
    arrs = [np.asarray(p, dtype=np.int64) for p in parts if p is not None and len(p) > 0]
    if len(arrs) == 0:
        return np.zeros((0,), dtype=np.int64)
    return np.unique(np.concatenate(arrs, axis=0)).astype(np.int64)


def build_dg_raincoat_partition(
    K,
    Z_adapt,
    gate_pred,
    Sid_adapt,
    Sdom_adapt,
    P_logit_adapt,
    P_proto_adapt,
    P_knn_adapt,
    protos,
    tau_conf,
    min_keep=64,
):
    idx_gate = np.where(gate_pred == 1)[0]
    n = len(gate_pred)
    empty = np.zeros((0,), dtype=np.int64)
    if len(idx_gate) == 0:
        aux = dict(
            common_score=np.zeros((n,), dtype=np.float32),
            rescue_score=np.zeros((n,), dtype=np.float32),
            unknown_score=np.zeros((n,), dtype=np.float32),
            P_seed=weighted_prob_fusion([P_logit_adapt, P_proto_adapt, P_knn_adapt], [W_LOGIT, W_PROTO, W_KNN]),
            w_rel=np.zeros((0,), dtype=np.float32),
            w_amb=np.zeros((0,), dtype=np.float32),
            support_idx=np.zeros((0,), dtype=np.int64),
            support_weight=np.zeros((0,), dtype=np.float32),
            rescue_idx=np.zeros((0,), dtype=np.int64),
        )
        info = dict(
            reliable_size=0, ambiguous_size=0, unknown_size=0, rescue_size=0, support_size=0,
            reliable_keep=0.0, ambiguous_keep=0.0, unknown_keep=0.0,
            common_score_gate_mean=float("nan"), unknown_score_gate_mean=float("nan"),
        )
        return empty, empty, empty, aux, info

    P_tri = weighted_prob_fusion([P_logit_adapt, P_proto_adapt, P_knn_adapt], [W_LOGIT, W_PROTO, W_KNN])
    y_tri = np.argmax(P_tri, axis=1)
    y_logit = np.argmax(P_logit_adapt, axis=1)
    y_proto = np.argmax(P_proto_adapt, axis=1)
    y_knn = np.argmax(P_knn_adapt, axis=1)
    conf_tri = msp_from_probs(P_tri)
    margin_tri = top2_margin(P_tri)
    _, proto_margin, proto_dist = prototype_margin_dist(Z_adapt, protos)

    agree = ((y_logit == y_proto).astype(np.float32) + (y_logit == y_knn).astype(np.float32) + (y_proto == y_knn).astype(np.float32)) / 3.0
    sid_n = robust_unit_interval(Sid_adapt, idx_gate)
    sdom_n = robust_unit_interval(Sdom_adapt, idx_gate)
    conf_n = robust_unit_interval(conf_tri, idx_gate)
    margin_n = robust_unit_interval(margin_tri, idx_gate)
    proto_margin_n = robust_unit_interval(proto_margin, idx_gate)
    proto_dist_n = robust_unit_interval(proto_dist, idx_gate)

    common_score = (
        0.32 * sid_n +
        0.18 * sdom_n +
        0.15 * conf_n +
        0.10 * margin_n +
        0.09 * agree +
        0.09 * proto_margin_n +
        0.07 * (1.0 - proto_dist_n)
    ).astype(np.float32)

    rescue_score = (
        0.24 * sid_n +
        0.22 * sdom_n +
        0.18 * conf_n +
        0.12 * margin_n +
        0.10 * agree +
        0.08 * proto_margin_n +
        0.06 * (1.0 - proto_dist_n)
    ).astype(np.float32)

    unknown_score = (
        0.42 * (1.0 - sid_n) +
        0.18 * proto_dist_n +
        0.14 * (1.0 - conf_n) +
        0.10 * (1.0 - margin_n) +
        0.08 * (1.0 - agree) +
        0.08 * (1.0 - sdom_n)
    ).astype(np.float32)

    ng = len(idx_gate)
    k_rel = min(ng, max(int(round(DG_REL_KEEP_Q * ng)), max(16, min_keep // 2)))
    k_amb = min(max(0, ng - k_rel), max(int(round(DG_AMB_KEEP_Q * ng)), max(16, min_keep // 3)))
    k_unk = min(max(0, ng - k_rel - k_amb), max(int(round(DG_UNK_KEEP_Q * ng)), max(8, min_keep // 5)))

    idx_rel = select_topk_class_balanced(idx_gate, y_tri, common_score, k_rel, K, min_per_class=1)
    rem1 = np.setdiff1d(idx_gate, idx_rel, assume_unique=False)

    high_sdom_thr = float(np.quantile(sdom_n[idx_gate], DG_RESCUE_SDOM_Q)) if ng >= 10 else 0.58
    rescue_mask = (
        (sid_n >= float(np.quantile(sid_n[idx_gate], 0.48)) if ng >= 10 else sid_n >= 0.55) &
        (conf_tri >= max(float(tau_conf) * 0.92, float(DG_RESCUE_CONF))) &
        (margin_tri >= float(DG_RESCUE_MARGIN)) &
        (proto_margin_n >= float(np.quantile(proto_margin_n[idx_gate], 0.32)) if ng >= 10 else proto_margin_n >= 0.35) &
        (sdom_n >= high_sdom_thr)
    )
    rescue_candidates = np.intersect1d(rem1, idx_gate[rescue_mask[idx_gate]], assume_unique=False)

    extreme_unk_thr = float(np.quantile(unknown_score[idx_gate], DG_UNKNOWN_EXTREME_Q)) if ng >= 10 else float(np.max(unknown_score[idx_gate]))
    hard_unk_candidates = np.setdiff1d(rem1[unknown_score[rem1] >= extreme_unk_thr], rescue_candidates, assume_unique=False)
    if len(hard_unk_candidates) < k_unk:
        ranked_unk = rem1[np.argsort(-unknown_score[rem1])]
        ranked_unk = np.setdiff1d(ranked_unk, rescue_candidates, assume_unique=False)
        idx_unk = ranked_unk[:k_unk]
    else:
        idx_unk = hard_unk_candidates[np.argsort(-unknown_score[hard_unk_candidates])[:k_unk]]

    rem2 = np.setdiff1d(rem1, idx_unk, assume_unique=False)
    rescue_rank = 0.56 * rescue_score + 0.32 * common_score + 0.12 * conf_tri
    k_rescue = min(len(rem2), max(8, int(round(0.08 * ng))))
    rescue_pool = np.intersect1d(rem2, rescue_candidates, assume_unique=False)
    idx_rescue = select_topk_class_balanced(
        rescue_pool,
        y_tri,
        rescue_rank,
        min(k_rescue, len(rescue_pool)),
        K,
        min_per_class=0,
    ) if len(rescue_pool) > 0 else np.zeros((0,), dtype=np.int64)

    rem3 = np.setdiff1d(rem2, idx_rescue, assume_unique=False)
    amb_score = 0.70 * common_score + 0.20 * rescue_score + 0.10 * conf_tri
    idx_amb_main = select_topk_class_balanced(rem3, y_tri, amb_score, k_amb, K, min_per_class=1) if len(rem3) > 0 else np.zeros((0,), dtype=np.int64)
    idx_amb = _concat_unique_idx(idx_rescue, idx_amb_main)

    support_idx = _concat_unique_idx(idx_rel, idx_amb)
    support_idx = support_idx if len(support_idx) > 0 else idx_rel if len(idx_rel) > 0 else idx_gate
    P_seed = refine_probs_multi_stage(Z_adapt, P_tri, protos, idx_support=support_idx, iters=max(REFINE_ITERS, 3), src_mix=0.76)
    if len(idx_amb) > 0:
        P_seed[idx_amb] = restrict_topk_probs(P_seed[idx_amb], k=max(2, AMBIG_TOPK), sharpen=1.00)

    support_score = np.maximum(common_score[support_idx], 0.95 * rescue_score[support_idx]) - 0.18 * unknown_score[support_idx]
    med = float(np.median(support_score)) if len(support_score) > 0 else 0.0
    std = float(np.std(support_score)) if len(support_score) > 0 else 1.0
    support_weight = DG_SUPPORT_WEIGHT_FLOOR + DG_SUPPORT_WEIGHT_SCALE * sigmoid_np((support_score - med) / max(1e-6, std + 1e-6))
    if len(idx_rescue) > 0:
        rescue_local = np.nonzero(np.isin(support_idx, idx_rescue))[0]
        support_weight[rescue_local] = np.maximum(support_weight[rescue_local], 0.48)
    if len(idx_rel) > 0:
        rel_local = np.nonzero(np.isin(support_idx, idx_rel))[0]
        support_weight[rel_local] = np.maximum(support_weight[rel_local], 0.88)

    w_rel = np.asarray([], dtype=np.float32)
    if len(idx_rel) > 0:
        rel_score = 0.72 * common_score[idx_rel] + 0.28 * conf_tri[idx_rel]
        w_rel = 0.88 + 0.12 * normalize_conf_weights(rel_score)
    w_amb = np.asarray([], dtype=np.float32)
    if len(idx_amb) > 0:
        amb_score_loc = 0.50 * np.maximum(common_score[idx_amb], rescue_score[idx_amb]) + 0.50 * conf_tri[idx_amb]
        w_amb = 0.12 + 0.18 * normalize_conf_weights(amb_score_loc)

    aux = dict(
        common_score=common_score.astype(np.float32),
        rescue_score=rescue_score.astype(np.float32),
        unknown_score=unknown_score.astype(np.float32),
        P_seed=P_seed.astype(np.float32),
        w_rel=w_rel.astype(np.float32),
        w_amb=w_amb.astype(np.float32),
        support_idx=support_idx.astype(np.int64),
        support_weight=support_weight.astype(np.float32),
        rescue_idx=idx_rescue.astype(np.int64),
    )
    info = dict(
        reliable_size=int(len(idx_rel)),
        ambiguous_size=int(len(idx_amb)),
        unknown_size=int(len(idx_unk)),
        rescue_size=int(len(idx_rescue)),
        support_size=int(len(support_idx)),
        reliable_keep=float(len(idx_rel) / max(1, len(gate_pred))),
        ambiguous_keep=float(len(idx_amb) / max(1, len(gate_pred))),
        unknown_keep=float(len(idx_unk) / max(1, len(gate_pred))),
        common_score_gate_mean=float(common_score[idx_gate].mean()),
        unknown_score_gate_mean=float(unknown_score[idx_gate].mean()),
    )
    return idx_rel.astype(np.int64), idx_amb.astype(np.int64), idx_unk.astype(np.int64), aux, info


def adapt_dg_raincoat_lite(
    fc_layer,
    Z_replay,
    y_replay,
    Z_target,
    idx_gate,
    idx_rel,
    idx_amb,
    idx_unk,
    q_seed,
    w_rel=None,
    w_amb=None,
    protos=None,
    Sid_adapt=None,
    Sdom_adapt=None,
    bottleneck=64,
    scale=0.3,
    epochs_stage1=DG_STAGE1_EPOCHS,
    epochs_stage2=DG_STAGE2_EPOCHS,
    lr=1e-3,
    wd=1e-4,
    batch=128,
    lambda_replay_ce=1.0,
    lambda_replay_anchor=1.0,
    lambda_align_stage1=DG_LAMBDA_ALIGN_STAGE1,
    lambda_align_stage2=DG_LAMBDA_ALIGN_STAGE2,
    lambda_ent_min=DG_LAMBDA_ENT_MIN,
    lambda_ent_max=DG_LAMBDA_ENT_MAX,
    lambda_cons=DG_LAMBDA_CONS,
    lambda_proto=DG_LAMBDA_PROTO,
    lambda_u_repel=DG_LAMBDA_UNKNOWN_REPEL,
):
    idx_gate = np.asarray(idx_gate, dtype=np.int64)
    idx_rel = np.asarray(idx_rel, dtype=np.int64)
    idx_amb = np.asarray(idx_amb, dtype=np.int64)
    idx_unk = np.asarray(idx_unk, dtype=np.int64)
    if len(idx_gate) == 0:
        return None, {}

    K = q_seed.shape[1]
    q_seed = normalize_rows(q_seed.astype(np.float32))
    w_rel = np.ones((len(idx_rel),), dtype=np.float32) if (w_rel is None or len(w_rel) == 0) else np.asarray(w_rel, dtype=np.float32)
    w_amb = np.ones((len(idx_amb),), dtype=np.float32) * 0.22 if (w_amb is None or len(w_amb) == 0) else np.asarray(w_amb, dtype=np.float32)

    idx_sup0 = _concat_unique_idx(idx_rel, idx_amb)
    w_sup0 = np.concatenate([0.95 * w_rel, 0.55 * w_amb], axis=0) if len(idx_sup0) > 0 else np.ones((0,), dtype=np.float32)

    warm = adapt_on_embeddings_generic(
        fc_layer=fc_layer,
        Z_replay=Z_replay, y_replay=y_replay,
        Z_sup=Z_target[idx_sup0] if len(idx_sup0) > 0 else None,
        q_sup=q_seed[idx_sup0] if len(idx_sup0) > 0 else None,
        w_sup=w_sup0 if len(idx_sup0) > 0 else None,
        Z_align=Z_target[idx_gate],
        Z_unrel=Z_target[idx_unk] if len(idx_unk) > 0 else None,
        proto_bank=protos,
        bottleneck=bottleneck, scale=scale, epochs=epochs_stage1,
        lr=lr, wd=wd, batch=batch,
        lambda_replay_ce=lambda_replay_ce, lambda_replay_anchor=lambda_replay_anchor,
        lambda_sup=1.0,
        lambda_align=lambda_align_stage1,
        lambda_ent_min=lambda_ent_min * 0.8,
        lambda_ent_max=lambda_ent_max * 0.35,
        lambda_cons=lambda_cons * 0.8,
        lambda_proto=lambda_proto * 0.8,
        dynamic_align=True,
    )
    if warm is None:
        return None, {}

    Z_gate1, logits_gate1 = apply_adapter_and_fc(warm, fc_layer, Z_target[idx_gate], batch=512)
    P_logit1 = softmax_np(logits_gate1)
    P_proto1 = proto_probs_cosine(Z_gate1, protos, temp=PROTO_T, l2norm=True)
    P_mix1 = weighted_prob_fusion([P_logit1, P_proto1, q_seed[idx_gate]], [1.0, 1.0, 0.70])

    conf1 = msp_from_probs(P_mix1)
    margin1 = top2_margin(P_mix1)
    y1 = np.argmax(P_mix1, axis=1)
    y0 = np.argmax(q_seed[idx_gate], axis=1)
    stable = (y1 == y0).astype(np.float32)

    sid_n = robust_unit_interval(Sid_adapt, idx_gate)[idx_gate] if Sid_adapt is not None else np.ones((len(idx_gate),), dtype=np.float32) * 0.5
    sdom_n = robust_unit_interval(Sdom_adapt, idx_gate)[idx_gate] if Sdom_adapt is not None else np.ones((len(idx_gate),), dtype=np.float32) * 0.5
    conf_n = robust_unit_interval(conf1, np.arange(len(idx_gate)))
    margin_n = robust_unit_interval(margin1, np.arange(len(idx_gate)))
    _, proto_margin1, proto_dist1 = prototype_margin_dist(Z_gate1, protos)
    proto_margin_n = robust_unit_interval(proto_margin1, np.arange(len(idx_gate)))
    proto_dist_n = robust_unit_interval(proto_dist1, np.arange(len(idx_gate)))

    Z0n = normalize_vec_rows_np(Z_target[idx_gate])
    Z1n = normalize_vec_rows_np(Z_gate1)
    move = np.sqrt(np.maximum(1e-12, np.sum((Z1n - Z0n) ** 2, axis=1)))
    move_n = robust_unit_interval(move, np.arange(len(idx_gate)))
    move_low_n = 1.0 - move_n

    common_score = (
        0.30 * sid_n +
        0.18 * sdom_n +
        0.14 * conf_n +
        0.08 * margin_n +
        0.10 * stable +
        0.08 * proto_margin_n +
        0.08 * move_low_n +
        0.04 * (1.0 - proto_dist_n)
    ).astype(np.float32)

    rescue_score = (
        0.22 * sid_n +
        0.22 * sdom_n +
        0.18 * conf_n +
        0.10 * margin_n +
        0.12 * stable +
        0.10 * proto_margin_n +
        0.06 * move_low_n
    ).astype(np.float32)

    unknown_score = (
        0.38 * (1.0 - sid_n) +
        0.18 * proto_dist_n +
        0.12 * (1.0 - conf_n) +
        0.08 * (1.0 - margin_n) +
        0.08 * move_n +
        0.08 * (1.0 - stable) +
        0.08 * (1.0 - sdom_n)
    ).astype(np.float32)

    ng = len(idx_gate)
    high_sdom_thr = float(np.quantile(sdom_n, DG_RESCUE_SDOM_Q)) if ng >= 10 else 0.58
    rescue_mask_local = (
        (sid_n >= float(np.quantile(sid_n, 0.48)) if ng >= 10 else sid_n >= 0.55) &
        (conf1 >= max(float(np.quantile(conf1, 0.52)) if ng >= 10 else 0.56, DG_RESCUE_CONF)) &
        (margin1 >= float(DG_RESCUE_MARGIN)) &
        (proto_margin_n >= float(np.quantile(proto_margin_n, 0.35)) if ng >= 10 else 0.35) &
        (sdom_n >= high_sdom_thr) &
        (stable >= 1.0)
    )

    extreme_unk_thr = float(np.quantile(unknown_score, DG_UNKNOWN_EXTREME_Q)) if ng >= 10 else float(np.max(unknown_score))
    local_all = np.arange(ng, dtype=np.int64)
    local_unk = local_all[(unknown_score >= extreme_unk_thr) & (~rescue_mask_local)]
    idx_unk1 = idx_gate[local_unk]

    candidate_mask = (
        rescue_mask_local |
        (common_score >= (float(np.quantile(common_score, 0.46)) if ng >= 10 else 0.50)) |
        ((stable >= 1.0) & (conf1 >= (float(np.quantile(conf1, 0.45)) if ng >= 10 else 0.52)) & (unknown_score <= (float(np.quantile(unknown_score, 0.72)) if ng >= 10 else 0.60)))
    ) & (unknown_score < extreme_unk_thr)
    support_local = local_all[candidate_mask]
    if len(support_local) < max(16, ng // 8):
        support_score_fallback = np.maximum(common_score, 0.95 * rescue_score) - 0.20 * unknown_score
        k_support = min(ng, max(16, ng // 6))
        support_local = np.argsort(-support_score_fallback)[:k_support]
        support_local = np.setdiff1d(support_local, local_unk, assume_unique=False)
    if len(support_local) == 0:
        support_local = np.setdiff1d(local_all, local_unk, assume_unique=False)
        if len(support_local) == 0:
            support_local = local_all.copy()
            idx_unk1 = np.zeros((0,), dtype=np.int64)
    support_idx = idx_gate[support_local]

    q_stage2_local = refine_probs_multi_stage(
        Z_gate1,
        P_mix1,
        protos,
        idx_support=support_local,
        iters=max(REFINE_ITERS, 3),
        src_mix=0.76,
    )
    q_stage2 = q_seed.copy()
    q_stage2[idx_gate] = q_stage2_local.astype(np.float32)

    support_score = np.maximum(common_score[support_local], 0.96 * rescue_score[support_local]) - 0.18 * unknown_score[support_local]
    med = float(np.median(support_score)) if len(support_score) > 0 else 0.0
    std = float(np.std(support_score)) if len(support_score) > 0 else 1.0
    support_weight = DG_SUPPORT_WEIGHT_FLOOR + DG_SUPPORT_WEIGHT_SCALE * sigmoid_np((support_score - med) / max(1e-6, std + 1e-6))
    support_weight = support_weight.astype(np.float32)

    rescue_local = support_local[rescue_mask_local[support_local]]
    if len(rescue_local) > 0:
        rescue_pos = np.nonzero(np.isin(support_local, rescue_local))[0]
        support_weight[rescue_pos] = np.maximum(support_weight[rescue_pos], 0.48)

    reliable_mask_local = common_score[support_local] >= max(float(np.quantile(common_score[support_local], 0.80)) if len(support_local) >= 10 else 0.72, 0.72)
    reliable_pos = np.nonzero(reliable_mask_local)[0]
    if len(reliable_pos) > 0:
        support_weight[reliable_pos] = np.maximum(support_weight[reliable_pos], 0.88)

    ambiguous_pos = np.nonzero((~reliable_mask_local) & (support_weight < 0.88))[0]
    if len(ambiguous_pos) > 0:
        support_weight[ambiguous_pos] = np.maximum(support_weight[ambiguous_pos], 0.12)

    if len(support_local) > 0:
        low_weight_mask = support_weight < 0.20
        if np.any(low_weight_mask):
            q_stage2[support_idx[low_weight_mask]] = restrict_topk_probs(q_stage2[support_idx[low_weight_mask]], k=2, sharpen=0.97)

    align_idx = np.setdiff1d(idx_gate, idx_unk1, assume_unique=False)
    if len(align_idx) == 0:
        align_idx = support_idx

    final_adapter = adapt_on_embeddings_generic(
        fc_layer=fc_layer,
        Z_replay=Z_replay, y_replay=y_replay,
        Z_sup=Z_target[support_idx] if len(support_idx) > 0 else None,
        q_sup=q_stage2[support_idx] if len(support_idx) > 0 else None,
        w_sup=support_weight if len(support_idx) > 0 else None,
        Z_align=Z_target[align_idx],
        Z_unrel=Z_target[idx_unk1] if len(idx_unk1) > 0 else None,
        proto_bank=protos,
        bottleneck=bottleneck, scale=scale, epochs=epochs_stage2,
        lr=lr, wd=wd, batch=batch,
        lambda_replay_ce=lambda_replay_ce, lambda_replay_anchor=lambda_replay_anchor,
        lambda_sup=1.0,
        lambda_align=lambda_align_stage2,
        lambda_ent_min=lambda_ent_min,
        lambda_ent_max=lambda_ent_max * 0.50,
        lambda_cons=lambda_cons,
        lambda_proto=lambda_proto,
        dynamic_align=True,
        init_state=clone_state_dict_cpu(warm),
    )

    info = dict(
        reliable_size=int(np.sum(support_weight >= 0.88)),
        ambiguous_size=int(np.sum((support_weight >= 0.12) & (support_weight < 0.88))),
        unknown_size=int(len(idx_unk1)),
        rescue_size=int(len(rescue_local)),
        support_size=int(len(support_idx)),
        reliable_keep=float(np.sum(support_weight >= 0.88) / max(1, len(Z_target))),
        ambiguous_keep=float(np.sum((support_weight >= 0.12) & (support_weight < 0.88)) / max(1, len(Z_target))),
        unknown_keep=float(len(idx_unk1) / max(1, len(Z_target))),
        common_score_gate_mean=float(common_score.mean()) if len(common_score) > 0 else float("nan"),
        unknown_score_gate_mean=float(unknown_score.mean()) if len(unknown_score) > 0 else float("nan"),
        move_gate_mean=float(move.mean()) if len(move) > 0 else float("nan"),
        stable_pred_ratio=float(stable.mean()) if len(stable) > 0 else float("nan"),
        stage1_reliable_size=int(len(idx_rel)),
        stage1_ambiguous_size=int(len(idx_amb)),
        stage1_unknown_size=int(len(idx_unk)),
        stage2_reliable_size=int(np.sum(support_weight >= 0.88)),
        stage2_ambiguous_size=int(np.sum((support_weight >= 0.12) & (support_weight < 0.88))),
        stage2_unknown_size=int(len(idx_unk1)),
        rescued_common_size=int(len(rescue_local)),
    )
    return final_adapter if final_adapter is not None else warm, info


def build_three_bucket_partition_v8(
    K,
    Z_adapt,
    gate_pred,
    Sid_adapt,
    Sdom_adapt,
    P_logit_adapt,
    P_proto_adapt,
    P_knn_adapt,
    protos,
    tau_conf,
    min_keep=64,
):
    idx_gate = np.where(gate_pred == 1)[0]
    n = len(gate_pred)
    empty = np.zeros((0,), dtype=np.int64)
    if len(idx_gate) == 0:
        aux = dict(
            bucket_score=np.zeros((n,), dtype=np.float32),
            unknown_score=np.zeros((n,), dtype=np.float32),
            P_seed=weighted_prob_fusion([P_logit_adapt, P_proto_adapt, P_knn_adapt], [W_LOGIT, W_PROTO, W_KNN]),
            w_rel=np.zeros((0,), dtype=np.float32),
            w_amb=np.zeros((0,), dtype=np.float32),
            w_weak=np.zeros((0,), dtype=np.float32),
            idx_weak_cold=np.zeros((0,), dtype=np.int64),
            w_weak_cold=np.zeros((0,), dtype=np.float32),
            support_idx=np.zeros((0,), dtype=np.int64),
        )
        info = dict(
            reliable_size=0, ambiguous_size=0, weak_size=0, cold_weak_size=0, unknown_size=0,
            reliable_keep=0.0, ambiguous_keep=0.0, weak_keep=0.0, unknown_keep=0.0,
            bucket_score_gate_mean=float("nan"), unknown_score_gate_mean=float("nan"),
        )
        return empty, empty, empty, empty, aux, info

    P_tri = weighted_prob_fusion([P_logit_adapt, P_proto_adapt, P_knn_adapt], [W_LOGIT, W_PROTO, W_KNN])
    y_tri = np.argmax(P_tri, axis=1)
    y_logit = np.argmax(P_logit_adapt, axis=1)
    y_proto = np.argmax(P_proto_adapt, axis=1)
    y_knn = np.argmax(P_knn_adapt, axis=1)
    conf_tri = msp_from_probs(P_tri)
    margin_tri = top2_margin(P_tri)
    _, proto_margin, proto_dist = prototype_margin_dist(Z_adapt, protos)

    agree = ((y_logit == y_proto).astype(np.float32) + (y_logit == y_knn).astype(np.float32) + (y_proto == y_knn).astype(np.float32)) / 3.0
    sid_n = robust_unit_interval(Sid_adapt, idx_gate)
    sdom_n = robust_unit_interval(Sdom_adapt, idx_gate)
    conf_n = robust_unit_interval(conf_tri, idx_gate)
    margin_n = robust_unit_interval(margin_tri, idx_gate)
    proto_margin_n = robust_unit_interval(proto_margin, idx_gate)
    proto_dist_n = robust_unit_interval(proto_dist, idx_gate)

    bucket_score = (
        0.30 * sid_n +
        0.14 * sdom_n +
        0.18 * conf_n +
        0.12 * agree +
        0.12 * margin_n +
        0.10 * proto_margin_n +
        0.04 * (1.0 - proto_dist_n)
    ).astype(np.float32)
    unknown_score = (
        0.38 * (1.0 - sid_n) +
        0.18 * proto_dist_n +
        0.14 * (1.0 - conf_n) +
        0.12 * (1.0 - margin_n) +
        0.10 * (1.0 - agree) +
        0.08 * (1.0 - sdom_n)
    ).astype(np.float32)

    ng = len(idx_gate)
    unk_gap = float(np.quantile(unknown_score[idx_gate], 0.92) - np.quantile(unknown_score[idx_gate], 0.65)) if ng >= 10 else 0.0
    unk_q = V8_UNKNOWN_KEEP_Q_HARD if unk_gap >= 0.08 else V8_UNKNOWN_KEEP_Q

    rel_mask = (bucket_score >= float(np.quantile(bucket_score[idx_gate], max(0.50, 1.0 - V8_RELIABLE_KEEP_Q)))) & (conf_tri >= max(float(tau_conf), 0.55))
    idx_rel = select_idx_by_mask_with_fallback(idx_gate, rel_mask[idx_gate], bucket_score, min_keep=max(16, min_keep // 2))

    rem1 = np.setdiff1d(idx_gate, idx_rel, assume_unique=False)
    k_unk = min(len(rem1), max(int(round(unk_q * ng)), max(8, min_keep // 4)))
    idx_unk = rem1[np.argsort(-unknown_score[rem1])[:k_unk]] if len(rem1) > 0 and k_unk > 0 else np.zeros((0,), dtype=np.int64)

    rem2 = np.setdiff1d(rem1, idx_unk, assume_unique=False)
    amb_mask = (bucket_score >= float(np.quantile(bucket_score[idx_gate], max(0.32, 1.0 - (V8_RELIABLE_KEEP_Q + V8_AMBIG_KEEP_Q))))) & (conf_tri >= max(float(tau_conf) * 0.85, 0.36))
    idx_amb = select_idx_by_mask_with_fallback(rem2, amb_mask[rem2] if len(rem2) > 0 else np.zeros((0,), dtype=bool), bucket_score, min_keep=max(16, min_keep // 3))
    rem3 = np.setdiff1d(rem2, idx_amb, assume_unique=False)

    weak_strength = 0.60 * bucket_score[rem3] + 0.40 * conf_tri[rem3] - 0.25 * unknown_score[rem3] if len(rem3) > 0 else np.zeros((0,), dtype=np.float32)
    if len(rem3) > 0:
        cold_thr = float(np.quantile(weak_strength, V81_COLD_WEAK_Q)) if len(rem3) >= 10 else float(np.median(weak_strength))
        idx_weak = rem3[weak_strength >= cold_thr]
        idx_weak_cold = rem3[weak_strength < cold_thr]
    else:
        idx_weak = np.zeros((0,), dtype=np.int64)
        idx_weak_cold = np.zeros((0,), dtype=np.int64)

    support_idx = _concat_unique_idx(idx_rel, idx_amb, idx_weak)
    support_idx = support_idx if len(support_idx) > 0 else idx_gate
    P_seed = refine_probs_multi_stage(Z_adapt, P_tri, protos, idx_support=support_idx, iters=max(REFINE_ITERS, 3), src_mix=0.82)
    if len(idx_amb) > 0:
        P_seed[idx_amb] = restrict_topk_probs(P_seed[idx_amb], k=max(2, AMBIG_TOPK), sharpen=1.01)
    if len(idx_weak) > 0:
        P_seed[idx_weak] = restrict_topk_probs(P_seed[idx_weak], k=max(2, V81_SEMIWEAK_TOPK), sharpen=0.99)
    if len(idx_weak_cold) > 0:
        P_seed[idx_weak_cold] = restrict_topk_probs(P_seed[idx_weak_cold], k=2, sharpen=0.96)

    w_rel = np.asarray([], dtype=np.float32)
    if len(idx_rel) > 0:
        rel_score = 0.65 * bucket_score[idx_rel] + 0.35 * conf_tri[idx_rel]
        w_rel = V8_REL_WEIGHT_FLOOR + (1.0 - V8_REL_WEIGHT_FLOOR) * normalize_conf_weights(rel_score)
    w_amb = np.asarray([], dtype=np.float32)
    if len(idx_amb) > 0:
        amb_score = 0.60 * bucket_score[idx_amb] + 0.40 * conf_tri[idx_amb]
        w_amb = V8_AMB_WEIGHT_FLOOR + V8_AMB_WEIGHT_SCALE * normalize_conf_weights(amb_score)
    w_weak = np.asarray([], dtype=np.float32)
    if len(idx_weak) > 0:
        weak_score = 0.60 * bucket_score[idx_weak] + 0.40 * conf_tri[idx_weak]
        w_weak = V8_WEAK_WEIGHT_FLOOR + (V8_WEAK_WEIGHT_SCALE + 0.04) * normalize_conf_weights(weak_score)
    w_weak_cold = np.asarray([], dtype=np.float32)
    if len(idx_weak_cold) > 0:
        cold_score = 0.60 * bucket_score[idx_weak_cold] + 0.40 * conf_tri[idx_weak_cold]
        w_weak_cold = V81_COLD_WEIGHT_FLOOR + V81_COLD_WEIGHT_SCALE * normalize_conf_weights(cold_score)

    aux = dict(
        bucket_score=bucket_score.astype(np.float32),
        unknown_score=unknown_score.astype(np.float32),
        P_seed=P_seed.astype(np.float32),
        w_rel=w_rel.astype(np.float32),
        w_amb=w_amb.astype(np.float32),
        w_weak=w_weak.astype(np.float32),
        idx_weak_cold=idx_weak_cold.astype(np.int64),
        w_weak_cold=w_weak_cold.astype(np.float32),
        support_idx=support_idx.astype(np.int64),
    )
    info = dict(
        reliable_size=int(len(idx_rel)),
        ambiguous_size=int(len(idx_amb)),
        weak_size=int(len(idx_weak)),
        cold_weak_size=int(len(idx_weak_cold)),
        unknown_size=int(len(idx_unk)),
        reliable_keep=float(len(idx_rel) / max(1, len(gate_pred))),
        ambiguous_keep=float(len(idx_amb) / max(1, len(gate_pred))),
        weak_keep=float(len(idx_weak) / max(1, len(gate_pred))),
        unknown_keep=float(len(idx_unk) / max(1, len(gate_pred))),
        bucket_score_gate_mean=float(bucket_score[idx_gate].mean()),
        unknown_score_gate_mean=float(unknown_score[idx_gate].mean()),
    )
    return idx_rel.astype(np.int64), idx_amb.astype(np.int64), idx_weak.astype(np.int64), idx_unk.astype(np.int64), aux, info


def adapt_three_bucket_v8(
    fc_layer,
    Z_replay,
    y_replay,
    Z_target,
    idx_gate,
    idx_rel,
    idx_amb,
    idx_weak,
    idx_unk,
    q_seed,
    w_rel=None,
    w_amb=None,
    w_weak=None,
    proto_bank=None,
    bucket_score=None,
    unknown_score=None,
    Sid_adapt=None,
    Sdom_adapt=None,
    idx_weak_cold=None,
    w_weak_cold=None,
    bottleneck=64,
    scale=0.3,
    stage0_epochs=V8_STAGE0_EPOCHS,
    refresh_epochs=V8_REFRESH_EPOCHS,
    refresh_rounds=V8_REFRESH_ROUNDS,
    lr=1e-3,
    wd=1e-4,
    batch=128,
    lambda_replay_ce=1.0,
    lambda_replay_anchor=1.0,
    lambda_align=V8_LAMBDA_ALIGN,
    lambda_ent_min=V8_LAMBDA_ENT_MIN,
    lambda_ent_max=V8_LAMBDA_ENT_MAX,
    lambda_cons=V8_LAMBDA_CONS,
    lambda_proto=V8_LAMBDA_PROTO,
    lambda_u_repel=V8_LAMBDA_UNKNOWN_REPEL,
):
    idx_gate = np.asarray(idx_gate, dtype=np.int64)
    if len(idx_gate) == 0:
        return None, {}

    K = q_seed.shape[1]
    q_curr = normalize_rows(q_seed.astype(np.float32).copy())
    idx_rel_curr = np.asarray(idx_rel, dtype=np.int64)
    idx_amb_curr = np.asarray(idx_amb, dtype=np.int64)
    idx_weak_curr = np.asarray(idx_weak, dtype=np.int64)
    idx_unk_curr = np.asarray(idx_unk, dtype=np.int64)
    idx_weak_cold_curr = np.asarray(idx_weak_cold if idx_weak_cold is not None else np.zeros((0,), dtype=np.int64), dtype=np.int64)

    if bucket_score is None:
        bucket_score = np.zeros((len(Z_target),), dtype=np.float32)
    if unknown_score is None:
        unknown_score = np.zeros((len(Z_target),), dtype=np.float32)

    sid_n_full = robust_unit_interval(Sid_adapt, idx_gate)[idx_gate] if Sid_adapt is not None else np.ones((len(idx_gate),), dtype=np.float32) * 0.5
    sdom_n_full = robust_unit_interval(Sdom_adapt, idx_gate)[idx_gate] if Sdom_adapt is not None else np.ones((len(idx_gate),), dtype=np.float32) * 0.5

    w_rel_curr = np.ones((len(idx_rel_curr),), dtype=np.float32) if (w_rel is None or len(w_rel) == 0) else np.asarray(w_rel, dtype=np.float32)
    w_amb_curr = np.ones((len(idx_amb_curr),), dtype=np.float32) * V8_AMB_WEIGHT_FLOOR if (w_amb is None or len(w_amb) == 0) else np.asarray(w_amb, dtype=np.float32)
    w_weak_curr = np.ones((len(idx_weak_curr),), dtype=np.float32) * V8_WEAK_WEIGHT_FLOOR if (w_weak is None or len(w_weak) == 0) else np.asarray(w_weak, dtype=np.float32)
    w_weak_cold_curr = np.ones((len(idx_weak_cold_curr),), dtype=np.float32) * V81_COLD_WEIGHT_FLOOR if (w_weak_cold is None or len(w_weak_cold) == 0) else np.asarray(w_weak_cold, dtype=np.float32)

    state = None
    promo_total_rel = 0
    promo_total_amb = 0
    refresh_stats = []

    prev_pred = np.full((len(idx_gate),), -1, dtype=np.int64)
    stable_count = np.zeros((len(idx_gate),), dtype=np.int64)

    def _pack_sup(rel_idx, amb_idx, weak_idx):
        sup_idx = _concat_unique_idx(rel_idx, amb_idx, weak_idx)
        if len(sup_idx) == 0:
            return sup_idx, None, None
        q_sup = q_curr[sup_idx]
        parts = []
        if len(rel_idx) > 0:
            parts.append(w_rel_curr)
        if len(amb_idx) > 0:
            parts.append(w_amb_curr)
        if len(weak_idx) > 0:
            parts.append(w_weak_curr)
        w_sup = np.concatenate(parts, axis=0) if len(parts) > 0 else None
        return sup_idx, q_sup, w_sup

    sup0_idx, sup0_q, sup0_w = _pack_sup(idx_rel_curr, idx_amb_curr, np.zeros((0,), dtype=np.int64))
    warm = adapt_on_embeddings_generic(
        fc_layer=fc_layer,
        Z_replay=Z_replay, y_replay=y_replay,
        Z_sup=Z_target[sup0_idx] if len(sup0_idx) > 0 else None,
        q_sup=sup0_q if sup0_q is not None else None,
        w_sup=sup0_w if sup0_w is not None else None,
        Z_align=Z_target[idx_gate],
        Z_unrel=Z_target[idx_unk_curr] if len(idx_unk_curr) > 0 else None,
        proto_bank=proto_bank,
        bottleneck=bottleneck, scale=scale, epochs=stage0_epochs,
        lr=lr, wd=wd, batch=batch,
        lambda_replay_ce=lambda_replay_ce, lambda_replay_anchor=lambda_replay_anchor,
        lambda_sup=1.0,
        lambda_align=lambda_align,
        lambda_ent_min=lambda_ent_min,
        lambda_ent_max=lambda_ent_max * 0.45,
        lambda_cons=lambda_cons * 0.75,
        lambda_proto=lambda_proto * 0.8,
        dynamic_align=True,
        init_state=state,
    )
    if warm is None:
        return None, {}
    state = clone_state_dict_cpu(warm)

    final_adapter = warm
    for rr in range(int(max(1, refresh_rounds))):
        sup_idx_round, sup_q_round, sup_w_round = _pack_sup(idx_rel_curr, idx_amb_curr, idx_weak_curr)
        adapter_r = adapt_on_embeddings_generic(
            fc_layer=fc_layer,
            Z_replay=Z_replay, y_replay=y_replay,
            Z_sup=Z_target[sup_idx_round] if len(sup_idx_round) > 0 else None,
            q_sup=sup_q_round if len(sup_idx_round) > 0 else None,
            w_sup=sup_w_round if len(sup_idx_round) > 0 else None,
            Z_align=Z_target[np.setdiff1d(idx_gate, idx_unk_curr, assume_unique=False)],
            Z_unrel=Z_target[idx_unk_curr] if len(idx_unk_curr) > 0 else None,
            proto_bank=proto_bank,
            bottleneck=bottleneck, scale=scale, epochs=refresh_epochs,
            lr=lr, wd=wd, batch=batch,
            lambda_replay_ce=lambda_replay_ce, lambda_replay_anchor=lambda_replay_anchor,
            lambda_sup=1.0,
            lambda_align=lambda_align,
            lambda_ent_min=lambda_ent_min,
            lambda_ent_max=lambda_ent_max,
            lambda_cons=lambda_cons,
            lambda_proto=lambda_proto,
            dynamic_align=True,
            init_state=state,
        )
        if adapter_r is None:
            break
        final_adapter = adapter_r
        state = clone_state_dict_cpu(adapter_r)

        Zg, logits_g = apply_adapter_and_fc(adapter_r, fc_layer, Z_target[idx_gate], batch=512)
        P_logit_g = softmax_np(logits_g)
        P_proto_g = proto_probs_cosine(Zg, proto_bank, temp=PROTO_T, l2norm=True)
        P_mix_g = weighted_prob_fusion([P_logit_g, P_proto_g, q_curr[idx_gate]], [1.0, 1.0, 0.80])

        conf_g = msp_from_probs(P_mix_g)
        margin_g = top2_margin(P_mix_g)
        pred_g = np.argmax(P_mix_g, axis=1)
        stable_now = (pred_g == prev_pred)
        stable_count = np.where(stable_now, stable_count + 1, 0)
        prev_pred = pred_g.copy()

        conf_n = robust_unit_interval(conf_g, np.arange(len(idx_gate)))
        margin_n = robust_unit_interval(margin_g, np.arange(len(idx_gate)))
        _, proto_margin_g, proto_dist_g = prototype_margin_dist(Zg, proto_bank)
        proto_margin_n = robust_unit_interval(proto_margin_g, np.arange(len(idx_gate)))
        proto_dist_n = robust_unit_interval(proto_dist_g, np.arange(len(idx_gate)))
        stability_n = np.clip(stable_count.astype(np.float32) / float(max(1, V8_STABLE_K_REL + 1)), 0.0, 1.0)

        class_counts = np.bincount(pred_g, minlength=K).astype(np.float32)
        class_counts[class_counts <= 0] = 1.0
        rarity = np.sqrt(class_counts.mean() / class_counts)
        rarity = rarity / np.maximum(1.0, rarity.max())

        promote_score = (
            0.24 * sid_n_full +
            0.12 * sdom_n_full +
            0.20 * conf_n +
            0.12 * margin_n +
            0.10 * proto_margin_n +
            0.14 * stability_n +
            0.08 * robust_unit_interval(bucket_score[idx_gate], np.arange(len(idx_gate)))
        ).astype(np.float32)
        promote_score_pc = promote_score * (0.85 + 0.15 * rarity[pred_g])

        unknown_score_dyn = (
            0.34 * (1.0 - sid_n_full) +
            0.16 * proto_dist_n +
            0.14 * (1.0 - conf_n) +
            0.10 * (1.0 - margin_n) +
            0.14 * (1.0 - stability_n) +
            0.12 * (1.0 - sdom_n_full)
        ).astype(np.float32)

        unk_thr = float(np.quantile(unknown_score_dyn, max(0.82, 1.0 - V8_UNKNOWN_KEEP_Q))) if len(idx_gate) >= 10 else float(np.max(unknown_score_dyn))
        local_unk = np.where(unknown_score_dyn >= unk_thr)[0]
        idx_unk_curr = idx_gate[local_unk]

        local_all = np.arange(len(idx_gate), dtype=np.int64)
        local_nonunk = np.setdiff1d(local_all, local_unk, assume_unique=False)
        rel_budget = int(sum(max(1, int(np.ceil(V81_CLASS_REL_FRAC * c))) for c in class_counts if c > 0))
        amb_budget = int(sum(max(1, int(np.ceil(V81_CLASS_AMB_FRAC * c))) for c in class_counts if c > 0))

        cand_rel_local = np.intersect1d(
            local_nonunk,
            np.where(
                (stable_count >= int(V8_STABLE_K_REL)) &
                (conf_g >= float(V8_PROMOTE_CONF_REL)) &
                (margin_g >= float(V8_PROMOTE_MARGIN_REL))
            )[0],
            assume_unique=False,
        )
        local_rel_sel = select_topk_class_balanced(
            cand_rel_local,
            pred_g,
            promote_score_pc,
            min(max(16, rel_budget), len(cand_rel_local)),
            K,
            min_per_class=1,
        ) if len(cand_rel_local) > 0 else np.zeros((0,), dtype=np.int64)
        idx_rel_new = idx_gate[local_rel_sel]

        local_rem_after_rel = np.setdiff1d(local_nonunk, local_rel_sel, assume_unique=False)
        cand_amb_local = np.intersect1d(
            local_rem_after_rel,
            np.where(
                (stable_count >= int(V8_STABLE_K_AMB)) &
                (conf_g >= float(V8_PROMOTE_CONF_AMB)) &
                (margin_g >= float(V8_PROMOTE_MARGIN_AMB))
            )[0],
            assume_unique=False,
        )
        local_amb_sel = select_topk_class_balanced(
            cand_amb_local,
            pred_g,
            promote_score_pc,
            min(max(16, amb_budget), len(cand_amb_local)),
            K,
            min_per_class=1,
        ) if len(cand_amb_local) > 0 else np.zeros((0,), dtype=np.int64)
        idx_amb_new = idx_gate[local_amb_sel]

        local_rem_after_amb = np.setdiff1d(local_rem_after_rel, local_amb_sel, assume_unique=False)
        if len(local_rem_after_amb) > 0:
            semiweak_candidates = local_rem_after_amb[
                (conf_g[local_rem_after_amb] >= max(0.45, float(V8_PROMOTE_CONF_AMB) - 0.05)) &
                (margin_g[local_rem_after_amb] >= max(0.02, float(V8_PROMOTE_MARGIN_AMB) - 0.02))
            ]
            extra_amb = []
            current_amb_pred = pred_g[local_amb_sel] if len(local_amb_sel) > 0 else np.zeros((0,), dtype=np.int64)
            for k in range(K):
                target_k = max(1, int(np.ceil(V81_CLASS_AMB_FRAC * class_counts[k]))) if class_counts[k] > 0 else 0
                cur_k = int(np.sum(current_amb_pred == k)) if len(current_amb_pred) > 0 else 0
                need_k = max(0, target_k - cur_k)
                if need_k <= 0:
                    continue
                cand_k = semiweak_candidates[pred_g[semiweak_candidates] == k]
                if len(cand_k) == 0:
                    continue
                score_k = promote_score_pc[cand_k]
                take_k = cand_k[np.argsort(-score_k)[:min(need_k, len(cand_k))]]
                extra_amb.append(take_k)
            if len(extra_amb) > 0:
                extra_amb = np.concatenate(extra_amb, axis=0).astype(np.int64)
                local_amb_sel = np.unique(np.concatenate([local_amb_sel, extra_amb], axis=0)).astype(np.int64)
        local_rem_after_amb = np.setdiff1d(local_rem_after_rel, local_amb_sel, assume_unique=False)
        weak_strength = 0.56 * promote_score[local_rem_after_amb] + 0.26 * conf_n[local_rem_after_amb] + 0.18 * stability_n[local_rem_after_amb] - 0.16 * unknown_score_dyn[local_rem_after_amb] if len(local_rem_after_amb) > 0 else np.zeros((0,), dtype=np.float32)
        if len(local_rem_after_amb) > 0:
            cold_thr = float(np.quantile(weak_strength, V81_COLD_WEAK_Q)) if len(local_rem_after_amb) >= 10 else float(np.median(weak_strength))
            local_weak_sel = local_rem_after_amb[weak_strength >= cold_thr]
            local_cold_sel = local_rem_after_amb[weak_strength < cold_thr]
        else:
            local_weak_sel = np.zeros((0,), dtype=np.int64)
            local_cold_sel = np.zeros((0,), dtype=np.int64)

        idx_rel_curr = idx_rel_new
        idx_amb_curr = idx_amb_new
        idx_weak_curr = idx_gate[local_weak_sel]
        idx_weak_cold_curr = idx_gate[local_cold_sel]

        support_curr = _concat_unique_idx(idx_rel_curr, idx_amb_curr, idx_weak_curr)
        local_support = np.nonzero(np.isin(idx_gate, support_curr))[0]
        if len(local_support) == 0:
            local_support = local_nonunk.copy()
            support_curr = idx_gate[local_support]

        q_local = refine_probs_multi_stage(Zg, P_mix_g, proto_bank, idx_support=local_support, iters=max(REFINE_ITERS, 3), src_mix=0.80)
        q_curr[idx_gate] = q_local.astype(np.float32)

        if len(idx_amb_curr) > 0:
            q_curr[idx_amb_curr] = restrict_topk_probs(q_curr[idx_amb_curr], k=max(2, AMBIG_TOPK), sharpen=1.01)
        if len(idx_weak_curr) > 0:
            q_curr[idx_weak_curr] = restrict_topk_probs(q_curr[idx_weak_curr], k=max(2, V81_SEMIWEAK_TOPK), sharpen=0.99)
        if len(idx_weak_cold_curr) > 0:
            q_curr[idx_weak_cold_curr] = restrict_topk_probs(q_curr[idx_weak_cold_curr], k=2, sharpen=0.96)

        if len(idx_rel_curr) > 0:
            rel_local = np.nonzero(np.isin(idx_gate, idx_rel_curr))[0]
            w_rel_curr = V8_REL_WEIGHT_FLOOR + (1.0 - V8_REL_WEIGHT_FLOOR) * normalize_conf_weights(0.60 * promote_score[rel_local] + 0.40 * conf_g[rel_local])
        else:
            w_rel_curr = np.zeros((0,), dtype=np.float32)
        if len(idx_amb_curr) > 0:
            amb_local = np.nonzero(np.isin(idx_gate, idx_amb_curr))[0]
            w_amb_curr = V8_AMB_WEIGHT_FLOOR + (V8_AMB_WEIGHT_SCALE + 0.06) * normalize_conf_weights(0.55 * promote_score[amb_local] + 0.45 * conf_g[amb_local])
        else:
            w_amb_curr = np.zeros((0,), dtype=np.float32)
        if len(idx_weak_curr) > 0:
            weak_local = np.nonzero(np.isin(idx_gate, idx_weak_curr))[0]
            w_weak_curr = V8_WEAK_WEIGHT_FLOOR + (V8_WEAK_WEIGHT_SCALE + 0.08) * normalize_conf_weights(0.55 * promote_score[weak_local] + 0.25 * conf_g[weak_local] + 0.20 * stability_n[weak_local])
        else:
            w_weak_curr = np.zeros((0,), dtype=np.float32)
        if len(idx_weak_cold_curr) > 0:
            cold_local = np.nonzero(np.isin(idx_gate, idx_weak_cold_curr))[0]
            w_weak_cold_curr = V81_COLD_WEIGHT_FLOOR + V81_COLD_WEIGHT_SCALE * normalize_conf_weights(0.50 * promote_score[cold_local] + 0.50 * conf_g[cold_local])
        else:
            w_weak_cold_curr = np.zeros((0,), dtype=np.float32)

        promo_total_rel += int(len(np.setdiff1d(idx_rel_curr, idx_rel, assume_unique=False)))
        promo_total_amb += int(len(np.setdiff1d(idx_amb_curr, idx_amb, assume_unique=False)))
        refresh_stats.append(dict(
            round=int(rr + 1),
            reliable_size=int(len(idx_rel_curr)),
            ambiguous_size=int(len(idx_amb_curr)),
            weak_size=int(len(idx_weak_curr)),
            cold_weak_size=int(len(idx_weak_cold_curr)),
            unknown_size=int(len(idx_unk_curr)),
            promote_score_gate_mean=float(promote_score.mean()),
            unknown_score_gate_mean=float(unknown_score_dyn.mean()),
            stability_mean=float(stability_n.mean()),
        ))

    info = dict(
        reliable_size=int(len(idx_rel_curr)),
        ambiguous_size=int(len(idx_amb_curr)),
        weak_size=int(len(idx_weak_curr)),
        cold_weak_size=int(len(idx_weak_cold_curr)),
        unknown_size=int(len(idx_unk_curr)),
        reliable_keep=float(len(idx_rel_curr) / max(1, len(Z_target))),
        ambiguous_keep=float(len(idx_amb_curr) / max(1, len(Z_target))),
        weak_keep=float(len(idx_weak_curr) / max(1, len(Z_target))),
        unknown_keep=float(len(idx_unk_curr) / max(1, len(Z_target))),
        promoted_reliable=float(promo_total_rel),
        promoted_ambiguous=float(promo_total_amb),
        stage2_reliable_size=int(len(idx_rel_curr)),
        stage2_ambiguous_size=int(len(idx_amb_curr)),
        stage2_weak_size=int(len(idx_weak_curr)),
        stage2_cold_weak_size=int(len(idx_weak_cold_curr)),
        stage2_unknown_size=int(len(idx_unk_curr)),
        promote_score_gate_mean=float(refresh_stats[-1]["promote_score_gate_mean"]) if len(refresh_stats) > 0 else float("nan"),
        stability_gate_mean=float(refresh_stats[-1]["stability_mean"]) if len(refresh_stats) > 0 else float("nan"),
        refresh_rounds=float(len(refresh_stats)),
    )
    return final_adapter, info


def build_pseudo_method_bank(
    K,
    Z_adapt,
    gate_pred,
    y_true_adapt,
    P_logit_adapt,
    P_proto_adapt,
    P_knn_adapt,
    protos,
    tau_conf,
    Sid_adapt=None,
    Sdom_adapt=None,
    min_keep=64,
):
    idx_gate = np.where(gate_pred == 1)[0]
    idx_all = np.arange(len(gate_pred))
    idx_oracle = np.where(y_true_adapt >= 0)[0]

    if Sid_adapt is None:
        Sid_adapt = np.zeros((len(gate_pred),), dtype=np.float32)
    if Sdom_adapt is None:
        Sdom_adapt = np.zeros((len(gate_pred),), dtype=np.float32)

    P_tri = weighted_prob_fusion([P_logit_adapt, P_proto_adapt, P_knn_adapt], [W_LOGIT, W_PROTO, W_KNN])
    P_lp = weighted_prob_fusion([P_logit_adapt, P_proto_adapt], [W_LOGIT, W_PROTO])

    y_logit = np.argmax(P_logit_adapt, axis=1)
    y_proto = np.argmax(P_proto_adapt, axis=1)
    y_knn = np.argmax(P_knn_adapt, axis=1)
    y_tri = np.argmax(P_tri, axis=1)

    conf_logit = msp_from_probs(P_logit_adapt)
    conf_proto = msp_from_probs(P_proto_adapt)
    conf_knn = msp_from_probs(P_knn_adapt)
    conf_lp = msp_from_probs(P_lp)
    conf_tri = msp_from_probs(P_tri)

    mask_lp_agree = (y_logit == y_proto)
    mask_tri_agree = (y_logit == y_proto) & (y_proto == y_knn)

    idx_lp = select_idx_by_mask_with_fallback(idx_gate, mask_lp_agree[idx_gate], conf_lp, min_keep=min_keep)
    idx_tri = select_idx_by_mask_with_fallback(idx_gate, mask_tri_agree[idx_gate], conf_tri, min_keep=min_keep)

    gate_conf_thr = float(tau_conf)
    if len(idx_gate) > 0:
        gate_conf_thr = max(gate_conf_thr, float(np.quantile(conf_tri[idx_gate], CONF_DRIFT_Q)))

    idx_conf = select_idx_by_mask_with_fallback(idx_gate, conf_tri[idx_gate] >= gate_conf_thr, conf_tri, min_keep=min_keep)
    idx_agree_conf = select_idx_by_mask_with_fallback(
        idx_gate,
        ((mask_tri_agree & (conf_tri >= gate_conf_thr))[idx_gate]),
        conf_tri,
        min_keep=min_keep,
    )

    support_idx = idx_agree_conf if len(idx_agree_conf) > 0 else idx_gate
    P_refine = refine_probs_multi_stage(Z_adapt, P_tri, protos, idx_support=support_idx, iters=REFINE_ITERS)
    conf_refine = msp_from_probs(P_refine)

    idx_rel = select_idx_by_mask_with_fallback(
        idx_gate,
        (((mask_tri_agree | mask_lp_agree) & (conf_tri >= gate_conf_thr))[idx_gate]),
        conf_tri,
        min_keep=min_keep,
    )
    idx_unrel = np.setdiff1d(idx_gate, idx_rel, assume_unique=False)

    idx_common, idx_unknown, common_score, split_info = split_common_unknown_candidates(
        Z_adapt, P_logit_adapt, P_proto_adapt, P_knn_adapt, gate_pred, protos, tau_conf=tau_conf, min_keep=min_keep
    )

    idx_rel3, idx_amb3, idx_unk3, three_aux, three_info = build_three_bucket_partition(
        Z_adapt=Z_adapt,
        gate_pred=gate_pred,
        Sid_adapt=Sid_adapt,
        Sdom_adapt=Sdom_adapt,
        P_logit_adapt=P_logit_adapt,
        P_proto_adapt=P_proto_adapt,
        P_knn_adapt=P_knn_adapt,
        protos=protos,
        tau_conf=tau_conf,
        min_keep=max(16, min_keep // 2),
        reliable_keep_q=RELIABLE_KEEP_Q,
        ambig_keep_q=AMBIG_KEEP_Q,
        unknown_keep_q=UNKNOWN_KEEP_Q,
    )
    P_three = three_aux["P_seed"]
    w_rel3 = three_aux["w_rel"]
    w_amb3 = three_aux["w_amb"]
    idx_sup3 = np.concatenate([idx_rel3, idx_amb3], axis=0) if (len(idx_rel3) + len(idx_amb3)) > 0 else np.zeros((0,), dtype=np.int64)

    idx_rel6, idx_amb6, idx_weak6, idx_unk6, six_aux, six_info = build_three_bucket_partition_v6(
        K=K,
        Z_adapt=Z_adapt,
        gate_pred=gate_pred,
        Sid_adapt=Sid_adapt,
        Sdom_adapt=Sdom_adapt,
        P_logit_adapt=P_logit_adapt,
        P_proto_adapt=P_proto_adapt,
        P_knn_adapt=P_knn_adapt,
        protos=protos,
        tau_conf=tau_conf,
        min_keep=max(16, min_keep // 2),
    )
    P_six = six_aux["P_seed"]
    w_rel6 = six_aux["w_rel"]
    w_amb6 = six_aux["w_amb"]
    w_weak6 = six_aux["w_weak"]
    idx_sup6 = np.concatenate([idx_rel6, idx_amb6, idx_weak6], axis=0) if (len(idx_rel6) + len(idx_amb6) + len(idx_weak6)) > 0 else np.zeros((0,), dtype=np.int64)

    idx_rel7, idx_amb7, idx_weak7, idx_unk7, seven_aux, seven_info = build_three_bucket_partition_v7(
        K=K,
        Z_adapt=Z_adapt,
        gate_pred=gate_pred,
        Sid_adapt=Sid_adapt,
        Sdom_adapt=Sdom_adapt,
        P_logit_adapt=P_logit_adapt,
        P_proto_adapt=P_proto_adapt,
        P_knn_adapt=P_knn_adapt,
        protos=protos,
        tau_conf=tau_conf,
        min_keep=max(16, min_keep // 2),
    )
    P_seven = seven_aux["P_seed"]
    w_rel7 = seven_aux["w_rel"]
    w_amb7 = seven_aux["w_amb"]
    w_weak7 = seven_aux["w_weak"]
    idx_sup7 = np.concatenate([idx_rel7, idx_amb7], axis=0) if (len(idx_rel7) + len(idx_amb7)) > 0 else np.zeros((0,), dtype=np.int64)

    idx_rel_dg, idx_amb_dg, idx_unk_dg, dg_aux, dg_info = build_dg_raincoat_partition(
        K=K,
        Z_adapt=Z_adapt,
        gate_pred=gate_pred,
        Sid_adapt=Sid_adapt,
        Sdom_adapt=Sdom_adapt,
        P_logit_adapt=P_logit_adapt,
        P_proto_adapt=P_proto_adapt,
        P_knn_adapt=P_knn_adapt,
        protos=protos,
        tau_conf=tau_conf,
        min_keep=max(16, min_keep // 2),
    )
    P_dg = dg_aux["P_seed"]
    w_rel_dg = dg_aux["w_rel"]
    w_amb_dg = dg_aux["w_amb"]
    idx_sup_dg = dg_aux.get("support_idx", np.concatenate([idx_rel_dg, idx_amb_dg], axis=0) if (len(idx_rel_dg) + len(idx_amb_dg)) > 0 else np.zeros((0,), dtype=np.int64))

    idx_rel8, idx_amb8, idx_weak8, idx_unk8, eight_aux, eight_info = build_three_bucket_partition_v8(
        K=K,
        Z_adapt=Z_adapt,
        gate_pred=gate_pred,
        Sid_adapt=Sid_adapt,
        Sdom_adapt=Sdom_adapt,
        P_logit_adapt=P_logit_adapt,
        P_proto_adapt=P_proto_adapt,
        P_knn_adapt=P_knn_adapt,
        protos=protos,
        tau_conf=tau_conf,
        min_keep=max(16, min_keep // 2),
    )
    P_eight = eight_aux["P_seed"]
    w_rel8 = eight_aux["w_rel"]
    w_amb8 = eight_aux["w_amb"]
    w_weak8 = eight_aux["w_weak"]
    idx_sup8 = eight_aux.get("support_idx", np.concatenate([idx_rel8, idx_amb8], axis=0) if (len(idx_rel8) + len(idx_amb8)) > 0 else np.zeros((0,), dtype=np.int64))


    methods = {
        "GatedAdapt": dict(train="hard", idx=idx_gate, y=y_logit[idx_gate], q=None, w=np.ones((len(idx_gate),), dtype=np.float32), P=P_logit_adapt),
        "GatedSelfSoft": dict(train="soft", idx=idx_gate, y=None, q=normalize_rows(P_logit_adapt[idx_gate] ** (1.0 / SOFT_T)), w=normalize_conf_weights(conf_logit[idx_gate]), P=P_logit_adapt),
        "GatedProtoAgree": dict(train="hard", idx=idx_lp, y=y_logit[idx_lp], q=None, w=normalize_conf_weights(conf_lp[idx_lp]), P=onehot_from_labels(y_logit, K)),
        "GatedProtoFusionSoft": dict(train="soft", idx=idx_gate, y=None, q=P_lp[idx_gate], w=normalize_conf_weights(conf_lp[idx_gate]), P=P_lp),
        "GatedTriAgree": dict(train="hard", idx=idx_tri, y=y_tri[idx_tri], q=None, w=normalize_conf_weights(conf_tri[idx_tri]), P=onehot_from_labels(y_tri, K)),
        "GatedTriFusionSoft": dict(train="soft", idx=idx_gate, y=None, q=P_tri[idx_gate], w=normalize_conf_weights(conf_tri[idx_gate]), P=P_tri),
        "ConfThreshSoft": dict(train="soft", idx=idx_conf, y=None, q=normalize_rows(P_logit_adapt[idx_conf] ** (1.0 / SOFT_T)), w=normalize_conf_weights(conf_tri[idx_conf]), P=P_logit_adapt),
        "AgreementConfSoft": dict(train="soft", idx=idx_agree_conf, y=None, q=P_tri[idx_agree_conf], w=normalize_conf_weights(conf_tri[idx_agree_conf]), P=P_tri),
        "ProtoRefineSoft": dict(train="soft", idx=idx_gate, y=None, q=P_refine[idx_gate], w=normalize_conf_weights(conf_refine[idx_gate]), P=P_refine),
        "ReliableEntropySplit": dict(
            train="generic", sup_mode="soft", sup_idx=idx_rel, q=P_tri[idx_rel], y=None,
            w=normalize_conf_weights(conf_tri[idx_rel]), align_idx=idx_gate, unrel_idx=idx_unrel,
            lambda_align=0.0, lambda_ent_min=0.0, lambda_ent_max=LAMBDA_ENT_MAX,
            lambda_cons=LAMBDA_CONS * 0.5, lambda_proto=LAMBDA_PROTO * 0.5, dynamic_align=False, P=P_tri,
        ),
        "TwoStageUnknown": dict(
            train="generic", sup_mode="soft", sup_idx=idx_common, q=P_refine[idx_common], y=None,
            w=normalize_conf_weights(common_score[idx_common]) if len(idx_common) > 0 else np.ones((0,), dtype=np.float32),
            align_idx=idx_gate, unrel_idx=idx_unknown,
            lambda_align=LAMBDA_ALIGN * 0.5, lambda_ent_min=LAMBDA_ENT_MIN * 0.5, lambda_ent_max=LAMBDA_ENT_MAX,
            lambda_cons=LAMBDA_CONS, lambda_proto=LAMBDA_PROTO, dynamic_align=False, P=P_refine,
        ),
        "GCODWFA_Lite": dict(
            train="generic", sup_mode="soft", sup_idx=idx_common, q=P_refine[idx_common], y=None,
            w=normalize_conf_weights(common_score[idx_common]) if len(idx_common) > 0 else np.ones((0,), dtype=np.float32),
            align_idx=idx_gate, unrel_idx=idx_unknown,
            lambda_align=GCODWFA_ALIGN_MAX, lambda_ent_min=LAMBDA_ENT_MIN, lambda_ent_max=0.0,
            lambda_cons=LAMBDA_CONS, lambda_proto=LAMBDA_PROTO * 0.5, dynamic_align=True, P=P_refine,
        ),
        "RAINCOAT_Lite": dict(
            train="raincoat", idx_gate=idx_gate, sup_idx=idx_common, unrel_idx=idx_unknown,
            q=P_refine[idx_common] if len(idx_common) > 0 else np.zeros((0, K), dtype=np.float32),
            w=normalize_conf_weights(common_score[idx_common]) if len(idx_common) > 0 else np.ones((0,), dtype=np.float32),
            P=P_refine,
        ),
        "DG_RAINCOAT": dict(
            train="dg_raincoat",
            idx_gate=idx_gate, idx_rel=idx_rel_dg, idx_amb=idx_amb_dg, idx_unk=idx_unk_dg,
            idx_sup_eval=idx_sup_dg,
            q_seed=P_dg, w_rel=w_rel_dg, w_amb=w_amb_dg,
            lambda_align_stage1=DG_LAMBDA_ALIGN_STAGE1, lambda_align_stage2=DG_LAMBDA_ALIGN_STAGE2,
            lambda_ent_min=DG_LAMBDA_ENT_MIN, lambda_ent_max=DG_LAMBDA_ENT_MAX,
            lambda_cons=DG_LAMBDA_CONS, lambda_proto=DG_LAMBDA_PROTO,
            lambda_u_repel=DG_LAMBDA_UNKNOWN_REPEL,
            P=P_dg, bucket_info=dg_info,
        ),
        "ThreeBucketV5Soft": dict(
            train="three_bucket",
            idx_gate=idx_gate, idx_rel=idx_rel3, idx_amb=idx_amb3, idx_unk=idx_unk3,
            idx_sup_eval=idx_sup3,
            q_seed=P_three, w_rel=w_rel3, w_amb=w_amb3,
            use_ema=False, teacher_blend=0.0,
            lambda_rel_sup=1.0, lambda_amb_sup=LAMBDA_AMB_SUP,
            lambda_align=LAMBDA_ALIGN * 0.70, lambda_ent_min=LAMBDA_ENT_MIN * 0.35,
            lambda_ent_max=LAMBDA_ENT_MAX * 0.60, lambda_cons=LAMBDA_CONS,
            lambda_proto=LAMBDA_PROTO * 0.80, lambda_src_proto=LAMBDA_SRC_PROTO,
            lambda_src_logit=LAMBDA_SRC_LOGIT, lambda_u_repel=LAMBDA_UNKNOWN_REPEL * 0.60,
            unknown_warmup_epochs=UNKNOWN_WARMUP_EPOCHS, unknown_ramp_epochs=UNKNOWN_RAMP_EPOCHS,
            P=P_three, bucket_info=three_info,
        ),
        "ThreeBucketV5EMA": dict(
            train="three_bucket",
            idx_gate=idx_gate, idx_rel=idx_rel3, idx_amb=idx_amb3, idx_unk=idx_unk3,
            idx_sup_eval=idx_sup3,
            q_seed=P_three, w_rel=w_rel3, w_amb=w_amb3,
            use_ema=True, teacher_blend=EMA_TEACHER_BLEND,
            lambda_rel_sup=1.0, lambda_amb_sup=LAMBDA_AMB_SUP,
            lambda_align=LAMBDA_ALIGN * 0.72, lambda_ent_min=LAMBDA_ENT_MIN * 0.30,
            lambda_ent_max=LAMBDA_ENT_MAX * 0.55, lambda_cons=LAMBDA_CONS,
            lambda_proto=LAMBDA_PROTO, lambda_src_proto=LAMBDA_SRC_PROTO,
            lambda_src_logit=LAMBDA_SRC_LOGIT, lambda_u_repel=LAMBDA_UNKNOWN_REPEL * 0.70,
            unknown_warmup_epochs=UNKNOWN_WARMUP_EPOCHS, unknown_ramp_epochs=UNKNOWN_RAMP_EPOCHS,
            P=P_three, bucket_info=three_info,
        ),
        "ThreeBucketV5EMAProto": dict(
            train="three_bucket",
            idx_gate=idx_gate, idx_rel=idx_rel3, idx_amb=idx_amb3, idx_unk=idx_unk3,
            idx_sup_eval=idx_sup3,
            q_seed=P_three, w_rel=w_rel3, w_amb=w_amb3,
            use_ema=True, teacher_blend=min(0.72, EMA_TEACHER_BLEND + 0.05),
            lambda_rel_sup=1.0, lambda_amb_sup=LAMBDA_AMB_SUP * 0.95,
            lambda_align=LAMBDA_ALIGN * 0.68, lambda_ent_min=LAMBDA_ENT_MIN * 0.25,
            lambda_ent_max=LAMBDA_ENT_MAX * 0.50, lambda_cons=LAMBDA_CONS,
            lambda_proto=LAMBDA_PROTO * 1.15, lambda_src_proto=LAMBDA_SRC_PROTO * 1.25,
            lambda_src_logit=LAMBDA_SRC_LOGIT * 1.20, lambda_u_repel=LAMBDA_UNKNOWN_REPEL * 0.85,
            unknown_warmup_epochs=UNKNOWN_WARMUP_EPOCHS, unknown_ramp_epochs=UNKNOWN_RAMP_EPOCHS,
            P=P_three, bucket_info=three_info,
        ),
        "ThreeBucketV6Curriculum": dict(
            train="three_bucket_v6",
            idx_gate=idx_gate, idx_rel=idx_rel6, idx_amb=idx_amb6, idx_weak=idx_weak6, idx_unk=idx_unk6,
            idx_sup_eval=idx_sup6,
            q_seed=P_six, w_rel=w_rel6, w_amb=w_amb6, w_weak=w_weak6,
            use_ema=True, teacher_blend=EMA_TEACHER_BLEND,
            weak_teacher_blend=V6_WEAK_TEACHER_BLEND, weak_start_epoch=V6_WEAK_START_EPOCH,
            lambda_rel_sup=1.0, lambda_amb_sup=V6_LAMBDA_AMB_SUP, lambda_weak_sup=V6_LAMBDA_WEAK_SUP,
            lambda_align=V6_LAMBDA_ALIGN, lambda_ent_min=V6_LAMBDA_ENT_MIN, lambda_ent_max=V6_LAMBDA_ENT_MAX,
            lambda_cons=V6_LAMBDA_CONS, lambda_proto=V6_LAMBDA_PROTO,
            lambda_src_proto=LAMBDA_SRC_PROTO, lambda_src_logit=LAMBDA_SRC_LOGIT,
            lambda_u_repel=V6_LAMBDA_UNKNOWN_REPEL,
            unknown_warmup_epochs=V6_UNKNOWN_WARMUP_EPOCHS, unknown_ramp_epochs=V6_UNKNOWN_RAMP_EPOCHS,
            P=P_six, bucket_info=six_info,
        ),
        "ThreeBucketV7Promotion": dict(
            train="three_bucket_v7",
            idx_gate=idx_gate, idx_rel=idx_rel7, idx_amb=idx_amb7, idx_weak=idx_weak7, idx_unk=idx_unk7,
            idx_sup_eval=idx_sup7,
            q_seed=P_seven, w_rel=w_rel7, w_amb=w_amb7, w_weak=w_weak7,
            bucket_score=seven_aux["bucket_score"], unknown_score=seven_aux["unknown_score"],
            use_ema=True, teacher_blend=EMA_TEACHER_BLEND,
            stage1_epochs=V7_STAGE1_EPOCHS, promote_blend=V7_PROMOTE_BLEND,
            promote_rel_fraction=V7_PROMOTE_REL_FRACTION, promote_amb_fraction=V7_PROMOTE_AMB_FRACTION,
            promote_conf=V7_PROMOTE_CONF, promote_margin=V7_PROMOTE_MARGIN,
            lambda_rel_sup=1.0, lambda_amb_sup=V7_LAMBDA_AMB_SUP, lambda_weak_sup=V7_LAMBDA_WEAK_SUP,
            lambda_align=V7_LAMBDA_ALIGN, lambda_ent_min=V7_LAMBDA_ENT_MIN, lambda_ent_max=V7_LAMBDA_ENT_MAX,
            lambda_cons=V7_LAMBDA_CONS, lambda_proto=V7_LAMBDA_PROTO,
            lambda_src_proto=LAMBDA_SRC_PROTO * 0.60, lambda_src_logit=LAMBDA_SRC_LOGIT * 0.60,
            lambda_u_repel=V7_LAMBDA_UNKNOWN_REPEL,
            unknown_warmup_epochs=V7_UNKNOWN_WARMUP_EPOCHS, unknown_ramp_epochs=V7_UNKNOWN_RAMP_EPOCHS,
            P=P_seven, bucket_info=seven_info,
        ),
        "ThreeBucketV8Adaptive": dict(
            train="three_bucket_v8",
            idx_gate=idx_gate, idx_rel=idx_rel8, idx_amb=idx_amb8, idx_weak=idx_weak8, idx_unk=idx_unk8,
            idx_sup_eval=idx_sup8,
            q_seed=P_eight, w_rel=w_rel8, w_amb=w_amb8, w_weak=w_weak8,
            idx_weak_cold=eight_aux.get("idx_weak_cold", np.zeros((0,), dtype=np.int64)),
            w_weak_cold=eight_aux.get("w_weak_cold", np.zeros((0,), dtype=np.float32)),
            bucket_score=eight_aux["bucket_score"], unknown_score=eight_aux["unknown_score"],
            lambda_align=V8_LAMBDA_ALIGN, lambda_ent_min=V8_LAMBDA_ENT_MIN, lambda_ent_max=V8_LAMBDA_ENT_MAX,
            lambda_cons=V8_LAMBDA_CONS, lambda_proto=V8_LAMBDA_PROTO,
            lambda_u_repel=V8_LAMBDA_UNKNOWN_REPEL,
            P=P_eight, bucket_info=eight_info,
        ),
        "UngatedAdapt": dict(train="hard", idx=idx_all, y=y_logit[idx_all], q=None, w=np.ones((len(idx_all),), dtype=np.float32), P=P_logit_adapt),
        "OracleGatePseudoAdapt": dict(train="hard", idx=idx_oracle, y=y_logit[idx_oracle], q=None, w=np.ones((len(idx_oracle),), dtype=np.float32), P=P_logit_adapt),
        "OracleLabelAdapt": dict(train="hard", idx=idx_oracle, y=y_true_adapt[idx_oracle], q=None, w=np.ones((len(idx_oracle),), dtype=np.float32), P=onehot_from_labels(np.clip(y_true_adapt, 0, K-1), K)),
    }

    info = {}
    total_known = max(1, int(np.sum(y_true_adapt >= 0)))
    for name, spec in methods.items():
        sel_idx = spec.get("idx", spec.get("sup_idx", spec.get("idx_sup_eval", np.zeros((0,), dtype=np.int64))))
        sel_idx = np.asarray(sel_idx, dtype=np.int64)
        stats = summarize_method_on_selected(sel_idx, y_true_adapt, spec["P"])
        sel_precision = float((y_true_adapt[sel_idx] >= 0).mean()) if len(sel_idx) > 0 else float("nan")
        sel_recall = float(np.sum(y_true_adapt[sel_idx] >= 0) / total_known) if len(sel_idx) > 0 else 0.0
        sel_keep = float(len(sel_idx) / max(1, len(y_true_adapt)))

        unknown_idx = np.asarray(spec.get("unrel_idx", spec.get("idx_unk", np.zeros((0,), dtype=np.int64))), dtype=np.int64)
        unknown_keep = float(len(unknown_idx) / max(1, len(y_true_adapt)))
        unknown_precision = float((y_true_adapt[unknown_idx] < 0).mean()) if len(unknown_idx) > 0 else float("nan")

        info[name] = dict(
            sel_precision=sel_precision,
            sel_recall=sel_recall,
            sel_keep_ratio=sel_keep,
            sel_size=stats["sel_size"],
            pseudo_acc_selected=stats["pseudo_acc"],
            pseudo_acc=stats["pseudo_acc"],
            unknown_cand_keep=unknown_keep,
            unknown_cand_precision=unknown_precision,
        )
        if "bucket_info" in spec:
            info[name].update(spec["bucket_info"])

    aux = dict(
        P_lp=P_lp, P_tri=P_tri, P_refine=P_refine, common_score=common_score, split_info=split_info,
        P_three=P_three, three_bucket=three_info, three_threshold_reliable=three_aux["threshold_reliable"],
        three_threshold_unknown=three_aux["threshold_unknown"], three_bucket_score=three_aux["bucket_score"],
        P_v6=P_six, v6_bucket=six_info, v6_threshold_reliable=six_aux["threshold_reliable"],
        v6_threshold_unknown=six_aux["threshold_unknown"], v6_bucket_score=six_aux["bucket_score"],
    )
    return methods, info, aux



## Evaluation block

This section computes:
- stable-domain accuracy
- drift-domain accuracy
- unknown rejection metrics
- open-set AUC from diagnosis scores

If a new method has good `drift_acc_all` but poor `FAR_unknown_accept` or poor `FRR_stable`,
it is **not** considered a good final operating point.

In [9]:
def evaluate_variant(adapter, model_fc, Z_A, y_A, D_A, Z_B, y_B, D_B, Z_C, y_C, D_C, Z_U, D_U,
                     mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A, models_kr, fallback_k, source_rx_ids,
                     tau_id, tau_dom):
    Z_A2, logits_A2 = apply_adapter_and_fc(adapter, model_fc, Z_A, batch=512)
    Z_B2, logits_B2 = apply_adapter_and_fc(adapter, model_fc, Z_B, batch=512)
    Z_C2, logits_C2 = apply_adapter_and_fc(adapter, model_fc, Z_C, batch=512)
    Z_U2, logits_U2 = apply_adapter_and_fc(adapter, model_fc, Z_U, batch=512)

    sc_A = compute_module2_scores_from_cached(Z_A2, logits_A2, D_A, mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A, models_kr, fallback_k, source_rx_ids, tau_id=tau_id, tau_dom=tau_dom)
    sc_B = compute_module2_scores_from_cached(Z_B2, logits_B2, D_B, mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A, models_kr, fallback_k, source_rx_ids, tau_id=tau_id, tau_dom=tau_dom)
    sc_C = compute_module2_scores_from_cached(Z_C2, logits_C2, D_C, mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A, models_kr, fallback_k, source_rx_ids, tau_id=tau_id, tau_dom=tau_dom)
    sc_U = compute_module2_scores_from_cached(Z_U2, logits_U2, D_U, mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A, models_kr, fallback_k, source_rx_ids, tau_id=tau_id, tau_dom=tau_dom)

    stable_acc = closed_set_acc_from_logits(logits_A2, y_A)
    drift_acc_rx = closed_set_acc_from_logits(logits_B2, y_B)
    drift_acc_day = closed_set_acc_from_logits(logits_C2, y_C)
    drift_acc_all = float((np.sum(np.argmax(logits_B2,1) == y_B) + np.sum(np.argmax(logits_C2,1) == y_C)) / (len(y_B) + len(y_C)))

    FRR_stable = float(1.0 - (sc_A["gate_pred"] == 0).mean())
    FAR_unknown_accept = float((sc_U["gate_pred"] == 0).mean())
    miss_drift_rx = float((sc_B["gate_pred"] == 0).mean())
    miss_drift_day = float((sc_C["gate_pred"] == 0).mean())
    miss_drift_all = float((np.sum(sc_B["gate_pred"] == 0) + np.sum(sc_C["gate_pred"] == 0)) / (len(sc_B["gate_pred"]) + len(sc_C["gate_pred"])))
    sid_A = np.nan_to_num(sc_A["Sid"], nan=-1e6, posinf=1e6, neginf=-1e6)
    sid_U = np.nan_to_num(sc_U["Sid"], nan=-1e6, posinf=1e6, neginf=-1e6)
    auc_unknown_eval = roc_auc(np.concatenate([np.zeros_like(sid_A, dtype=np.int64), np.ones_like(sid_U, dtype=np.int64)]), np.concatenate([-sid_A, -sid_U]))

    return dict(
        stable_acc=stable_acc,
        drift_acc_rx=drift_acc_rx,
        drift_acc_day=drift_acc_day,
        drift_acc_all=drift_acc_all,
        FRR_stable=FRR_stable,
        FAR_unknown_accept=FAR_unknown_accept,
        miss_drift_rx=miss_drift_rx,
        miss_drift_day=miss_drift_day,
        miss_drift_all=miss_drift_all,
        auc_unknown_eval=auc_unknown_eval,
    )



## Main experiment loop

This section does the full experiment:
- iterate over unique TX splits
- iterate over folds
- train / adapt each method
- evaluate all methods on stable, drift, and unknown sets
- save per-fold metrics and diagnostic figures

When adding a new method in future versions, update:
- method configuration list
- fold print abbreviations
- final summary aggregation

In [10]:

def run_one_split(split_id, KNOWN_TX, UNKNOWN_TX):
    split_dir = os.path.join(RUN_DIR, f"txsplit_{split_id}")
    os.makedirs(split_dir, exist_ok=True)
    with open(os.path.join(split_dir, "tx_split.json"), "w", encoding="utf-8") as f:
        json.dump({"KNOWN_TX": KNOWN_TX, "UNKNOWN_TX": UNKNOWN_TX, "SOURCE_RXS": SOURCE_RXS, "DRIFT_RX": DRIFT_RX}, f, indent=2)

    X_src, y_src, D_src, rx_src = build_source_train(compact_dataset, KNOWN_TX)
    K = len(KNOWN_TX)

    X_B, y_B, D_B = build_eval_set(compact_dataset, KNOWN_TX, KNOWN_TX, DRIFT_RX, TEST_DATES_RX)
    X_C, y_C, D_C = build_eval_set(compact_dataset, KNOWN_TX, KNOWN_TX, SOURCE_RXS, TEST_DATES_DAY)
    X_D, y_D, D_D = build_eval_set(compact_dataset, KNOWN_TX, UNKNOWN_TX, SOURCE_RXS, TEST_DATES_RX)
    X_E, y_E, D_E = build_eval_set(compact_dataset, KNOWN_TX, UNKNOWN_TX, DRIFT_RX, TEST_DATES_RX)
    X_F, y_F, D_F = build_eval_set(compact_dataset, KNOWN_TX, UNKNOWN_TX, SOURCE_RXS, TEST_DATES_DAY)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED + 1000*split_id)
    rows = []

    for fold, (idx_tr, idx_te) in enumerate(skf.split(X_src, y_src), start=1):
        fold_dir = os.path.join(split_dir, f"fold_{fold}")
        os.makedirs(fold_dir, exist_ok=True)

        rng = np.random.RandomState(SEED + 1000*split_id + fold)
        perm = rng.permutation(idx_tr)
        n_val = max(1, int(0.1 * len(perm)))
        idx_val = perm[:n_val]
        idx_train = perm[n_val:]

        train_loader = DataLoader(ArrayDataset(X_src[idx_train], y_src[idx_train]), batch_size=BATCH_SIZE, shuffle=True)
        val_loader   = DataLoader(ArrayDataset(X_src[idx_val],   y_src[idx_val]),   batch_size=BATCH_SIZE, shuffle=False)

        model = ResNet18_1D(num_classes=K, in_planes=IN_PLANES, dropout=DROPOUT).to(device)
        opt = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

        best_val = -1.0
        best_state = None
        patience = 0
        for ep in range(1, EPOCHS + 1):
            _ = run_epoch(model, train_loader, optimizer=opt)
            _, va_acc = run_epoch(model, val_loader, optimizer=None)
            if va_acc > best_val + 1e-4:
                best_val = va_acc
                best_state = {k:v.cpu().clone() for k,v in model.state_dict().items()}
                patience = 0
            else:
                patience += 1
                if patience >= PATIENCE:
                    break

        model.load_state_dict(best_state)
        torch.save(best_state, os.path.join(fold_dir, "best_source_model.pth"))

        logits_tr, Z_tr = infer_logits_embed(model, X_src[idx_train], batch=512)
        logits_A, Z_A = infer_logits_embed(model, X_src[idx_te], batch=512)
        D_A = D_src[idx_te]

        mu_z_src, var_z_src = fit_emb_maha_diag(Z_tr, y_src[idx_train], K)
        ref_sid_emb_A = sid_embmaha(Z_A, mu_z_src, var_z_src)
        ref_sid_en_A  = sid_energy(logits_A)
        source_rx_ids = sorted({RX_USE.index(r) for r in SOURCE_RXS})
        models_kr, fallback_k = fit_device_rx_models(D_src[idx_train], y_src[idx_train], rx_src[idx_train], K, source_rx_ids, reg=1e-3, min_n=20)
        sc_A = compute_module2_scores_from_cached(Z_A, logits_A, D_A, mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A, models_kr, fallback_k, source_rx_ids)
        tau_id, tau_dom = calibrate_module2_thresholds(sc_A["Sid"], sc_A["Sdom"], FRR_TARGET, FALSE_DRIFT_TARGET)

        protos = fit_class_prototypes(Z_tr, y_src[idx_train], K, l2norm=True)
        Zm, ym = build_knn_memory(Z_tr, y_src[idx_train], per_class=KNN_MEM_PER_CLASS, seed=SEED + 31*fold)

        P_logit_A = softmax_np(logits_A)
        tau_conf = float(np.quantile(msp_from_probs(P_logit_A), CONF_STABLE_Q))

        B_sp = split_adapt_eval(X_B, y_B, D_B, frac=ADAPT_FRAC, max_adapt=MAX_ADAPT_PER_SET, max_eval=MAX_EVAL_PER_SET, seed=10000+fold)
        C_sp = split_adapt_eval(X_C, y_C, D_C, frac=ADAPT_FRAC, max_adapt=MAX_ADAPT_PER_SET, max_eval=MAX_EVAL_PER_SET, seed=11000+fold)
        D_sp = split_adapt_eval(X_D, y_D, D_D, frac=ADAPT_FRAC, max_adapt=MAX_ADAPT_PER_SET, max_eval=MAX_EVAL_PER_SET, seed=12000+fold)
        E_sp = split_adapt_eval(X_E, y_E, D_E, frac=ADAPT_FRAC, max_adapt=MAX_ADAPT_PER_SET, max_eval=MAX_EVAL_PER_SET, seed=13000+fold)
        F_sp = split_adapt_eval(X_F, y_F, D_F, frac=ADAPT_FRAC, max_adapt=MAX_ADAPT_PER_SET, max_eval=MAX_EVAL_PER_SET, seed=14000+fold)

        X_adapt, y_adapt, D_adapt = concat_sets([
            (B_sp[0], B_sp[1], B_sp[2]),
            (C_sp[0], C_sp[1], C_sp[2]),
            (D_sp[0], D_sp[1], D_sp[2]),
            (E_sp[0], E_sp[1], E_sp[2]),
            (F_sp[0], F_sp[1], F_sp[2]),
        ])

        X_B_ev, y_B_ev, D_B_ev = B_sp[3], B_sp[4], B_sp[5]
        X_C_ev, y_C_ev, D_C_ev = C_sp[3], C_sp[4], C_sp[5]
        X_U_ev, y_U_ev, D_U_ev = concat_sets([
            (D_sp[3], D_sp[4], D_sp[5]),
            (E_sp[3], E_sp[4], E_sp[5]),
            (F_sp[3], F_sp[4], F_sp[5]),
        ])

        logits_adapt, Z_adapt = infer_logits_embed(model, X_adapt, batch=512)
        sc_adapt = compute_module2_scores_from_cached(Z_adapt, logits_adapt, D_adapt, mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A, models_kr, fallback_k, source_rx_ids, tau_id=tau_id, tau_dom=tau_dom)
        gate_pred_adapt = sc_adapt["gate_pred"]

        P_logit_adapt = softmax_np(logits_adapt)
        P_proto_adapt = proto_probs_cosine(Z_adapt, protos, temp=PROTO_T, l2norm=True)
        P_knn_adapt   = knn_class_probs(Z_adapt, Zm, ym, K, topk=KNN_TOPK)

        methods, method_info, aux = build_pseudo_method_bank(
            K=K,
            Z_adapt=Z_adapt,
            gate_pred=gate_pred_adapt,
            y_true_adapt=y_adapt,
            P_logit_adapt=P_logit_adapt,
            P_proto_adapt=P_proto_adapt,
            P_knn_adapt=P_knn_adapt,
            protos=protos,
            tau_conf=tau_conf,
            Sid_adapt=sc_adapt["Sid"],
            Sdom_adapt=sc_adapt["Sdom"],
            min_keep=PSEUDO_MIN_KEEP,
        )

        logits_B_ev, Z_B_ev = infer_logits_embed(model, X_B_ev, batch=512)
        logits_C_ev, Z_C_ev = infer_logits_embed(model, X_C_ev, batch=512)
        logits_U_ev, Z_U_ev = infer_logits_embed(model, X_U_ev, batch=512)
        P_logit_B = softmax_np(logits_B_ev)
        P_proto_B = proto_probs_cosine(Z_B_ev, protos, temp=PROTO_T, l2norm=True)
        P_knn_B   = knn_class_probs(Z_B_ev, Zm, ym, K, topk=KNN_TOPK)
        P_lp_B    = weighted_prob_fusion([P_logit_B, P_proto_B], [W_LOGIT, W_PROTO])
        P_tri_B   = weighted_prob_fusion([P_logit_B, P_proto_B, P_knn_B], [W_LOGIT, W_PROTO, W_KNN])

        P_logit_C = softmax_np(logits_C_ev)
        P_proto_C = proto_probs_cosine(Z_C_ev, protos, temp=PROTO_T, l2norm=True)
        P_knn_C   = knn_class_probs(Z_C_ev, Zm, ym, K, topk=KNN_TOPK)
        P_lp_C    = weighted_prob_fusion([P_logit_C, P_proto_C], [W_LOGIT, W_PROTO])
        P_tri_C   = weighted_prob_fusion([P_logit_C, P_proto_C, P_knn_C], [W_LOGIT, W_PROTO, W_KNN])

        pseudo_eval = {
            "logit": dict(rx=pseudo_acc_from_probs(P_logit_B, y_B_ev), day=pseudo_acc_from_probs(P_logit_C, y_C_ev)),
            "lp":    dict(rx=pseudo_acc_from_probs(P_lp_B,    y_B_ev), day=pseudo_acc_from_probs(P_lp_C,    y_C_ev)),
            "tri":   dict(rx=pseudo_acc_from_probs(P_tri_B,   y_B_ev), day=pseudo_acc_from_probs(P_tri_C,   y_C_ev)),
        }
        pseudo_eval["logit"]["all"] = float((np.sum(np.argmax(P_logit_B,1) == y_B_ev) + np.sum(np.argmax(P_logit_C,1) == y_C_ev)) / (len(y_B_ev) + len(y_C_ev)))
        pseudo_eval["lp"]["all"]    = float((np.sum(np.argmax(P_lp_B,1)    == y_B_ev) + np.sum(np.argmax(P_lp_C,1)    == y_C_ev)) / (len(y_B_ev) + len(y_C_ev)))
        pseudo_eval["tri"]["all"]   = float((np.sum(np.argmax(P_tri_B,1)   == y_B_ev) + np.sum(np.argmax(P_tri_C,1)   == y_C_ev)) / (len(y_B_ev) + len(y_C_ev)))

        Z_replay, y_replay = make_replay_subset(Z_tr, y_src[idx_train], per_class=REPLAY_PER_CLASS, seed=SEED+fold)

        adapters = {"NoAdapt": None}
        extra_method_info = {}
        for name, spec in methods.items():
            if spec["train"] == "hard":
                idx = spec["idx"]
                adapters[name] = adapt_on_embeddings_hard(
                    model.fc, Z_adapt[idx], spec["y"], Z_replay, y_replay, target_weights=spec["w"],
                    bottleneck=ADAPTER_BOTTLENECK, scale=ADAPTER_SCALE, epochs=ADAPT_EPOCHS,
                    lr=ADAPT_LR, wd=ADAPT_WEIGHT_DECAY, batch=ADAPT_BATCH,
                    lambda_replay_ce=LAMBDA_REPLAY_CE, lambda_replay_anchor=LAMBDA_REPLAY_ANCHOR
                )
            elif spec["train"] == "soft":
                idx = spec["idx"]
                adapters[name] = adapt_on_embeddings_soft(
                    model.fc, Z_adapt[idx], spec["q"], Z_replay, y_replay, target_weights=spec["w"],
                    bottleneck=ADAPTER_BOTTLENECK, scale=ADAPTER_SCALE, epochs=ADAPT_EPOCHS,
                    lr=ADAPT_LR, wd=ADAPT_WEIGHT_DECAY, batch=ADAPT_BATCH,
                    lambda_replay_ce=LAMBDA_REPLAY_CE, lambda_replay_anchor=LAMBDA_REPLAY_ANCHOR
                )
            elif spec["train"] == "generic":
                sup_idx = spec["sup_idx"]
                align_idx = spec.get("align_idx", np.zeros((0,), dtype=np.int64))
                unrel_idx = spec.get("unrel_idx", np.zeros((0,), dtype=np.int64))
                adapters[name] = adapt_on_embeddings_generic(
                    fc_layer=model.fc,
                    Z_replay=Z_replay, y_replay=y_replay,
                    Z_sup=Z_adapt[sup_idx] if len(sup_idx) > 0 else None,
                    y_sup=spec.get("y", None),
                    q_sup=spec.get("q", None),
                    w_sup=spec.get("w", None),
                    Z_align=Z_adapt[align_idx] if len(align_idx) > 0 else None,
                    Z_unrel=Z_adapt[unrel_idx] if len(unrel_idx) > 0 else None,
                    proto_bank=protos,
                    bottleneck=ADAPTER_BOTTLENECK, scale=ADAPTER_SCALE, epochs=ADAPT_EPOCHS,
                    lr=ADAPT_LR, wd=ADAPT_WEIGHT_DECAY, batch=ADAPT_BATCH,
                    lambda_replay_ce=LAMBDA_REPLAY_CE, lambda_replay_anchor=LAMBDA_REPLAY_ANCHOR,
                    lambda_align=spec.get("lambda_align", 0.0),
                    lambda_ent_min=spec.get("lambda_ent_min", 0.0),
                    lambda_ent_max=spec.get("lambda_ent_max", 0.0),
                    lambda_cons=spec.get("lambda_cons", 0.0),
                    lambda_proto=spec.get("lambda_proto", 0.0),
                    dynamic_align=spec.get("dynamic_align", False),
                )
            elif spec["train"] == "three_bucket":
                adapters[name] = adapt_three_bucket_v5(
                    fc_layer=model.fc,
                    Z_replay=Z_replay, y_replay=y_replay,
                    Z_target=Z_adapt,
                    idx_gate=spec["idx_gate"],
                    idx_rel=spec["idx_rel"],
                    idx_amb=spec["idx_amb"],
                    idx_unk=spec["idx_unk"],
                    q_seed=spec["q_seed"],
                    w_rel=spec.get("w_rel", None),
                    w_amb=spec.get("w_amb", None),
                    proto_bank=protos,
                    bottleneck=ADAPTER_BOTTLENECK, scale=ADAPTER_SCALE,
                    epochs=THREE_BUCKET_EPOCHS,
                    lr=ADAPT_LR, wd=ADAPT_WEIGHT_DECAY, batch=ADAPT_BATCH,
                    lambda_replay_ce=LAMBDA_REPLAY_CE, lambda_replay_anchor=LAMBDA_REPLAY_ANCHOR,
                    lambda_rel_sup=spec.get("lambda_rel_sup", 1.0),
                    lambda_amb_sup=spec.get("lambda_amb_sup", LAMBDA_AMB_SUP),
                    lambda_align=spec.get("lambda_align", LAMBDA_ALIGN),
                    lambda_ent_min=spec.get("lambda_ent_min", LAMBDA_ENT_MIN),
                    lambda_ent_max=spec.get("lambda_ent_max", LAMBDA_ENT_MAX),
                    lambda_cons=spec.get("lambda_cons", LAMBDA_CONS),
                    lambda_proto=spec.get("lambda_proto", LAMBDA_PROTO),
                    lambda_src_proto=spec.get("lambda_src_proto", LAMBDA_SRC_PROTO),
                    lambda_u_repel=spec.get("lambda_u_repel", LAMBDA_UNKNOWN_REPEL),
                    lambda_src_logit=spec.get("lambda_src_logit", LAMBDA_SRC_LOGIT),
                    use_ema=spec.get("use_ema", True),
                    ema_momentum=EMA_MOMENTUM,
                    teacher_blend=spec.get("teacher_blend", EMA_TEACHER_BLEND),
                    unknown_warmup_epochs=spec.get("unknown_warmup_epochs", UNKNOWN_WARMUP_EPOCHS),
                    unknown_ramp_epochs=spec.get("unknown_ramp_epochs", UNKNOWN_RAMP_EPOCHS),
                )
                extra_method_info[name] = spec.get("bucket_info", {})

            elif spec["train"] == "raincoat":
                adapters[name], rain_info = adapt_raincoat_lite(
                    fc_layer=model.fc,
                    Z_target=Z_adapt,
                    idx_gate=spec["idx_gate"],
                    idx_common_seed=spec["sup_idx"],
                    idx_unknown_seed=spec["unrel_idx"],
                    Z_replay=Z_replay, y_replay=y_replay,
                    protos=protos, tau_conf=tau_conf, min_keep=PSEUDO_MIN_KEEP,
                    bottleneck=ADAPTER_BOTTLENECK, scale=ADAPTER_SCALE,
                    epochs_stage1=RAIN_STAGE1_EPOCHS, epochs_stage2=RAIN_STAGE2_EPOCHS,
                    lr=ADAPT_LR, wd=ADAPT_WEIGHT_DECAY, batch=ADAPT_BATCH,
                    lambda_replay_ce=LAMBDA_REPLAY_CE, lambda_replay_anchor=LAMBDA_REPLAY_ANCHOR,
                )
                extra_method_info[name] = rain_info
            elif spec["train"] == "dg_raincoat":
                adapters[name], dg_info = adapt_dg_raincoat_lite(
                    fc_layer=model.fc,
                    Z_replay=Z_replay, y_replay=y_replay,
                    Z_target=Z_adapt,
                    idx_gate=spec["idx_gate"], idx_rel=spec["idx_rel"], idx_amb=spec["idx_amb"], idx_unk=spec["idx_unk"],
                    q_seed=spec["q_seed"], w_rel=spec["w_rel"], w_amb=spec["w_amb"],
                    protos=protos,
                    Sid_adapt=sc_adapt["Sid"], Sdom_adapt=sc_adapt["Sdom"],
                    bottleneck=ADAPTER_BOTTLENECK, scale=ADAPTER_SCALE,
                    epochs_stage1=DG_STAGE1_EPOCHS, epochs_stage2=DG_STAGE2_EPOCHS,
                    lr=ADAPT_LR, wd=ADAPT_WEIGHT_DECAY, batch=ADAPT_BATCH,
                    lambda_replay_ce=LAMBDA_REPLAY_CE, lambda_replay_anchor=LAMBDA_REPLAY_ANCHOR,
                    lambda_align_stage1=spec.get("lambda_align_stage1", DG_LAMBDA_ALIGN_STAGE1),
                    lambda_align_stage2=spec.get("lambda_align_stage2", DG_LAMBDA_ALIGN_STAGE2),
                    lambda_ent_min=spec.get("lambda_ent_min", DG_LAMBDA_ENT_MIN),
                    lambda_ent_max=spec.get("lambda_ent_max", DG_LAMBDA_ENT_MAX),
                    lambda_cons=spec.get("lambda_cons", DG_LAMBDA_CONS),
                    lambda_proto=spec.get("lambda_proto", DG_LAMBDA_PROTO),
                    lambda_u_repel=spec.get("lambda_u_repel", DG_LAMBDA_UNKNOWN_REPEL),
                )
                extra_method_info[name] = {**spec.get("bucket_info", {}), **(dg_info or {})}
            elif spec["train"] == "three_bucket_v6":
                adapters[name] = adapt_three_bucket_v6(
                    fc_layer=model.fc,
                    Z_replay=Z_replay, y_replay=y_replay,
                    Z_target=Z_adapt,
                    idx_gate=spec["idx_gate"], idx_rel=spec["idx_rel"], idx_amb=spec["idx_amb"],
                    idx_weak=spec["idx_weak"], idx_unk=spec["idx_unk"],
                    q_seed=spec["q_seed"], w_rel=spec["w_rel"], w_amb=spec["w_amb"], w_weak=spec["w_weak"],
                    proto_bank=protos,
                    idx_weak_cold=spec.get("idx_weak_cold", None), w_weak_cold=spec.get("w_weak_cold", None),
                    bottleneck=ADAPTER_BOTTLENECK, scale=ADAPTER_SCALE, epochs=THREE_BUCKET_EPOCHS,
                    lr=ADAPT_LR, wd=ADAPT_WEIGHT_DECAY, batch=ADAPT_BATCH,
                    lambda_replay_ce=LAMBDA_REPLAY_CE, lambda_replay_anchor=LAMBDA_REPLAY_ANCHOR,
                    lambda_rel_sup=spec.get("lambda_rel_sup", 1.0),
                    lambda_amb_sup=spec.get("lambda_amb_sup", V6_LAMBDA_AMB_SUP),
                    lambda_weak_sup=spec.get("lambda_weak_sup", V6_LAMBDA_WEAK_SUP),
                    lambda_align=spec.get("lambda_align", V6_LAMBDA_ALIGN),
                    lambda_ent_min=spec.get("lambda_ent_min", V6_LAMBDA_ENT_MIN),
                    lambda_ent_max=spec.get("lambda_ent_max", V6_LAMBDA_ENT_MAX),
                    lambda_cons=spec.get("lambda_cons", V6_LAMBDA_CONS),
                    lambda_proto=spec.get("lambda_proto", V6_LAMBDA_PROTO),
                    lambda_src_proto=spec.get("lambda_src_proto", LAMBDA_SRC_PROTO),
                    lambda_src_logit=spec.get("lambda_src_logit", LAMBDA_SRC_LOGIT),
                    lambda_u_repel=spec.get("lambda_u_repel", V6_LAMBDA_UNKNOWN_REPEL),
                    use_ema=spec.get("use_ema", True),
                    teacher_blend=spec.get("teacher_blend", EMA_TEACHER_BLEND),
                    weak_teacher_blend=spec.get("weak_teacher_blend", V6_WEAK_TEACHER_BLEND),
                    weak_start_epoch=spec.get("weak_start_epoch", V6_WEAK_START_EPOCH),
                    unknown_warmup_epochs=spec.get("unknown_warmup_epochs", V6_UNKNOWN_WARMUP_EPOCHS),
                    unknown_ramp_epochs=spec.get("unknown_ramp_epochs", V6_UNKNOWN_RAMP_EPOCHS),
                )
                extra_method_info[name] = spec.get("bucket_info", {})
            elif spec["train"] == "three_bucket_v7":
                adapters[name], v7_info = adapt_three_bucket_v7(
                    fc_layer=model.fc,
                    Z_replay=Z_replay, y_replay=y_replay,
                    Z_target=Z_adapt,
                    idx_gate=spec["idx_gate"], idx_rel=spec["idx_rel"], idx_amb=spec["idx_amb"],
                    idx_weak=spec["idx_weak"], idx_unk=spec["idx_unk"],
                    q_seed=spec["q_seed"], w_rel=spec["w_rel"], w_amb=spec["w_amb"], w_weak=spec["w_weak"],
                    proto_bank=protos,
                    idx_weak_cold=spec.get("idx_weak_cold", None), w_weak_cold=spec.get("w_weak_cold", None),
                    bucket_score=spec.get("bucket_score", None), unknown_score=spec.get("unknown_score", None),
                    bottleneck=ADAPTER_BOTTLENECK, scale=ADAPTER_SCALE, epochs=THREE_BUCKET_EPOCHS,
                    stage1_epochs=spec.get("stage1_epochs", V7_STAGE1_EPOCHS),
                    lr=ADAPT_LR, wd=ADAPT_WEIGHT_DECAY, batch=ADAPT_BATCH,
                    lambda_replay_ce=LAMBDA_REPLAY_CE, lambda_replay_anchor=LAMBDA_REPLAY_ANCHOR,
                    lambda_rel_sup=spec.get("lambda_rel_sup", 1.0),
                    lambda_amb_sup=spec.get("lambda_amb_sup", V7_LAMBDA_AMB_SUP),
                    lambda_weak_sup=spec.get("lambda_weak_sup", V7_LAMBDA_WEAK_SUP),
                    lambda_align=spec.get("lambda_align", V7_LAMBDA_ALIGN),
                    lambda_ent_min=spec.get("lambda_ent_min", V7_LAMBDA_ENT_MIN),
                    lambda_ent_max=spec.get("lambda_ent_max", V7_LAMBDA_ENT_MAX),
                    lambda_cons=spec.get("lambda_cons", V7_LAMBDA_CONS),
                    lambda_proto=spec.get("lambda_proto", V7_LAMBDA_PROTO),
                    lambda_src_proto=spec.get("lambda_src_proto", LAMBDA_SRC_PROTO * 0.60),
                    lambda_src_logit=spec.get("lambda_src_logit", LAMBDA_SRC_LOGIT * 0.60),
                    lambda_u_repel=spec.get("lambda_u_repel", V7_LAMBDA_UNKNOWN_REPEL),
                    use_ema=spec.get("use_ema", True),
                    teacher_blend=spec.get("teacher_blend", EMA_TEACHER_BLEND),
                    promote_blend=spec.get("promote_blend", V7_PROMOTE_BLEND),
                    promote_rel_fraction=spec.get("promote_rel_fraction", V7_PROMOTE_REL_FRACTION),
                    promote_amb_fraction=spec.get("promote_amb_fraction", V7_PROMOTE_AMB_FRACTION),
                    promote_conf=spec.get("promote_conf", V7_PROMOTE_CONF),
                    promote_margin=spec.get("promote_margin", V7_PROMOTE_MARGIN),
                    unknown_warmup_epochs=spec.get("unknown_warmup_epochs", V7_UNKNOWN_WARMUP_EPOCHS),
                    unknown_ramp_epochs=spec.get("unknown_ramp_epochs", V7_UNKNOWN_RAMP_EPOCHS),
                )
                extra_method_info[name] = {**spec.get("bucket_info", {}), **(v7_info or {})}
            elif spec["train"] == "three_bucket_v8":
                adapters[name], v8_info = adapt_three_bucket_v8(
                    fc_layer=model.fc,
                    Z_replay=Z_replay, y_replay=y_replay,
                    Z_target=Z_adapt,
                    idx_gate=spec["idx_gate"], idx_rel=spec["idx_rel"], idx_amb=spec["idx_amb"],
                    idx_weak=spec["idx_weak"], idx_unk=spec["idx_unk"],
                    q_seed=spec["q_seed"], w_rel=spec["w_rel"], w_amb=spec["w_amb"], w_weak=spec["w_weak"],
                    proto_bank=protos,
                    idx_weak_cold=spec.get("idx_weak_cold", None), w_weak_cold=spec.get("w_weak_cold", None),
                    bucket_score=spec.get("bucket_score", None), unknown_score=spec.get("unknown_score", None),
                    Sid_adapt=sc_adapt["Sid"], Sdom_adapt=sc_adapt["Sdom"],
                    bottleneck=ADAPTER_BOTTLENECK, scale=ADAPTER_SCALE,
                    stage0_epochs=V8_STAGE0_EPOCHS, refresh_epochs=V8_REFRESH_EPOCHS, refresh_rounds=V8_REFRESH_ROUNDS,
                    lr=ADAPT_LR, wd=ADAPT_WEIGHT_DECAY, batch=ADAPT_BATCH,
                    lambda_replay_ce=LAMBDA_REPLAY_CE, lambda_replay_anchor=LAMBDA_REPLAY_ANCHOR,
                    lambda_align=spec.get("lambda_align", V8_LAMBDA_ALIGN),
                    lambda_ent_min=spec.get("lambda_ent_min", V8_LAMBDA_ENT_MIN),
                    lambda_ent_max=spec.get("lambda_ent_max", V8_LAMBDA_ENT_MAX),
                    lambda_cons=spec.get("lambda_cons", V8_LAMBDA_CONS),
                    lambda_proto=spec.get("lambda_proto", V8_LAMBDA_PROTO),
                    lambda_u_repel=spec.get("lambda_u_repel", V8_LAMBDA_UNKNOWN_REPEL),
                )
                extra_method_info[name] = {**spec.get("bucket_info", {}), **(v8_info or {})}
            else:
                raise ValueError(f"Unknown training mode: {spec['train']}")


        fold_metrics = dict(
            split=split_id, fold=fold,
            tau_id=tau_id, tau_dom=tau_dom, tau_conf=tau_conf,
            pseudo_logit_rx=pseudo_eval["logit"]["rx"], pseudo_logit_day=pseudo_eval["logit"]["day"], pseudo_logit_all=pseudo_eval["logit"]["all"],
            pseudo_lp_rx=pseudo_eval["lp"]["rx"],       pseudo_lp_day=pseudo_eval["lp"]["day"],       pseudo_lp_all=pseudo_eval["lp"]["all"],
            pseudo_tri_rx=pseudo_eval["tri"]["rx"],     pseudo_tri_day=pseudo_eval["tri"]["day"],     pseudo_tri_all=pseudo_eval["tri"]["all"],
        )

        for name, info in method_info.items():
            fold_metrics[f"{name}_sel_precision"] = info["sel_precision"]
            fold_metrics[f"{name}_sel_recall"] = info["sel_recall"]
            fold_metrics[f"{name}_sel_keep"] = info["sel_keep_ratio"]
            fold_metrics[f"{name}_pseudo_acc_selected"] = info["pseudo_acc_selected"]
            fold_metrics[f"{name}_sel_size"] = info["sel_size"]
            fold_metrics[f"{name}_unknown_cand_keep"] = info["unknown_cand_keep"]
            fold_metrics[f"{name}_unknown_cand_precision"] = info["unknown_cand_precision"]
            for k, v in info.items():
                if k not in {"sel_precision","sel_recall","sel_keep_ratio","pseudo_acc_selected","sel_size","unknown_cand_keep","unknown_cand_precision","pseudo_acc"}:
                    fold_metrics[f"{name}_{k}"] = v
        for name, info in extra_method_info.items():
            for k, v in info.items():
                fold_metrics[f"{name}_{k}"] = v

        for name in METHOD_ORDER:
            if name == "NoAdapt":
                adapter = None
            else:
                adapter = adapters[name]
            fold_metrics[name] = evaluate_variant(
                adapter, model.fc,
                Z_A, y_src[idx_te], D_A,
                Z_B_ev, y_B_ev, D_B_ev,
                Z_C_ev, y_C_ev, D_C_ev,
                Z_U_ev, D_U_ev,
                mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A, models_kr, fallback_k, source_rx_ids,
                tau_id, tau_dom
            )

        plot_bar_compare({k: fold_metrics[k]["drift_acc_all"] for k in METHOD_ORDER}, os.path.join(fold_dir, "drift_acc_all_compare.png"), "Drift accuracy (all) compare")
        plot_bar_compare({k: fold_metrics[k]["FAR_unknown_accept"] for k in METHOD_ORDER}, os.path.join(fold_dir, "far_unknown_compare.png"), "Unknown FAR compare")

        with open(os.path.join(fold_dir, "metrics_module3_adaptation_suite.json"), "w", encoding="utf-8") as f:
            json.dump(fold_metrics, f, indent=2)

        rows.append(fold_metrics)
        print(
            f"[split {split_id} fold {fold}] "
            f"pLogit={pseudo_eval['logit']['all']:.3f} pLP={pseudo_eval['lp']['all']:.3f} pTri={pseudo_eval['tri']['all']:.3f} | "
            f"NoAdapt={fold_metrics['NoAdapt']['drift_acc_all']:.3f} "
            f"TriSoft={fold_metrics['GatedTriFusionSoft']['drift_acc_all']:.3f} "
            f"Refine={fold_metrics['ProtoRefineSoft']['drift_acc_all']:.3f} "
            f"RAIN={fold_metrics['RAINCOAT_Lite']['drift_acc_all']:.3f} "
            f"DGR={fold_metrics['DG_RAINCOAT']['drift_acc_all']:.3f} "
            f"TBV5S={fold_metrics['ThreeBucketV5Soft']['drift_acc_all']:.3f} "
            f"TBV5E={fold_metrics['ThreeBucketV5EMA']['drift_acc_all']:.3f} "
            f"TBV5P={fold_metrics['ThreeBucketV5EMAProto']['drift_acc_all']:.3f} "
            f"TBV6={fold_metrics['ThreeBucketV6Curriculum']['drift_acc_all']:.3f} "
            f"TBV7={fold_metrics['ThreeBucketV7Promotion']['drift_acc_all']:.3f} "
            f"TBV8={fold_metrics['ThreeBucketV8Adaptive']['drift_acc_all']:.3f} "
            f"OracleLbl={fold_metrics['OracleLabelAdapt']['drift_acc_all']:.3f}"
        )

    with open(os.path.join(split_dir, "metrics_all_folds.json"), "w", encoding="utf-8") as f:
        json.dump(rows, f, indent=2)
    return rows

all_rows = []
for split_id, (KNOWN_TX, UNKNOWN_TX) in enumerate(TX_SPLITS, start=1):
    print("\n=== TX split", split_id, "===")
    print("KNOWN_TX:", KNOWN_TX)
    print("UNKNOWN_TX:", UNKNOWN_TX)
    all_rows.extend(run_one_split(split_id, KNOWN_TX, UNKNOWN_TX))

with open(os.path.join(RUN_DIR, "metrics_all_splits_folds.json"), "w", encoding="utf-8") as f:
    json.dump(all_rows, f, indent=2)

print("\nSaved all metrics to:", RUN_DIR)




=== TX split 1 ===
KNOWN_TX: ['14-7', '20-19', '6-15', '8-20']
UNKNOWN_TX: ['14-10', '20-15']
[split 1 fold 1] pLogit=0.717 pLP=0.722 pTri=0.733 | NoAdapt=0.717 TriSoft=0.731 Refine=0.745 RAIN=0.807 DGR=0.682 TBV5S=0.666 TBV5E=0.677 TBV5P=0.680 TBV6=0.716 TBV7=0.723 TBV8=0.685 OracleLbl=0.911
[split 1 fold 2] pLogit=0.727 pLP=0.735 pTri=0.752 | NoAdapt=0.727 TriSoft=0.748 Refine=0.759 RAIN=0.751 DGR=0.714 TBV5S=0.703 TBV5E=0.717 TBV5P=0.707 TBV6=0.732 TBV7=0.735 TBV8=0.731 OracleLbl=0.908
[split 1 fold 3] pLogit=0.728 pLP=0.731 pTri=0.752 | NoAdapt=0.728 TriSoft=0.726 Refine=0.723 RAIN=0.698 DGR=0.703 TBV5S=0.687 TBV5E=0.682 TBV5P=0.692 TBV6=0.717 TBV7=0.722 TBV8=0.692 OracleLbl=0.901
[split 1 fold 4] pLogit=0.718 pLP=0.715 pTri=0.722 | NoAdapt=0.718 TriSoft=0.716 Refine=0.715 RAIN=0.747 DGR=0.715 TBV5S=0.704 TBV5E=0.711 TBV5P=0.707 TBV6=0.710 TBV7=0.714 TBV8=0.704 OracleLbl=0.900
[split 1 fold 5] pLogit=0.701 pLP=0.701 pTri=0.721 | NoAdapt=0.701 TriSoft=0.713 Refine=0.708 RAIN=0.668 

C:\Users\zy_seven\AppData\Local\Temp\ipykernel_10044\3141885056.py:246: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1839.)
  scale = z.detach().std(dim=0, keepdim=True).mean().clamp(min=1e-4)


[split 4 fold 4] pLogit=0.725 pLP=0.729 pTri=0.734 | NoAdapt=0.725 TriSoft=0.724 Refine=0.724 RAIN=0.663 DGR=0.694 TBV5S=0.691 TBV5E=0.695 TBV5P=0.701 TBV6=0.724 TBV7=0.247 TBV8=0.694 OracleLbl=0.877
[split 4 fold 5] pLogit=0.647 pLP=0.658 pTri=0.689 | NoAdapt=0.647 TriSoft=0.682 Refine=0.682 RAIN=0.740 DGR=0.658 TBV5S=0.651 TBV5E=0.638 TBV5P=0.636 TBV6=0.661 TBV7=0.665 TBV8=0.667 OracleLbl=0.890

=== TX split 5 ===
KNOWN_TX: ['14-10', '14-7', '20-15', '8-20']
UNKNOWN_TX: ['20-19', '6-15']
[split 5 fold 1] pLogit=0.670 pLP=0.686 pTri=0.720 | NoAdapt=0.670 TriSoft=0.696 Refine=0.743 RAIN=0.723 DGR=0.752 TBV5S=0.719 TBV5E=0.709 TBV5P=0.712 TBV6=0.711 TBV7=0.700 TBV8=0.708 OracleLbl=0.956
[split 5 fold 2] pLogit=0.585 pLP=0.593 pTri=0.669 | NoAdapt=0.585 TriSoft=0.779 Refine=0.804 RAIN=0.796 DGR=0.790 TBV5S=0.769 TBV5E=0.736 TBV5P=0.746 TBV6=0.751 TBV7=0.736 TBV8=0.792 OracleLbl=0.958
[split 5 fold 3] pLogit=0.685 pLP=0.688 pTri=0.714 | NoAdapt=0.685 TriSoft=0.755 Refine=0.782 RAIN=0.841 

## Summary and diagnostics block

This section aggregates:
- pseudo-label quality
- selection quality
- per-method final metrics
- bucket / promotion diagnostics

For every future version, add new markdown here if:
- a new metric is introduced
- a new diagnostic block is added
- the meaning of an existing metric changes

In [11]:
def mean_std(vals):
    vals = np.array(vals, dtype=np.float64)
    return float(vals.mean()), float(vals.std(ddof=0))

summary = {}
scalar_keys = [
    "pseudo_logit_rx","pseudo_logit_day","pseudo_logit_all",
    "pseudo_lp_rx","pseudo_lp_day","pseudo_lp_all",
    "pseudo_tri_rx","pseudo_tri_day","pseudo_tri_all",
    "tau_id","tau_dom","tau_conf"
]
for key in scalar_keys:
    summary[key] = dict(zip(["mean","std"], mean_std([r[key] for r in all_rows])))

metrics = ["stable_acc","drift_acc_rx","drift_acc_day","drift_acc_all","FRR_stable","FAR_unknown_accept","miss_drift_rx","miss_drift_day","miss_drift_all","auc_unknown_eval"]
summary["variants"] = {}
for v in METHOD_ORDER:
    summary["variants"][v] = {}
    for m in metrics:
        vals = [r[v][m] for r in all_rows]
        summary["variants"][v][m] = dict(zip(["mean","std"], mean_std(vals)))

pseudo_methods = [v for v in METHOD_ORDER if v != "NoAdapt"]
summary["selection"] = {}
sel_keys = ["sel_precision","sel_recall","sel_keep","pseudo_acc_selected","unknown_cand_keep","unknown_cand_precision"]
for pm in pseudo_methods:
    summary["selection"][pm] = {}
    for sk in sel_keys:
        vals = [r[f"{pm}_{sk}"] for r in all_rows if f"{pm}_{sk}" in r]
        if len(vals) == 0:
            continue
        summary["selection"][pm][sk] = dict(zip(["mean","std"], mean_std(vals)))

bucket_keys = [
    "reliable_size","ambiguous_size","weak_size","unknown_size",
    "reliable_keep","ambiguous_keep","weak_keep","unknown_keep",
    "bucket_score_gate_mean",
    "promoted_reliable","promoted_ambiguous",
    "stage2_reliable_size","stage2_ambiguous_size","stage2_weak_size","stage2_unknown_size",
    "promote_score_gate_mean","stability_gate_mean","refresh_rounds",
    "common_score_gate_mean","unknown_score_gate_mean","move_gate_mean","stable_pred_ratio"
]
summary["bucket_stats"] = {}
for pm in pseudo_methods:
    avail = [bk for bk in bucket_keys if all(f"{pm}_{bk}" in r for r in all_rows)]
    if len(avail) == 0:
        continue
    summary["bucket_stats"][pm] = {}
    for bk in avail:
        vals = [r[f"{pm}_{bk}"] for r in all_rows]
        summary["bucket_stats"][pm][bk] = dict(zip(["mean","std"], mean_std(vals)))

with open(os.path.join(RUN_DIR, "summary_mean_std.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("=== Pseudo-label accuracy on true drift pools ===")
for k in ["pseudo_logit_all","pseudo_lp_all","pseudo_tri_all"]:
    print(f"{k:24s}: {summary[k]['mean']:.3f} ± {summary[k]['std']:.3f}")

print("\n=== Selection quality on adaptation stream ===")
for pm in pseudo_methods:
    print(f"\n[{pm}]")
    for sk in sel_keys:
        if sk in summary["selection"][pm]:
            print(f"{sk:24s}: {summary['selection'][pm][sk]['mean']:.3f} ± {summary['selection'][pm][sk]['std']:.3f}")

for v in METHOD_ORDER:
    print(f"\n=== {v} ===")
    for m in metrics:
        print(f"{m:24s}: {summary['variants'][v][m]['mean']:.3f} ± {summary['variants'][v][m]['std']:.3f}")

print("\n=== Bucket diagnostics ===")
for pm, diag in summary["bucket_stats"].items():
    if len(diag) == 0:
        continue
    print(f"\n[{pm}]")
    for bk, st in diag.items():
        print(f"{bk:24s}: {st['mean']:.3f} ± {st['std']:.3f}")



=== Pseudo-label accuracy on true drift pools ===
pseudo_logit_all        : 0.721 ± 0.053
pseudo_lp_all           : 0.727 ± 0.051
pseudo_tri_all          : 0.749 ± 0.044

=== Selection quality on adaptation stream ===

[UngatedAdapt]
sel_precision           : 0.400 ± 0.000
sel_recall              : 1.000 ± 0.000
sel_keep                : 1.000 ± 0.000
pseudo_acc_selected     : 0.723 ± 0.053
unknown_cand_keep       : 0.000 ± 0.000
unknown_cand_precision  : nan ± nan

[GatedAdapt]
sel_precision           : 0.520 ± 0.203
sel_recall              : 0.174 ± 0.037
sel_keep                : 0.163 ± 0.091
pseudo_acc_selected     : 0.921 ± 0.070
unknown_cand_keep       : 0.000 ± 0.000
unknown_cand_precision  : nan ± nan

[GatedSelfSoft]
sel_precision           : 0.520 ± 0.203
sel_recall              : 0.174 ± 0.037
sel_keep                : 0.163 ± 0.091
pseudo_acc_selected     : 0.921 ± 0.070
unknown_cand_keep       : 0.000 ± 0.000
unknown_cand_precision  : nan ± nan

[GatedProtoAgree]
sel_prec